# World Vision Australia — Donor Retention Analytics

> **Private review draft — not cleared for public GitHub release.** The original client datasets are not distributed. All saved execution outputs and supporter-level previews have been removed from this derivative copy. The notebook has **not** been rerun. Source-code and aggregate disclosure permissions still require confirmation with the client or university before publishing.

This notebook preserves the original Part D workflow: data understanding and preparation, 24-month repeat-gift cohort construction, exploratory analysis, feature engineering, Logistic Regression / Random Forest / CatBoost comparison, retrospective cohort testing, Top-K ranking and recommendations.

**Reproducibility:** Execution requires authorised access to the six original WVA extracts, a compatible Python environment and review of existing source-file paths. Historical numerical findings in the portfolio come from recorded Part D outputs, not a fresh run of this public-facing copy.

**Portfolio sections:** [Business Understanding](../01-business-understanding.md) · [Data Preparation](../02-data-understanding-and-preparation.md) · [EDA](../03-exploratory-analysis-and-insights.md) · [Predictive Evaluation](../04-predictive-modelling-and-evaluation.md) · [Recommendations](../05-recommendations-and-business-value.md)

---


## Before running

This code is preserved for methodological review and was not re-executed during portfolio preparation. Use only within an authorised environment containing the original client data. Running cells can display sensitive data or generate files containing supporter information; do not commit those outputs. The original notebook retains the unmodified outputs privately.


# 1. Data Setup

This section prepares the Google Colab environment and makes the supplied World Vision Australia data available for analysis.

The original ZIP file is uploaded and extracted in one step so that the project can be reproduced easily without requiring individual dataset uploads.

## 1.1 Import Required Libraries

The basic Python libraries required for data handling and file management are imported.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import os
import zipfile
import matplotlib.pyplot as plt
print("Libraries imported successfully.")

## 1.2 Upload and Extract Dataset

The original World Vision Australia ZIP file is uploaded directly into Google Colab and extracted into a single working folder.

This keeps all supplied datasets together and avoids uploading each file separately.

In [ ]:
from google.colab import files

# Upload the original ZIP file
uploaded = files.upload()

# Get the uploaded ZIP file name
zip_file = list(uploaded.keys())[0]

# Create a working folder
data_folder = "/content/world_vision_data"
os.makedirs(data_folder, exist_ok=True)

# Extract the ZIP file
with zipfile.ZipFile(zip_file, "r") as zip_ref:
    zip_ref.extractall(data_folder)

print("Dataset extracted successfully.")

## 1.3 Verify Extracted Files

The extracted files are listed to confirm that all required datasets and supporting files are available before loading them into Python.

In [ ]:
# Portfolio privacy edit: local dataset file listing suppressed.
# Original workflow verified extracted files before loading.


## 1.4 Define Dataset File Paths

File paths are defined for the six datasets so they can be loaded consistently in the following sections.

In [ ]:
# Define paths for the six datasets
one_off_path = os.path.join(data_folder, "One_Off_Donations_20260318.txt")
ongoing_path = os.path.join(data_folder, "Ongoing_Commitments_20260318.txt")
demographics_path = os.path.join(data_folder, "Supporter_Demographics_20260318.txt")
comms_sent_path = os.path.join(data_folder, "Comms_Sent_20260318.txt")
comms_clicked_path = os.path.join(data_folder, "Comms_Clicked_20260318.txt")
portal_path = os.path.join(data_folder, "Portal_Engagement_20260318.txt")

print("Dataset paths created successfully.")

# 2. Data Understanding

This section examines the six supplied datasets and identifies the information that is relevant to understanding and predicting repeat one-off giving.

The datasets are reviewed before cleaning or modelling so that their structure, coverage and limitations are understood correctly.

## 2.1 Load Working Datasets

### 2.1.1 One-Off Donations

The One-Off Donations dataset provides the transaction-level history required to identify first gifts and subsequent repeat giving.

A structured working version is created while retaining donation amount, channel, product, catalogue, campaign, activity and acquisition information. These variables provide useful context for later analysis of the types of first gifts associated with repeat giving.

Some descriptive fields contain embedded commas, so the consistently formatted financial-year fields are used as anchors to load the dataset reliably without removing valid records.

In [ ]:
# ---------------------------------------------------------
# Load One-Off Donations
# ---------------------------------------------------------

import re
import csv

# Final variables retained from the raw dataset
one_off_columns = [
    "Deakin_Supporter_ID",
    "DMRT_Commitment_Key",
    "Fulfilment_Date",
    "Total_Paid",
    "Attribution_Channel",
    "Product_Category",
    "Product_Code",
    "Catalogue_Item_Description",
    "Acquisition_FY",
    "Acquisition_FM",
    "Campaign_Name",
    "Activity_Name",
    "Acquisition_Segment_Business_Unit",
    "Acquisition_Segment_Team",
    "Acquisition_Segment_Sub_Team",
    "Age_At_Acquisition"
]

# Create a reliable working file
working_one_off_path = os.path.join(
    data_folder,
    "One_Off_Donations_Working.csv"
)

loaded_rows = 0
unreadable_rows = 0

with open(
    one_off_path,
    "r",
    encoding="cp1252",
    errors="replace"
) as source, open(
    working_one_off_path,
    "w",
    encoding="utf-8",
    newline=""
) as output:

    writer = csv.writer(output)
    writer.writerow(one_off_columns)

    next(source)  # Skip original header

    for line in source:

        parts = line.rstrip("\r\n").split(",")

        # Find the reliable Acquisition FY and FM position
        fy_position = None

        for i in range(7, len(parts) - 4):
            if (
                re.fullmatch(r"FY\d{2}", parts[i].strip())
                and re.fullmatch(
                    r"[A-Za-z]{3} FY\d{4}",
                    parts[i + 1].strip()
                )
            ):
                fy_position = i
                break

        if fy_position is None:
            unreadable_rows += 1
            continue

        # Restore descriptive fields that may contain commas
        catalogue_item = ",".join(
            parts[7:fy_position]
        ).strip()

        campaign_name = parts[
            fy_position + 2
        ].strip()

        activity_name = ",".join(
            parts[fy_position + 3:-4]
        ).strip()

        # Write the reconstructed 16-column row
        writer.writerow([
            parts[0],
            parts[1],
            parts[2],
            parts[3],
            parts[4],
            parts[5],
            parts[6],
            catalogue_item,
            parts[fy_position],
            parts[fy_position + 1],
            campaign_name,
            activity_name,
            parts[-4],
            parts[-3],
            parts[-2],
            parts[-1]
        ])

        loaded_rows += 1


# Load final working dataset into Python
one_off = pd.read_csv(
    working_one_off_path,
    low_memory=False
)

print("Rows loaded:", f"{loaded_rows:,}")
print("Unreadable rows:", f"{unreadable_rows:,}")
print("Shape:", one_off.shape)

# Portfolio privacy edit: supporter-level preview suppressed.

### 2.1.2 Supporter Demographics

The Supporter Demographics dataset provides one record per supporter, including location, World Vision segmentation, Helix profile, Lewers segment and the date the supporter joined the WVA database.

These variables are retained for descriptive analysis and to understand the wider supporter profile. Some fields represent the supporter's current profile rather than their historical profile at the time of the first gift, so their suitability for predictive modelling will be assessed carefully later.

Some Helix persona values contain embedded commas, so these are reconstructed as a single field while retaining all supporter records.

In [ ]:
# Demographic dataset columns
demographic_columns = [
    "Deakin_Supporter_ID",
    "Date_Joined_WVA_Database",
    "Age_At_Joining_WVA",
    "Address_Postcode",
    "Address_Suburb",
    "Address_State",
    "Address_Country",
    "Current_Segment_Business_Unit",
    "Current_Segment_Team",
    "Current_Segment_Sub_Team",
    "Helix_Community",
    "Helix_Persona",
    "Lewers_Segment_Description",
    "Joined_WVA_FY",
    "Joined_WVA_FM"
]

demographic_rows = []
unreadable_rows = 0

with open(demographics_path, "r", encoding="utf-16", errors="replace") as file:

    next(file)  # Skip header

    for line in file:
        parts = line.rstrip("\r\n").split(",")

        if len(parts) >= 15:

            # Rejoin any commas that belong inside Helix_Persona
            helix_persona = ",".join(parts[11:-3]).strip()

            demographic_rows.append(
                parts[:11]
                + [helix_persona]
                + parts[-3:]
            )

        else:
            unreadable_rows += 1

# Create working DataFrame
demographics = pd.DataFrame(
    demographic_rows,
    columns=demographic_columns
)

print("Rows loaded:", f"{len(demographics):,}")
print("Unreadable rows:", f"{unreadable_rows:,}")
print("Shape:", demographics.shape)

# Portfolio privacy edit: supporter-level preview suppressed.

### 2.1.3 Ongoing Commitments

The Ongoing Commitments dataset provides the history of supporters' regular-giving relationships with World Vision, including child sponsorship and other ongoing commitments.

These records help identify whether a supporter had no previous ongoing relationship, an active ongoing commitment, or a former ongoing relationship when their first eligible one-off gift was made. They also provide information such as previous product, channel, commitment dates and payment type that can support later analysis of repeat giving.

Some free-text fields contain inconsistent commas and line breaks, so a reliable working version is created using the structured relationship-history fields. These records will later be summarised to supporter level before being combined with the one-off donor data.

In [ ]:
# Fields required to understand ongoing relationship history
ongoing_columns = [
    "DMRT_Commitment_Key",
    "Deakin_Supporter_ID",
    "Product_Group",
    "Product_Category",
    "Acquisition_Type_Group",
    "Attribution_Channel",
    "Attribution_Channel_Group",
    "Fulfilment_Date",
    "Fulfilment_Ended_Date",
    "Closure_Date",
    "Payee_Signup_Age",
    "Payee_Signup_Age_Group",
    "Acquisition_Segment_Business_Unit",
    "Acquisition_Segment_Team",
    "Acquisition_Segment_Sub_Team",
    "Cancellation_Segment_Business_Unit",
    "Cancellation_Segment_Team",
    "Cancellation_Segment_Sub_Team",
    "Payment_Type",
    "Acquisition_FY",
    "Acquisition_FM",
    "Age_At_Acquisition"
]

# Rebuild complete records where free-text comments span multiple lines
records = []
current_record = ""

with open(ongoing_path, "r", encoding="cp1252", errors="replace") as file:
    next(file)  # Skip header

    for line in file:
        line = line.rstrip("\r\n")

        # A new commitment record starts with Commitment Key and Supporter ID
        if re.match(r"^\d+(?:\.\d+)?,\d+,", line):
            if current_record:
                records.append(current_record)

            current_record = line
        else:
            current_record += " " + line

    if current_record:
        records.append(current_record)

# Extract reliable fields from the beginning and end of each record
ongoing_rows = []
unreadable_rows = 0

for record in records:

    first = record.split(",", 7)

    if len(first) < 8:
        unreadable_rows += 1
        continue

    # Last 15 fields have a stable structure
    last = first[7].rsplit(",", 15)

    if len(last) != 16:
        unreadable_rows += 1
        continue

    ongoing_rows.append(first[:7] + last[1:])

# Create working DataFrame
ongoing = pd.DataFrame(
    ongoing_rows,
    columns=ongoing_columns
)

print("Logical records found:", f"{len(records):,}")
print("Rows loaded:", f"{len(ongoing):,}")
print("Unreadable rows:", f"{unreadable_rows:,}")
print("Shape:", ongoing.shape)

# Portfolio privacy edit: supporter-level preview suppressed.

### 2.1.4 Communications and Portal Engagement

The Communications Sent, Communications Clicked and Portal Engagement datasets provide information about recorded supporter contact and digital engagement with World Vision.

These datasets are retained to examine whether engagement patterns differ between repeat and non-repeat donors. For predictive modelling, only engagement recorded before the first eligible one-off gift can be considered, preventing future supporter behaviour from influencing the prediction.

The usefulness of these datasets will also depend on their historical coverage, which is reviewed in the following sections.

In [ ]:
# Load communication and portal datasets
comms_sent = pd.read_csv(comms_sent_path, encoding="cp1252")
comms_clicked = pd.read_csv(comms_clicked_path, encoding="cp1252")
portal = pd.read_csv(portal_path, encoding="cp1252")

print("Communications Sent:", comms_sent.shape)
print("Communications Clicked:", comms_clicked.shape)
print("Portal Engagement:", portal.shape)

## 2.2 Check Dataset Structure

### 2.2.1 Dataset Size and Supporter Coverage

The size and supporter coverage of each dataset are compared.

This distinguishes the number of records or events from the number of individual supporters represented in each data source.

In [ ]:
# Store the six working datasets
datasets = {
    "One-Off Donations": one_off,
    "Supporter Demographics": demographics,
    "Ongoing Commitments": ongoing,
    "Communications Sent": comms_sent,
    "Communications Clicked": comms_clicked,
    "Portal Engagement": portal
}

# Create a simple dataset summary
summary = []

for name, df in datasets.items():
    summary.append({
        "Dataset": name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Unique Supporters": df["Deakin_Supporter_ID"].nunique()
    })

dataset_summary = pd.DataFrame(summary)

display(dataset_summary)

**Observation:** The One-Off Donations and Supporter Demographics datasets cover the full one-off supporter population, while the ongoing commitment, communication and portal datasets cover smaller subsets of supporters. Therefore, missing records in these additional datasets need to be interpreted carefully rather than automatically treated as no previous relationship or no engagement.

### 2.2.2 Check Date Coverage

The available time period in each dataset is reviewed to understand how much historical information is available and whether any data sources have limited coverage.

In [ ]:
# Helper function to display date range
def show_date_range(name, series):
    dates = pd.to_datetime(series, errors="coerce")
    print(f"{name}: {dates.min().date()} to {dates.max().date()}")

# Check relevant date ranges
show_date_range("One-Off Donations",
                one_off["Fulfilment_Date"])

show_date_range("Supporter Demographics",
                demographics["Date_Joined_WVA_Database"])

show_date_range("Ongoing Commitments",
                ongoing["Fulfilment_Date"])

show_date_range("Communications Sent",
                comms_sent["Comm_Sent_Date"])

show_date_range("Communications Clicked",
                comms_clicked["Comm_Clicked_Date"])

show_date_range("Portal Engagement",
                portal["LoggedIn_Datetime"])

**Observation:** The datasets have different historical coverage. One-off donations and ongoing commitments provide long-term history, while communication and portal datasets are much more recent. Communications Clicked is particularly limited to 2018–2019, so the absence of a click record outside this period cannot be interpreted as no engagement.

This uneven coverage means engagement variables need to be used carefully in later analysis and are less suitable as primary predictors across the full modelling period.

### 2.2.3 Check Data Types

The data types of the variables are reviewed to identify fields that require conversion before analysis, particularly dates and numerical values.

In [ ]:
# Display data types for each dataset
for name, df in datasets.items():
    print(f"\n{name}")
    print(df.dtypes)

**Observation:** Date fields are currently stored as text and supporter identifiers use different data types across the datasets. These fields will be standardised during data cleaning so dates can be compared correctly and supporter records can be linked reliably. Text-based fields such as product, channel, campaign and activity are appropriately retained as categorical information for later analysis.

### 2.2.4 Check Missing Values in Key Fields

Missing values are reviewed in the main variables required for donor identification, donation behaviour, supporter characteristics, relationship history and engagement.

This helps identify potential data limitations before cleaning and modelling.

In [ ]:
# ---------------------------------------------------------
# Check Missing Values in Key Fields
# ---------------------------------------------------------

key_fields = {

    "One-Off Donations": [
        "Deakin_Supporter_ID",
        "Fulfilment_Date",
        "Total_Paid",
        "Attribution_Channel",
        "Product_Category",
        "Product_Code",
        "Catalogue_Item_Description",
        "Campaign_Name",
        "Activity_Name",
        "Age_At_Acquisition"
    ],

    "Supporter Demographics": [
        "Deakin_Supporter_ID",
        "Date_Joined_WVA_Database",
        "Age_At_Joining_WVA",
        "Address_State",
        "Helix_Community",
        "Helix_Persona",
        "Lewers_Segment_Description"
    ],

    "Ongoing Commitments": [
        "Deakin_Supporter_ID",
        "Product_Group",
        "Product_Category",
        "Fulfilment_Date",
        "Fulfilment_Ended_Date",
        "Closure_Date",
        "Age_At_Acquisition"
    ],

    "Communications Sent": [
        "Deakin_Supporter_ID",
        "Communication_Method",
        "Comm_Sent_Date"
    ],

    "Communications Clicked": [
        "Deakin_Supporter_ID",
        "Communication_Method",
        "Comm_Clicked_Date"
    ],

    "Portal Engagement": [
        "Deakin_Supporter_ID",
        "Registered_Datetime",
        "LoggedIn_Datetime"
    ]
}


# Check missing values without changing the datasets
for name, columns in key_fields.items():

    df = datasets[name]

    print(f"\n{name}")

    for column in columns:

        missing = (
            df[column].isna()
            | df[column].fillna("").astype(str).str.strip().eq("")
        ).sum()

        percentage = missing / len(df) * 100

        print(
            f"{column}: "
            f"{missing:,} ({percentage:.1f}%)"
        )

**Observation:** Core supporter identifiers, donation dates, amounts, channels and products are complete. Campaign and activity information are also almost fully available, while catalogue item information is available for most donation records, making these fields suitable for further exploratory analysis.

Age information has considerably more missing values, particularly in the demographic dataset, so age-based findings will be interpreted cautiously. Missing end and closure dates in ongoing commitments may represent active relationships rather than data errors and will therefore be assessed during relationship-history preparation.

### 2.2.5 Check Record Uniqueness

Record uniqueness is checked to identify accidental duplicate records before further analysis.

Repeated supporter IDs are expected in transaction and event-based datasets because one supporter can have multiple donations, commitments or interactions. Therefore, exact duplicate rows and identifiers that should uniquely represent individual records are checked separately.

In [ ]:
# Check exact duplicate rows
print("Exact duplicate rows:")

for name, df in datasets.items():
    print(f"{name}: {df.duplicated().sum():,}")


# Check identifiers that should uniquely identify records
print("\nDuplicate key values:")

print(
    "Demographics - Supporter ID:",
    demographics["Deakin_Supporter_ID"].duplicated().sum()
)

print(
    "One-Off Donations - Commitment Key:",
    one_off["DMRT_Commitment_Key"].duplicated().sum()
)

print(
    "Ongoing Commitments - Commitment Key:",
    ongoing["DMRT_Commitment_Key"].duplicated().sum()
)

**Observation:** No exact duplicate records or duplicate primary record identifiers were identified. Repeated supporter IDs in transaction and engagement datasets represent multiple legitimate activities rather than duplicate records, so no duplicate removal is required.

## 2.3 Understand Dataset Relationships

### 2.3.1 Check Supporter Overlap

Supporter IDs are compared across the datasets to understand how much additional demographic, relationship and engagement information is available for the one-off donor population.

This is particularly important for identifying supporters with an ongoing-giving history and understanding the coverage of the communication and portal datasets before these sources are combined.

In [ ]:
# All supporters represented in the one-off donor population
one_off_ids = set(one_off["Deakin_Supporter_ID"].astype(str))

# Compare supporter coverage with the other datasets
overlap_summary = pd.DataFrame({
    "Dataset": [
        "Supporter Demographics",
        "Ongoing Commitments",
        "Communications Sent",
        "Communications Clicked",
        "Portal Engagement"
    ],
    "Supporters also in One-Off Population": [
        len(one_off_ids & set(demographics["Deakin_Supporter_ID"].astype(str))),
        len(one_off_ids & set(ongoing["Deakin_Supporter_ID"].astype(str))),
        len(one_off_ids & set(comms_sent["Deakin_Supporter_ID"].astype(str))),
        len(one_off_ids & set(comms_clicked["Deakin_Supporter_ID"].astype(str))),
        len(one_off_ids & set(portal["Deakin_Supporter_ID"].astype(str)))
    ]
})

overlap_summary["% of One-Off Supporters"] = (
    overlap_summary["Supporters also in One-Off Population"]
    / len(one_off_ids) * 100
).round(1)

display(overlap_summary)

**Observation:** Demographic information is available for the full one-off supporter population. Around 41% of one-off supporters also appear in the Ongoing Commitments dataset, indicating that previous or current regular-giving relationships are common enough to warrant separate analysis.

Communication and portal information is available for smaller subsets of supporters and has more limited historical coverage. Therefore, absence from these datasets should not automatically be interpreted as no engagement.

# 3. Data Cleaning

This section prepares the datasets for reliable analysis and modelling.

Cleaning is limited to issues that could affect supporter matching, time-based analysis, donor classification or model features.

## 3.1 Standardise IDs and Date Fields

Supporter identifiers are converted to a consistent format across all datasets, and relevant date fields are converted from text into datetime format.

This is required for reliable dataset linking and time-based donor analysis.

In [ ]:
# Standardise supporter IDs
for df in datasets.values():
    df["Deakin_Supporter_ID"] = (
        pd.to_numeric(df["Deakin_Supporter_ID"], errors="coerce")
        .astype("Int64")
        .astype(str)
    )

# Convert date fields
one_off["Fulfilment_Date"] = pd.to_datetime(one_off["Fulfilment_Date"])

demographics["Date_Joined_WVA_Database"] = pd.to_datetime(
    demographics["Date_Joined_WVA_Database"]
)

ongoing["Fulfilment_Date"] = pd.to_datetime(ongoing["Fulfilment_Date"])
ongoing["Fulfilment_Ended_Date"] = pd.to_datetime(
    ongoing["Fulfilment_Ended_Date"], errors="coerce"
)
ongoing["Closure_Date"] = pd.to_datetime(
    ongoing["Closure_Date"], errors="coerce"
)

comms_sent["Comm_Sent_Date"] = pd.to_datetime(comms_sent["Comm_Sent_Date"])
comms_clicked["Comm_Clicked_Date"] = pd.to_datetime(comms_clicked["Comm_Clicked_Date"])

portal["Registered_Datetime"] = pd.to_datetime(portal["Registered_Datetime"])
portal["LoggedIn_Datetime"] = pd.to_datetime(portal["LoggedIn_Datetime"])

print("Supporter IDs and date fields standardised successfully.")

## 3.2 Standardise Numeric and Missing Values

Numeric variables are converted into appropriate numerical formats, while blank text values are standardised as missing values.

This ensures that numerical analysis and later feature preparation are based on consistent data formats without removing incomplete records.

In [ ]:
# Convert relevant age and amount fields to numeric
one_off["Total_Paid"] = pd.to_numeric(
    one_off["Total_Paid"], errors="coerce"
)

one_off["Age_At_Acquisition"] = pd.to_numeric(
    one_off["Age_At_Acquisition"], errors="coerce"
)

demographics["Age_At_Joining_WVA"] = pd.to_numeric(
    demographics["Age_At_Joining_WVA"], errors="coerce"
)

ongoing["Payee_Signup_Age"] = pd.to_numeric(
    ongoing["Payee_Signup_Age"], errors="coerce"
)

ongoing["Age_At_Acquisition"] = pd.to_numeric(
    ongoing["Age_At_Acquisition"], errors="coerce"
)

# Standardise blank text values as missing
for df in datasets.values():
    text_columns = df.select_dtypes(include="object").columns

    for column in text_columns:
        df[column] = df[column].str.strip()
        df[column] = df[column].replace("", pd.NA)

print("Numeric and missing values standardised successfully.")

## 3.3 Validate Numeric Values

Key numerical variables are checked for impossible or unusual values before cleaning decisions are made.

Donation amounts are reviewed for non-positive values, while age fields are checked for unrealistic ranges. Unusual high-value donations are not automatically treated as errors.

In [ ]:
# Check donation amounts
print("DONATION AMOUNTS")
print("Minimum:", one_off["Total_Paid"].min())
print("Maximum:", one_off["Total_Paid"].max())
print("Zero or negative values:", (one_off["Total_Paid"] <= 0).sum())


# Check age ranges
age_fields = {
    "One-Off Age at Acquisition": one_off["Age_At_Acquisition"],
    "Age at Joining WVA": demographics["Age_At_Joining_WVA"],
    "Ongoing Signup Age": ongoing["Payee_Signup_Age"],
    "Ongoing Age at Acquisition": ongoing["Age_At_Acquisition"]
}

print("\nAGE CHECKS")

for name, age in age_fields.items():
    print(f"\n{name}")
    print("Minimum:", age.min())
    print("Maximum:", age.max())
    print("Below 0:", (age < 0).sum())
    print("Age 0–17:", ((age >= 0) & (age < 18)).sum())
    print("Above 100:", (age > 100).sum())

**Observation:** All recorded donation amounts are positive, so no amount records are removed at this stage. Age variables contain impossible negative values and a small number of unusually high ages that require further review. Ages below 18 are also examined before any cleaning rule is applied, rather than being automatically treated as errors.

## 3.4 Check Unusual Age Values

Age values below 18 are reviewed before cleaning to distinguish potentially valid young supporters from possible placeholder or incorrect values.

In [ ]:
# Check the distribution of ages below 18

print("One-Off Age at Acquisition:")
print(
    one_off.loc[
        one_off["Age_At_Acquisition"] < 18,
        "Age_At_Acquisition"
    ].value_counts().sort_index()
)

print("\nAge at Joining WVA:")
print(
    demographics.loc[
        demographics["Age_At_Joining_WVA"] < 18,
        "Age_At_Joining_WVA"
    ].value_counts().sort_index()
)

**Observation:** Age fields contain clearly invalid negative values and a small number of values above 100. Ages below 18 are retained because the available client information does not establish that they are invalid.

## 3.5 Clean Invalid Age Values

Clearly invalid age values are replaced with missing values rather than removing supporter records.

Negative ages and ages above 100 are treated as unreliable, while ages from 0 to 100 are retained for further analysis.

In [ ]:
# Age columns to clean
age_columns = [
    (one_off, "Age_At_Acquisition"),
    (demographics, "Age_At_Joining_WVA"),
    (ongoing, "Payee_Signup_Age"),
    (ongoing, "Age_At_Acquisition")
]

# Replace clearly invalid ages with missing values
for df, column in age_columns:
    df.loc[
        (df[column] < 0) | (df[column] > 100),
        column
    ] = np.nan

print("Invalid age values replaced with missing values.")

## 3.6 Check Geographic Categories

State and country values are reviewed for inconsistent labels or formatting.

Consistent categories are important because different spellings of the same location would otherwise be treated as separate groups during analysis and modelling.

In [ ]:
# Check state and country categories

print("ADDRESS STATE")
print(demographics["Address_State"].value_counts(dropna=False))

print("\nADDRESS COUNTRY")
print(demographics["Address_Country"].value_counts(dropna=False).head(20))

**Observation:** Country values contain inconsistent capitalisation and abbreviations. The state field also includes overseas regions, so Australian state standardisation should only be applied to supporters recorded in Australia.

## 3.7 Standardise Country Values

Country labels are standardised to remove differences in capitalisation and common abbreviations.

This prevents the same country from being treated as multiple categories during analysis.

In [ ]:
# ---------------------------------------------------------
# Standardise Country Values
# ---------------------------------------------------------

# Convert country names to a consistent format
demographics["Address_Country"] = (
    demographics["Address_Country"]
    .str.strip()
    .str.upper()
    .replace({

        # Australia
        "AUS": "AUSTRALIA",

        # Australian state codes mistakenly recorded as country
        "NSW": "AUSTRALIA",
        "VIC": "AUSTRALIA",
        "QLD": "AUSTRALIA",
        "WA": "AUSTRALIA",
        "SA": "AUSTRALIA",
        "TAS": "AUSTRALIA",
        "ACT": "AUSTRALIA",
        "NT": "AUSTRALIA",

        # United Kingdom
        "UK": "UNITED KINGDOM",

        # United States
        "USA": "UNITED STATES",
        "UNITED STATES OF AMERICA": "UNITED STATES"
    })
)

# Check the cleaned country categories
print(
    demographics["Address_Country"]
    .value_counts(dropna=False)
    .head(15)
)

## 3.8 Check Australian State Values

State values are reviewed only for supporters recorded in Australia.

This allows Australian state labels to be standardised without incorrectly changing overseas geographic information.

In [ ]:
# Check state values for Australian supporters only
australian_states = demographics.loc[
    demographics["Address_Country"] == "AUSTRALIA",
    "Address_State"
]

print(australian_states.value_counts(dropna=False).head(30))

## 3.9 Inspect Irregular Australian State Values

A small number of non-standard state values are reviewed using their suburb and postcode before applying corrections.

This avoids making unsupported assumptions when standardising geographic information.

In [ ]:
# Standard Australian codes and obvious spelling variations
valid_state_values = [
    "NSW", "VIC", "QLD", "WA", "SA", "TAS", "ACT", "NT",
    "Vic", "Victoria", "New South Wales", "South Australia"
]

# Show unusual state records for Australian supporters
unusual_states = demographics[
    (demographics["Address_Country"] == "AUSTRALIA") &
    (~demographics["Address_State"].isin(valid_state_values)) &
    (demographics["Address_State"].notna())
][
    ["Deakin_Supporter_ID", "Address_Postcode",
     "Address_Suburb", "Address_State"]
]

display(unusual_states)

## 3.10 Correct and Standardise Geographic Values

Australian state labels are standardised into consistent state codes. The small number of irregular address records identified above are corrected using their postcode and location information.

Where suburb, state or country information has shifted into the wrong field, only the affected records are corrected rather than changing the wider dataset.

In [ ]:
# ---------------------------------------------------------
# Correct and Standardise Geographic Values
# ---------------------------------------------------------

# Standardise obvious Australian state-name variations
demographics["Address_State"] = demographics["Address_State"].replace({
    "Vic": "VIC",
    "Victoria": "VIC",
    "New South Wales": "NSW",
    "South Australia": "SA"
})


# Correct the small number of irregular address records
# using their postcode and available location information
corrections = {

    # Australian records
    "60606": {
        "Address_Suburb": "MACKAY",
        "Address_State": "QLD",
        "Address_Country": "AUSTRALIA"
    },

    "6876": {
        "Address_Suburb": "HAMILTON",
        "Address_State": "QLD",
        "Address_Country": "AUSTRALIA"
    },

    "16467": {
        "Address_Suburb": "MEREWETHER",
        "Address_State": "NSW",
        "Address_Country": "AUSTRALIA"
    },

    "26073": {
        "Address_Suburb": "MOSS VALE",
        "Address_State": "NSW",
        "Address_Country": "AUSTRALIA"
    },

    "47361": {
        "Address_Suburb": "CRAIGIEBURN",
        "Address_State": "VIC",
        "Address_Country": "AUSTRALIA"
    },

    "47891": {
        "Address_Suburb": "ROSEBERY",
        "Address_State": "NSW",
        "Address_Country": "AUSTRALIA"
    },

    "55573": {
        "Address_Suburb": "MOONAH",
        "Address_State": "TAS",
        "Address_Country": "AUSTRALIA"
    },

    "56946": {
        "Address_Suburb": "BORONIA",
        "Address_State": "VIC",
        "Address_Country": "AUSTRALIA"
    },

    "35015": {
        "Address_Suburb": "BELCONNEN",
        "Address_State": "ACT",
        "Address_Country": "AUSTRALIA"
    },

    # Overseas records incorrectly recorded as Australia
    "7789": {
        "Address_Suburb": "TAI TAM",
        "Address_State": pd.NA,
        "Address_Country": "HONG KONG"
    },

    "16452": {
        "Address_Suburb": "MASTERTON",
        "Address_State": pd.NA,
        "Address_Country": "NEW ZEALAND"
    },

    "34126": {
        "Address_Suburb": "MUNICH",
        "Address_State": pd.NA,
        "Address_Country": "GERMANY"
    }
}


# Apply corrections only to the identified supporters
for supporter_id, values in corrections.items():

    mask = (
        demographics["Deakin_Supporter_ID"]
        == supporter_id
    )

    for column, value in values.items():
        demographics.loc[mask, column] = value


# Final check of Australian state values
valid_states = [
    "NSW", "VIC", "QLD", "WA",
    "SA", "TAS", "ACT", "NT"
]

invalid_australian_states = demographics[
    (demographics["Address_Country"] == "AUSTRALIA") &
    (demographics["Address_State"].notna()) &
    (~demographics["Address_State"].isin(valid_states))
]

print(
    "Invalid Australian state values remaining:",
    len(invalid_australian_states)
)

print("\nCleaned Australian state counts:")

print(
    demographics.loc[
        demographics["Address_Country"] == "AUSTRALIA",
        "Address_State"
    ].value_counts(dropna=False)
)

## 3.11 Validate Date Relationships

Date fields are checked for logical consistency, because a correctly formatted date can still represent an impossible sequence.

The checks identify ongoing commitments that appear to end or close before they started and portal activity where login dates occur before registration.

In [ ]:
# Check logical relationships between dates

# Ongoing commitment ended before it started
ongoing_end_before_start = (
    ongoing["Fulfilment_Ended_Date"].notna() &
    (ongoing["Fulfilment_Ended_Date"] < ongoing["Fulfilment_Date"])
).sum()

# Ongoing commitment closed before it started
ongoing_close_before_start = (
    ongoing["Closure_Date"].notna() &
    (ongoing["Closure_Date"] < ongoing["Fulfilment_Date"])
).sum()

# Portal login before registration
portal_login_before_registration = (
    portal["LoggedIn_Datetime"] < portal["Registered_Datetime"]
).sum()

print("Ongoing end date before start date:", ongoing_end_before_start)
print("Ongoing closure date before start date:", ongoing_close_before_start)
print("Portal login before registration:", portal_login_before_registration)

## 3.12 Inspect Inconsistent Ongoing Dates

Seven ongoing commitment records contain an end or closure date earlier than the recorded start date.

These records are inspected individually before applying any correction so that valid historical information is not changed without evidence.

In [ ]:
# Identify records with logically inconsistent ongoing dates

invalid_ongoing_dates = ongoing[
    (
        ongoing["Fulfilment_Ended_Date"].notna() &
        (ongoing["Fulfilment_Ended_Date"] < ongoing["Fulfilment_Date"])
    )
    |
    (
        ongoing["Closure_Date"].notna() &
        (ongoing["Closure_Date"] < ongoing["Fulfilment_Date"])
    )
]

print("Records requiring review:", len(invalid_ongoing_dates))

# Portfolio privacy edit: supporter-level preview suppressed.

## 3.13 Flag Inconsistent Ongoing Dates

Seven historical ongoing commitment records contain end or closure dates earlier than their recorded start date.

Because there is insufficient evidence to determine the correct dates, these values are not manually corrected. Instead, the affected records are flagged so that their unreliable timing is excluded from later relationship-status calculations while the commitment history itself is retained.

In [ ]:
# ---------------------------------------------------------
# Flag Inconsistent Ongoing Dates
# ---------------------------------------------------------

ongoing["Date_Consistency_Flag"] = np.where(
    (
        ongoing["Fulfilment_Ended_Date"].notna() &
        (ongoing["Fulfilment_Ended_Date"] < ongoing["Fulfilment_Date"])
    )
    |
    (
        ongoing["Closure_Date"].notna() &
        (ongoing["Closure_Date"] < ongoing["Fulfilment_Date"])
    ),
    "Invalid",
    "Valid"
)

print(ongoing["Date_Consistency_Flag"].value_counts())

## 3.14 Final Data Reliability Audit

A final reliability audit is performed after cleaning to confirm that records have been preserved and the main data-quality issues have been handled appropriately.

The audit checks record counts, identifiers, donation values, age ranges, geographic categories, duplicate records, ongoing-date flags and supporter links across the datasets.

In [ ]:
# ---------------------------------------------------------
# Final Data Reliability Audit
# ---------------------------------------------------------

# Expected row counts from the original loaded datasets
expected_rows = {
    "One-Off Donations": 1900156,
    "Supporter Demographics": 61016,
    "Ongoing Commitments": 60025,
    "Communications Sent": 966410,
    "Communications Clicked": 99198,
    "Portal Engagement": 243316
}

# Valid Australian state and territory codes
valid_states = [
    "NSW", "VIC", "QLD", "WA",
    "SA", "TAS", "ACT", "NT"
]


# ---------------------------------------------------------
# Check supporter links across datasets
# ---------------------------------------------------------

# Demographics contains one record for each supporter
master_supporters = set(
    demographics["Deakin_Supporter_ID"]
)

supporter_link_checks = {
    "One-Off Donations":
        set(one_off["Deakin_Supporter_ID"]) - master_supporters,

    "Ongoing Commitments":
        set(ongoing["Deakin_Supporter_ID"]) - master_supporters,

    "Communications Sent":
        set(comms_sent["Deakin_Supporter_ID"]) - master_supporters,

    "Communications Clicked":
        set(comms_clicked["Deakin_Supporter_ID"]) - master_supporters,

    "Portal Engagement":
        set(portal["Deakin_Supporter_ID"]) - master_supporters
}


# ---------------------------------------------------------
# Create final reliability checks
# ---------------------------------------------------------

checks = {

    # Row counts
    "All original row counts preserved":
        (
            len(one_off) == expected_rows["One-Off Donations"]
            and len(demographics) == expected_rows["Supporter Demographics"]
            and len(ongoing) == expected_rows["Ongoing Commitments"]
            and len(comms_sent) == expected_rows["Communications Sent"]
            and len(comms_clicked) == expected_rows["Communications Clicked"]
            and len(portal) == expected_rows["Portal Engagement"]
        ),

    # Key identifiers
    "Key identifiers remain valid":
        (
            one_off["Deakin_Supporter_ID"].notna().all()
            and demographics["Deakin_Supporter_ID"].is_unique
            and one_off["DMRT_Commitment_Key"].is_unique
            and ongoing["DMRT_Commitment_Key"].is_unique
        ),

    # Donation data
    "Donation dates and amounts valid":
        (
            one_off["Fulfilment_Date"].notna().all()
            and one_off["Total_Paid"].notna().all()
            and (one_off["Total_Paid"] > 0).all()
        ),

    # Age values
    "Invalid age values removed":
        (
            ~(
                (one_off["Age_At_Acquisition"] < 0)
                | (one_off["Age_At_Acquisition"] > 100)
            )
        ).all()
        and
        (
            ~(
                (demographics["Age_At_Joining_WVA"] < 0)
                | (demographics["Age_At_Joining_WVA"] > 100)
            )
        ).all(),

    # Geography
    "Australian state values valid":
        demographics.loc[
            demographics["Address_Country"] == "AUSTRALIA",
            "Address_State"
        ].dropna().isin(valid_states).all(),

    # Duplicate records
    "No exact duplicate records":
        (
            one_off.duplicated().sum() == 0
            and demographics.duplicated().sum() == 0
            and ongoing.duplicated().sum() == 0
        ),

    # Known date-quality issue
    "Ongoing date anomalies flagged":
        (
            ongoing["Date_Consistency_Flag"] == "Invalid"
        ).sum() == 7,

    # Cross-dataset supporter matching
    "All supporter IDs link to demographics":
        all(
            len(unmatched_ids) == 0
            for unmatched_ids in supporter_link_checks.values()
        )
}


# ---------------------------------------------------------
# Display audit results
# ---------------------------------------------------------

audit_results = pd.DataFrame({
    "Check": checks.keys(),
    "Status": [
        "PASS" if result else "REVIEW"
        for result in checks.values()
    ]
})

display(audit_results)

print(
    "\nOverall result:",
    "ALL CHECKS PASSED"
    if all(checks.values())
    else "SOME CHECKS REQUIRE REVIEW"
)

## 3.15 Data Cleaning Summary

The six datasets were cleaned and validated to create a reliable foundation for analysis.

Supporter identifiers and dates were standardised, numerical fields were validated, invalid age values were treated as missing, and geographic inconsistencies were corrected using the available address information.

Seven historical ongoing commitment records contained inconsistent date sequences. These records were retained and flagged rather than manually changing dates that could not be verified.

The final reliability audit confirmed that original record counts were preserved, key identifiers remained valid, donation values and geographic categories were consistent, duplicate records were not introduced, and supporter IDs linked correctly across the datasets.

Known source limitations, including missing age information and uneven historical coverage of engagement datasets, are retained transparently and will be considered during later analysis and modelling.

The cleaned datasets are now ready for analytical population definition.

# 4. Define the Analytical Donor Population

Before creating the repeat-donation target, the donation records must be aligned with the business problem.

The supplied one-off donation table contains different giving situations, so the relevant analytical population is defined before modelling.

## 4.1 Identify Bounceback Donations

Bounceback donations are identified separately because they are associated with existing sponsorship relationships and represent a different giving context from standard one-off donations.

The original donation records are retained while the different donation types are identified for later analytical population selection.

In [ ]:
# Standardise product codes so comparisons are reliable
one_off["Product_Code_Clean"] = (
    one_off["Product_Code"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Bounceback product codes identified by WVA
bounceback_codes = [
    "BB",
    "BB COMSPON",
    "BB COST REC"
]

# Count Bounceback transactions
bounceback_count = one_off[
    "Product_Code_Clean"
].isin(bounceback_codes).sum()

# Count all other one-off transactions
non_bounceback_count = len(one_off) - bounceback_count

print("Total one-off transactions:", f"{len(one_off):,}")
print("Bounceback transactions:", f"{bounceback_count:,}")
print("Other one-off transactions:", f"{non_bounceback_count:,}")

print("\nBounceback product codes:")
print(
    one_off.loc[
        one_off["Product_Code_Clean"].isin(bounceback_codes),
        "Product_Code_Clean"
    ].value_counts()
)

**Observation:** Bounceback accounts for a substantial number of transactions, but all 61,016 supporters also have at least one standard non-Bounceback one-off donation. Therefore, separating Bounceback transactions does not remove supporters from the one-off donor population. It instead creates a cleaner basis for analysing repeat standard one-off giving.

## 4.2 Create the Qualifying One-Off Donation Dataset

Bounceback transactions are separated from standard one-off donations.

A new analytical dataset is created containing only non-Bounceback transactions, while the original donation dataset is retained unchanged.

This provides a more appropriate foundation for analysing repeat one-off giving.

In [ ]:
# Create a separate dataset containing non-Bounceback one-off donations
qualifying_one_off = one_off[
    ~one_off["Product_Code_Clean"].isin(bounceback_codes)
].copy()

print("Original one-off transactions:", f"{len(one_off):,}")
print("Qualifying one-off transactions:", f"{len(qualifying_one_off):,}")

print(
    "Unique qualifying one-off supporters:",
    f"{qualifying_one_off['Deakin_Supporter_ID'].nunique():,}"
)

print(
    "Date range:",
    qualifying_one_off["Fulfilment_Date"].min().date(),
    "to",
    qualifying_one_off["Fulfilment_Date"].max().date()
)

**Observation:** After excluding Bounceback transactions, 1,005,819 qualifying one-off transactions remain. All 61,016 supporters are still represented, confirming that the Bounceback separation removes a different donation context without reducing the supporter population.

## 4.3 Identify the First Qualifying One-Off Donation

Each supporter's first qualifying non-Bounceback one-off donation is identified.

This first donation date is referred to as the **Index Donation Date**. It represents the starting point of the supporter journey used in this analysis.

For example, if a supporter made qualifying one-off donations in 2021, 2022 and 2024, their first donation in 2021 becomes their Index Donation Date.

This reference date is later used to determine the supporter's relationship with WVA at that time and to measure whether another qualifying one-off donation occurs afterwards.

In [ ]:
# Find the first qualifying one-off donation date for each supporter
supporter_index = (
    qualifying_one_off
    .groupby("Deakin_Supporter_ID", as_index=False)
    .agg(
        Index_Donation_Date=("Fulfilment_Date", "min")
    )
)

print("Supporters with an index donation:", f"{len(supporter_index):,}")

print(
    "Earliest index donation:",
    supporter_index["Index_Donation_Date"].min().date()
)

print(
    "Latest index donation:",
    supporter_index["Index_Donation_Date"].max().date()
)

# Portfolio privacy edit: supporter-level preview suppressed.

**Observation:** An Index Donation Date was successfully identified for all 61,016 qualifying one-off supporters, providing one consistent starting point per supporter for the remaining analysis.

## 4.4 Determine Ongoing Relationship at the Index Donation

Each supporter's ongoing commitment history is compared with their Index Donation Date to understand their relationship with WVA when their first qualifying one-off gift occurred.

A new variable, **Relationship_At_Index**, is created with three categories:

- **No Prior Ongoing:** no ongoing commitment had started before the Index Donation Date.
- **Former Ongoing:** an ongoing commitment had existed previously but had ended before the Index Donation Date.
- **Active Ongoing:** an ongoing commitment was still active when the Index Donation occurred.

This distinction is important because an existing active regular giver represents a different supporter situation from a new or previously lapsed supporter.

In [ ]:
# Use only ongoing records with reliable date sequences
ongoing_valid = ongoing[
    ongoing["Date_Consistency_Flag"] == "Valid"
].copy()

# Match each supporter's index donation with their ongoing history
relationship_history = supporter_index.merge(
    ongoing_valid[
        [
            "Deakin_Supporter_ID",
            "Fulfilment_Date",
            "Fulfilment_Ended_Date",
            "Closure_Date"
        ]
    ],
    on="Deakin_Supporter_ID",
    how="left"
)

# Was the ongoing commitment already started by the index donation?
started_before_index = (
    relationship_history["Fulfilment_Date"].notna() &
    (
        relationship_history["Fulfilment_Date"]
        <= relationship_history["Index_Donation_Date"]
    )
)

# Did that commitment finish before the index donation?
ended_before_index = (
    (
        relationship_history["Fulfilment_Ended_Date"].notna() &
        (
            relationship_history["Fulfilment_Ended_Date"]
            < relationship_history["Index_Donation_Date"]
        )
    )
    |
    (
        relationship_history["Closure_Date"].notna() &
        (
            relationship_history["Closure_Date"]
            < relationship_history["Index_Donation_Date"]
        )
    )
)

# Commitment was active at the index donation
relationship_history["Active_At_Index"] = (
    started_before_index & ~ended_before_index
)

# Commitment existed previously but had ended
relationship_history["Former_Before_Index"] = (
    started_before_index & ended_before_index
)

# Summarise to one row per supporter
relationship_summary = (
    relationship_history
    .groupby("Deakin_Supporter_ID", as_index=False)
    .agg(
        Active_At_Index=("Active_At_Index", "max"),
        Former_Before_Index=("Former_Before_Index", "max")
    )
)

# Assign one relationship category
relationship_summary["Relationship_At_Index"] = np.select(
    [
        relationship_summary["Active_At_Index"],
        relationship_summary["Former_Before_Index"]
    ],
    [
        "Active Ongoing",
        "Former Ongoing"
    ],
    default="No Prior Ongoing"
)

# Add relationship status to supporter index table
supporter_index = supporter_index.merge(
    relationship_summary[
        ["Deakin_Supporter_ID", "Relationship_At_Index"]
    ],
    on="Deakin_Supporter_ID",
    how="left"
)

print(
    supporter_index["Relationship_At_Index"]
    .value_counts()
)

print("\nPercentages:")
print(
    (
        supporter_index["Relationship_At_Index"]
        .value_counts(normalize=True)
        .mul(100)
        .round(1)
    )
)

**Observation:** Most supporters (67.4%) had no previous ongoing commitment when they made their first qualifying one-off donation. However, 31.2% were already active ongoing supporters, showing that one-off giving also occurs alongside existing regular-giving relationships. A smaller group (1.4%) had a former ongoing relationship with WVA.

## 4.5 Define the Main Eligible Supporter Population

For the main repeat one-off analysis, supporters who already had an active ongoing commitment at their Index Donation Date are separated from the primary analytical population.

These supporters can still make one-off donations, but they already have an established regular-giving relationship with WVA and therefore represent a different starting situation.

The main eligible population retains:

- **No Prior Ongoing:** supporters without a previous ongoing commitment.
- **Former Ongoing:** supporters whose previous ongoing commitment had already ended.

Active ongoing supporters are not deleted from the project and may still be examined separately where useful.

In [ ]:
# Create the main supporter population for repeat one-off analysis
eligible_supporters = supporter_index[
    supporter_index["Relationship_At_Index"] != "Active Ongoing"
].copy()

print("All qualifying supporters:", f"{len(supporter_index):,}")
print("Active ongoing supporters separated:",
      f"{(supporter_index['Relationship_At_Index'] == 'Active Ongoing').sum():,}")
print("Main eligible supporter population:", f"{len(eligible_supporters):,}")

print("\nEligible population breakdown:")
print(
    eligible_supporters["Relationship_At_Index"].value_counts()
)

## 4.6 Review the Modelling Time Window

The timing of supporters' Index Donations is reviewed before defining the repeat-donation outcome.

A new variable, **Index_Year**, is created from the Index Donation Date to show when each supporter made their first qualifying one-off gift.

Using the complete historical period may include donor behaviour that is less representative of the current environment. At the same time, very recent supporters cannot yet be fairly classified because they have not had enough time to make a repeat donation.

A 24-month observation period is therefore considered so that every supporter included in the final cohort has the same opportunity to demonstrate repeat giving.

In [ ]:
# ---------------------------------------------------------
# Review the Modelling Time Window
# ---------------------------------------------------------

# Create the year of each supporter's Index Donation
eligible_supporters["Index_Year"] = (
    eligible_supporters["Index_Donation_Date"].dt.year
)

# Count eligible supporters by Index Year
index_year_counts = (
    eligible_supporters["Index_Year"]
    .value_counts()
    .sort_index()
)

print("Eligible supporters by index year:")
print(index_year_counts)


# Latest qualifying donation available in the dataset
data_end_date = qualifying_one_off["Fulfilment_Date"].max()

# Latest Index Donation Date that allows
# a complete 24-month repeat-donation observation period
outcome_cutoff = (
    data_end_date - pd.DateOffset(months=24)
)

# Review a recent five-year period
recent_start = (
    data_end_date - pd.DateOffset(years=5)
)

print("\nData end date:", data_end_date.date())

print(
    "Recent five-year start:",
    recent_start.date()
)

print(
    "Latest index date with full 24-month follow-up:",
    outcome_cutoff.date()
)


# Supporters entering during the recent five-year period
recent_supporters = eligible_supporters[
    eligible_supporters["Index_Donation_Date"] >= recent_start
]

# Among them, retain supporters with a complete 24-month follow-up
fully_observed_recent = recent_supporters[
    recent_supporters["Index_Donation_Date"] <= outcome_cutoff
]

print(
    "\nEligible supporters in recent five-year period:",
    f"{len(recent_supporters):,}"
)

print(
    "Recent supporters with full 24-month follow-up:",
    f"{len(fully_observed_recent):,}"
)

## 4.7 Compare Candidate Historical Cohorts

Alternative historical starting points are compared before defining the final modelling cohort.

Using a longer history provides more supporters for model development, but very old donor behaviour may be less representative of the current fundraising environment. A shorter history is more recent but provides fewer observations.

All candidate cohorts use the same end date of 17 March 2024 so that every included supporter has a complete 24-month follow-up period.

In [ ]:
# ---------------------------------------------------------
# Compare Candidate Historical Cohorts
# ---------------------------------------------------------

# Compare different historical starting points
candidate_starts = {
    "2013–2024": pd.Timestamp("2013-03-17"),
    "2016–2024": pd.Timestamp("2016-03-17"),
    "2019–2024": pd.Timestamp("2019-03-17"),
    "2021–2024": pd.Timestamp("2021-03-17")
}

cohort_comparison = []

for label, start_date in candidate_starts.items():

    # Keep eligible supporters within each candidate period
    cohort = eligible_supporters[
        (eligible_supporters["Index_Donation_Date"] >= start_date) &
        (eligible_supporters["Index_Donation_Date"] <= outcome_cutoff)
    ]

    cohort_comparison.append({
        "Period": label,
        "Start_Date": start_date.date(),
        "End_Date": outcome_cutoff.date(),
        "Supporters": len(cohort)
    })

# Convert results into a simple comparison table
cohort_comparison = pd.DataFrame(cohort_comparison)

display(cohort_comparison)

**Observation:** Extending the historical starting point increases the available supporter population from 19,194 in the 2021–2024 cohort to 25,882 in the 2013–2024 cohort. However, a larger historical sample does not necessarily provide a better model because older donor behaviour may be less representative of current supporters. The effect of using older training history will therefore be evaluated during predictive modelling rather than determined by sample size alone.

## 4.8 Finalise the Main Analytical Cohort

The main analytical cohort includes eligible supporters whose Index Donation occurred between 17 March 2021 and 17 March 2024.

This recent period provides a large and relevant population for understanding current one-off donor behaviour while ensuring that every supporter has a complete 24-month follow-up period.

Earlier eligible supporters are not discarded. Their historical records remain available and can later be tested as additional model-training data to determine whether older behaviour improves prediction.

In [ ]:
# Final modelling period
model_start = pd.Timestamp("2021-03-17")
model_end = pd.Timestamp("2024-03-17")

# Create final modelling cohort
modelling_cohort = eligible_supporters[
    (eligible_supporters["Index_Donation_Date"] >= model_start) &
    (eligible_supporters["Index_Donation_Date"] <= model_end)
].copy()

print("Final modelling cohort:", f"{len(modelling_cohort):,}")

print("\nIndex donation period:")
print("Start:", modelling_cohort["Index_Donation_Date"].min().date())
print("End:", modelling_cohort["Index_Donation_Date"].max().date())

print("\nRelationship breakdown:")
print(modelling_cohort["Relationship_At_Index"].value_counts())

## 4.9 Create the Repeat-Donor Target

For each supporter in the main analytical cohort, repeat giving is measured during the 24 months following their Index Donation Date.

A supporter is classified as:

- **Repeat_Donor = 1:** at least one additional qualifying one-off donation was made on a later date within 24 months.
- **Repeat_Donor = 0:** no additional qualifying one-off donation was made during the 24-month period.

Additional transactions made on the same day as the Index Donation are treated as part of the original giving occasion and do not count as repeat giving.

The date of the first repeat donation and the number of days taken to repeat are also retained for later timing analysis.

In [ ]:
# ---------------------------------------------------------
# Create the Repeat-Donor Target
# ---------------------------------------------------------

# Create the end of each supporter's 24-month follow-up period
modelling_cohort["Followup_End_Date"] = (
    modelling_cohort["Index_Donation_Date"]
    + pd.DateOffset(months=24)
)


# Match the qualifying donation history
# to supporters in the analytical cohort
future_donations = qualifying_one_off.merge(
    modelling_cohort[
        [
            "Deakin_Supporter_ID",
            "Index_Donation_Date",
            "Followup_End_Date"
        ]
    ],
    on="Deakin_Supporter_ID",
    how="inner"
)


# Keep only donations that occurred:
# 1. after the Index Donation Date
# 2. within the 24-month follow-up period
future_donations = future_donations[
    (
        future_donations["Fulfilment_Date"]
        > future_donations["Index_Donation_Date"]
    )
    &
    (
        future_donations["Fulfilment_Date"]
        <= future_donations["Followup_End_Date"]
    )
]


# Find the first qualifying repeat donation for each supporter
first_repeat = (
    future_donations
    .groupby("Deakin_Supporter_ID", as_index=False)
    .agg(
        Repeat_Donation_Date=("Fulfilment_Date", "min")
    )
)


# Remove these columns first if the cell is rerun
modelling_cohort = modelling_cohort.drop(
    columns=[
        "Repeat_Donation_Date",
        "Repeat_Donor",
        "Days_to_Repeat"
    ],
    errors="ignore"
)


# Add the first repeat donation date
modelling_cohort = modelling_cohort.merge(
    first_repeat,
    on="Deakin_Supporter_ID",
    how="left"
)


# Create the binary target variable
# 1 = repeated within 24 months
# 0 = did not repeat within 24 months
modelling_cohort["Repeat_Donor"] = (
    modelling_cohort["Repeat_Donation_Date"]
    .notna()
    .astype(int)
)


# Calculate how many days it took to make the first repeat donation
modelling_cohort["Days_to_Repeat"] = (
    modelling_cohort["Repeat_Donation_Date"]
    - modelling_cohort["Index_Donation_Date"]
).dt.days


# Summarise the target
repeat_count = modelling_cohort["Repeat_Donor"].sum()
non_repeat_count = len(modelling_cohort) - repeat_count
repeat_rate = modelling_cohort["Repeat_Donor"].mean() * 100

print(
    "Total supporters:",
    f"{len(modelling_cohort):,}"
)

print(
    "Repeat donors:",
    f"{repeat_count:,}"
)

print(
    "Non-repeat donors:",
    f"{non_repeat_count:,}"
)

print(
    "Repeat rate:",
    f"{repeat_rate:.1f}%"
)

print(
    "\nMedian days to repeat:",
    modelling_cohort.loc[
        modelling_cohort["Repeat_Donor"] == 1,
        "Days_to_Repeat"
    ].median()
)

**Observation:** Of the 19,194 supporters in the main analytical cohort, 3,194 (16.6%) made another qualifying one-off donation within 24 months, while 16,000 did not. Among repeat donors, the median time to the first repeat donation was 240 days, indicating that repeat giving often occurs several months after the initial donation.

## 4.10 Validate the Repeat-Donor Target

The final target variable is validated to ensure that repeat donations occur after the index donation and within the defined 24-month follow-up period.

This confirms that the modelling outcome is logically consistent before further analysis.

In [ ]:
# ---------------------------------------------------------
# Validate the Repeat-Donor Target
# ---------------------------------------------------------

# Check target values
print("Target values:")
print(
    modelling_cohort["Repeat_Donor"]
    .value_counts()
)


# Repeat donations must occur after the Index Donation Date
invalid_before_index = modelling_cohort[
    (modelling_cohort["Repeat_Donor"] == 1) &
    (
        modelling_cohort["Repeat_Donation_Date"]
        <= modelling_cohort["Index_Donation_Date"]
    )
]


# Repeat donations must occur within the 24-month window
invalid_after_followup = modelling_cohort[
    (modelling_cohort["Repeat_Donor"] == 1) &
    (
        modelling_cohort["Repeat_Donation_Date"]
        > modelling_cohort["Followup_End_Date"]
    )
]


# Check the Days_to_Repeat range
repeat_days = modelling_cohort.loc[
    modelling_cohort["Repeat_Donor"] == 1,
    "Days_to_Repeat"
]


print(
    "\nRepeat before/on index date:",
    len(invalid_before_index)
)

print(
    "Repeat after follow-up window:",
    len(invalid_after_followup)
)

print(
    "\nMinimum days to repeat:",
    repeat_days.min()
)

print(
    "Maximum days to repeat:",
    repeat_days.max()
)

print(
    "\nMissing target values:",
    modelling_cohort["Repeat_Donor"].isna().sum()
)

**Observation:** The repeat-donor target passed all validation checks. No repeat donations occurred on or before the Index Donation Date, none fell outside the 24-month follow-up period, and the target contains no missing values. The observed repeat timing ranges from 1 to 731 days.

## 4.11 Check Index-Date Transaction Uniqueness

The index donation date is checked for multiple qualifying transactions by the same supporter.

This ensures that first-gift characteristics are not selected arbitrarily when more than one transaction occurred on the supporter's initial donation date.

In [ ]:
# Get qualifying transactions occurring on each supporter's index date
index_transactions = qualifying_one_off.merge(
    modelling_cohort[
        ["Deakin_Supporter_ID", "Index_Donation_Date"]
    ],
    on="Deakin_Supporter_ID",
    how="inner"
)

index_transactions = index_transactions[
    index_transactions["Fulfilment_Date"] ==
    index_transactions["Index_Donation_Date"]
]

# Count index-date transactions per supporter
index_transaction_counts = (
    index_transactions
    .groupby("Deakin_Supporter_ID")
    .size()
)

print(
    "Supporters with exactly 1 transaction on index date:",
    f"{(index_transaction_counts == 1).sum():,}"
)

print(
    "Supporters with multiple transactions on index date:",
    f"{(index_transaction_counts > 1).sum():,}"
)

print(
    "Maximum transactions for one supporter on index date:",
    index_transaction_counts.max()
)

**Observation:** Most supporters had a single transaction on their Index Donation Date, but 2,321 supporters (12.1%) made multiple qualifying transactions on the same day. The maximum was 91 transactions for one supporter. These transactions should therefore be treated as one initial giving occasion rather than selecting a single transaction arbitrarily.

## 4.12 Check Same-Day Index Transaction Consistency

Supporters with multiple qualifying transactions on their index donation date are examined to determine whether those transactions share the same channel and product information.

This prevents first-gift characteristics from being selected arbitrarily when multiple transactions occurred during the same initial giving occasion.

In [ ]:
# Focus on supporters with multiple transactions on their index date
multi_index_ids = index_transaction_counts[
    index_transaction_counts > 1
].index

multi_index_transactions = index_transactions[
    index_transactions["Deakin_Supporter_ID"].isin(multi_index_ids)
]

# Check how many different values appear within each supporter's index date
same_day_consistency = (
    multi_index_transactions
    .groupby("Deakin_Supporter_ID")
    .agg(
        Transactions=("DMRT_Commitment_Key", "count"),
        Unique_Channels=("Attribution_Channel", "nunique"),
        Unique_Product_Categories=("Product_Category", "nunique"),
        Unique_Product_Codes=("Product_Code_Clean", "nunique")
    )
)

print("Multiple index-date supporters:", f"{len(same_day_consistency):,}")

print(
    "Different channels on same index date:",
    f"{(same_day_consistency['Unique_Channels'] > 1).sum():,}"
)

print(
    "Different product categories on same index date:",
    f"{(same_day_consistency['Unique_Product_Categories'] > 1).sum():,}"
)

print(
    "Different product codes on same index date:",
    f"{(same_day_consistency['Unique_Product_Codes'] > 1).sum():,}"
)

# Portfolio privacy edit: supporter-level preview suppressed.

**Observation:** Among the 2,321 supporters with multiple transactions on their Index Donation Date, only 5 used more than one donation channel. However, 1,582 (68.2%) donated across multiple product categories and product codes. The first giving occasion should therefore combine same-day transactions, while product information requires a consistent rule rather than selecting one transaction arbitrarily.

## 4.13 Create the First Giving Occasion

All qualifying transactions made by a supporter on their Index Donation Date are combined into one **First Giving Occasion**.

Several new supporter-level variables are created, including the total amount donated, number of transactions, number of products and donation channel.

Where several products were supported on the same day, the product linked to the largest transaction is treated as the **Primary Product**. The campaign, activity and catalogue information associated with that transaction are also retained.

This approach captures the complete first-day giving behaviour without selecting one transaction arbitrarily and creates useful variables for later exploratory analysis and predictive modelling.

In [ ]:
# ---------------------------------------------------------
# Create the First Giving Occasion
# ---------------------------------------------------------

# Sort first-day transactions so the largest donation
# for each supporter appears first
first_gift_transactions_sorted = index_transactions.sort_values(
    [
        "Deakin_Supporter_ID",
        "Total_Paid"
    ],
    ascending=[True, False]
)


# Use the largest transaction to represent the primary
# product and its associated campaign information
first_gift_primary = (
    first_gift_transactions_sorted
    .drop_duplicates("Deakin_Supporter_ID")
    [
        [
            "Deakin_Supporter_ID",
            "Product_Category",
            "Product_Code_Clean",
            "Campaign_Name",
            "Activity_Name",
            "Catalogue_Item_Description"
        ]
    ]
    .rename(
        columns={
            "Product_Category":
                "First_Gift_Primary_Product",

            "Product_Code_Clean":
                "First_Gift_Primary_Product_Code",

            "Campaign_Name":
                "First_Gift_Campaign",

            "Activity_Name":
                "First_Gift_Activity",

            "Catalogue_Item_Description":
                "First_Gift_Catalogue_Item"
        }
    )
)


# Combine all transactions made on the first donation day
first_gift = (
    index_transactions
    .groupby(
        "Deakin_Supporter_ID",
        as_index=False
    )
    .agg(
        First_Gift_Total_Amount=(
            "Total_Paid",
            "sum"
        ),

        First_Gift_Transaction_Count=(
            "DMRT_Commitment_Key",
            "count"
        ),

        First_Gift_Channel_Count=(
            "Attribution_Channel",
            "nunique"
        ),

        First_Gift_Product_Count=(
            "Product_Code_Clean",
            "nunique"
        ),

        First_Gift_Channel_Temp=(
            "Attribution_Channel",
            "first"
        ),

        Age_At_First_Gift=(
            "Age_At_Acquisition",
            "first"
        )
    )
)


# Keep the channel when all first-day transactions
# used the same channel
first_gift["First_Gift_Channel"] = np.where(
    first_gift["First_Gift_Channel_Count"] == 1,
    first_gift["First_Gift_Channel_Temp"],
    "MULTIPLE"
)


# Flag supporters with more than one first-day transaction
first_gift["First_Gift_Multiple_Transactions"] = (
    first_gift["First_Gift_Transaction_Count"] > 1
).astype(int)


# Flag supporters who supported multiple products
# during their first giving occasion
first_gift["First_Gift_Multiple_Products"] = (
    first_gift["First_Gift_Product_Count"] > 1
).astype(int)


# Remove temporary helper column
first_gift = first_gift.drop(
    columns="First_Gift_Channel_Temp"
)


# Add primary product, campaign, activity and catalogue information
first_gift = first_gift.merge(
    first_gift_primary,
    on="Deakin_Supporter_ID",
    how="left"
)


# Review the new supporter-level first-gift dataset
print(
    "First-gift records:",
    f"{len(first_gift):,}"
)

print(
    "Multiple transactions on first gift date:",
    f"{first_gift['First_Gift_Multiple_Transactions'].sum():,}"
)

print(
    "Multiple products on first gift date:",
    f"{first_gift['First_Gift_Multiple_Products'].sum():,}"
)

print(
    "Multiple channels on first gift date:",
    f"{(first_gift['First_Gift_Channel'] == 'MULTIPLE').sum():,}"
)

# Portfolio privacy edit: supporter-level preview suppressed.

**Observation:** The first giving occasion was successfully created for all 19,194 supporters. Most supporters made a single transaction and supported one product, while 2,321 made multiple transactions and 1,582 supported multiple products on their first giving date. Campaign, activity and catalogue information were also retained to provide additional context for later EDA and modelling.

## 4.14 Build the Supporter-Level Modelling Table

The first-giving characteristics are combined with the supporter relationship information and repeat-donation outcome to create one record per supporter.

Technical Index terminology is replaced with clearer business-friendly names, such as **First_Gift_Date** and **Relationship_At_First_Gift**.

The resulting table provides the supporter-level foundation for later EDA and modelling. Future outcome variables are retained for analysis and evaluation but will not be used as predictors.

In [ ]:
# ---------------------------------------------------------
# Build the Supporter-Level Modelling Table
# ---------------------------------------------------------

# Rename technical Index fields using clearer first-gift terminology
model_data = modelling_cohort.rename(
    columns={
        "Index_Donation_Date": "First_Gift_Date",
        "Relationship_At_Index": "Relationship_At_First_Gift",
        "Index_Year": "First_Gift_Year",
        "Followup_End_Date": "Repeat_Window_End_Date"
    }
).copy()


# Add the first-giving characteristics created in Section 4.13
model_data = model_data.merge(
    first_gift,
    on="Deakin_Supporter_ID",
    how="left",
    validate="one_to_one"
)


# Check that one row still represents one supporter
print(
    "Supporter-level records:",
    f"{len(model_data):,}"
)

print(
    "Unique supporters:",
    f"{model_data['Deakin_Supporter_ID'].nunique():,}"
)

print(
    "Missing first-gift information:",
    model_data["First_Gift_Total_Amount"].isna().sum()
)

print(
    "Missing repeat target:",
    model_data["Repeat_Donor"].isna().sum()
)


# Display example supporter records
# Portfolio privacy edit: supporter-level preview suppressed.

**Observation:** The supporter-level table contains 19,194 records representing 19,194 unique supporters, with no missing first-gift amount or repeat-donor target information. First-gift characteristics and repeat outcomes are now aligned within a single supporter-level dataset.

## 4.15 Validate the Supporter-Level Dataset

The supporter-level analytical dataset is checked to confirm that each supporter appears once, first-gift information is complete, and the repeat-donor target remains logically consistent.

Outcome information is retained for evaluation but will not be used as model predictors.

In [ ]:
# ---------------------------------------------------------
# Validate the Supporter-Level Dataset
# ---------------------------------------------------------

print(
    "Total records:",
    f"{len(model_data):,}"
)

print(
    "Unique supporters:",
    f"{model_data['Deakin_Supporter_ID'].nunique():,}"
)

print(
    "Duplicate supporter rows:",
    model_data["Deakin_Supporter_ID"].duplicated().sum()
)

print(
    "Missing first gift dates:",
    model_data["First_Gift_Date"].isna().sum()
)

print(
    "Missing first gift amounts:",
    model_data["First_Gift_Total_Amount"].isna().sum()
)

print(
    "Invalid first gift amounts:",
    (model_data["First_Gift_Total_Amount"] <= 0).sum()
)

print(
    "Missing target values:",
    model_data["Repeat_Donor"].isna().sum()
)

print("\nTarget distribution:")
print(
    model_data["Repeat_Donor"].value_counts()
)

print(
    "\nFirst gift date range:",
    model_data["First_Gift_Date"].min().date(),
    "to",
    model_data["First_Gift_Date"].max().date()
)

**Observation:** All supporter-level validation checks passed. The dataset contains 19,194 unique supporters with no duplicate records, missing first-gift dates, invalid first-gift amounts or missing target values. The repeat-donor target remains correctly distributed between 3,194 repeat donors and 16,000 non-repeat donors.

## 4.16 Analytical Population Summary

The main analytical population contains 19,194 supporters whose first qualifying non-Bounceback one-off donation occurred between 17 March 2021 and 17 March 2024.

Supporters who already had an active ongoing commitment at their first qualifying gift were separated from the main population. Supporters with no prior ongoing relationship and former ongoing supporters were retained.

Multiple qualifying transactions occurring on the first donation date were combined into one First Giving Occasion. This created supporter-level characteristics including first-gift amount, transaction count, channel, product, campaign and activity information.

Repeat giving was defined as another qualifying one-off donation made on a later date within 24 months of the first gift. This produced 3,194 repeat donors and 16,000 non-repeat donors, representing a repeat rate of 16.6%.

The 2021–2024 population will be used for the main exploratory and business analysis. Earlier eligible supporter history remains available for later testing where additional historical training data may improve predictive modelling.

# 5. Build the Analytical Dataset

Relevant information from the supplied datasets is combined into a single supporter-level analytical dataset before exploratory analysis.

Raw event tables are not joined directly because this could create duplicate supporter records. Instead, useful information is prepared at supporter level before merging.

## 5.1 Add Demographic Information

Demographic variables are added to help describe supporter characteristics and explore differences between repeat and non-repeat donors.

Some demographic fields represent the supporter's current recorded profile rather than necessarily their profile at the time of the first gift. They are therefore initially retained for exploratory analysis, while their suitability for predictive modelling will be assessed separately.

### 5.1.1 Merge Demographic Variables

Selected demographic information is linked to the analytical population using the unique supporter identifier.

These variables are initially included for exploratory analysis. Their suitability for predictive modelling will be reviewed later based on data quality and whether the information would have been available at the time of the first gift.

In [ ]:
# ---------------------------------------------------------
# Merge Demographic Variables
# ---------------------------------------------------------

# Start with the supporter-level dataset created in Section 4
analysis_data = model_data.copy()

# Add selected demographic information
analysis_data = analysis_data.merge(
    demographics[
        [
            "Deakin_Supporter_ID",
            "Date_Joined_WVA_Database",
            "Age_At_Joining_WVA",
            "Address_State",
            "Address_Country",
            "Helix_Community",
            "Helix_Persona",
            "Lewers_Segment_Description"
        ]
    ],
    on="Deakin_Supporter_ID",
    how="left",
    validate="one_to_one"
)

# Confirm that one row still represents one supporter
print("Analytical records:", f"{len(analysis_data):,}")
print(
    "Unique supporters:",
    f"{analysis_data['Deakin_Supporter_ID'].nunique():,}"
)

# Review missing demographic information
print("\nMissing demographic information:")

print(
    analysis_data[
        [
            "Date_Joined_WVA_Database",
            "Age_At_Joining_WVA",
            "Address_State",
            "Address_Country",
            "Helix_Community",
            "Helix_Persona",
            "Lewers_Segment_Description"
        ]
    ]
    .isna()
    .sum()
)

### 5.1.2 Measure WVA History Before the First Gift

The time between joining the WVA database and making the first qualifying one-off gift is calculated.

This provides a measure of how long the supporter had already been known to WVA before the first gift. The values are checked for logical inconsistencies before being used in later analysis.

In [ ]:
# ---------------------------------------------------------
# Measure WVA History Before the First Gift
# ---------------------------------------------------------

# Calculate how many days the supporter had been
# in the WVA database before their first qualifying gift
analysis_data["Days_With_WVA_Before_First_Gift"] = (
    analysis_data["First_Gift_Date"] -
    analysis_data["Date_Joined_WVA_Database"]
).dt.days


# Check whether any recorded join dates occur
# after the first qualifying gift
print(
    "Supporters with join date after first gift:",
    (
        analysis_data[
            "Days_With_WVA_Before_First_Gift"
        ] < 0
    ).sum()
)


# Check how many supporters joined the database
# on exactly the same day as their first gift
print(
    "Supporters who joined on first gift date:",
    (
        analysis_data[
            "Days_With_WVA_Before_First_Gift"
        ] == 0
    ).sum()
)


# Review the range of the derived variable
print(
    "Minimum days before first gift:",
    analysis_data[
        "Days_With_WVA_Before_First_Gift"
    ].min()
)

print(
    "Maximum days before first gift:",
    analysis_data[
        "Days_With_WVA_Before_First_Gift"
    ].max()
)

print(
    "Median days before first gift:",
    analysis_data[
        "Days_With_WVA_Before_First_Gift"
    ].median()
)

### 5.1.3 Prepare WVA History Variable

A small number of supporters have a recorded WVA database join date after their first qualifying gift, producing an invalid negative history period.

These derived values are treated as missing rather than manually corrected. Supporters are also grouped according to whether they joined the WVA database on their first gift date or were already known to WVA beforehand.

In [ ]:
# Remove logically invalid negative history values
analysis_data.loc[
    analysis_data["Days_With_WVA_Before_First_Gift"] < 0,
    "Days_With_WVA_Before_First_Gift"
] = np.nan

# Create a simple supporter-history category
analysis_data["WVA_History_Before_First_Gift"] = np.select(
    [
        analysis_data["Days_With_WVA_Before_First_Gift"] == 0,
        analysis_data["Days_With_WVA_Before_First_Gift"] > 0
    ],
    [
        "Joined on First Gift Date",
        "Known to WVA Before First Gift"
    ],
    default="Unknown / Inconsistent"
)

print(
    analysis_data["WVA_History_Before_First_Gift"]
    .value_counts()
)

print("\nMissing history duration:",
      analysis_data["Days_With_WVA_Before_First_Gift"].isna().sum())

**Observation:** Most supporters (17,136) joined the WVA database on the same date as their first qualifying one-off gift, while 1,699 were already known to WVA beforehand. A further 359 supporters had inconsistent join-date histories, so their derived history duration was treated as missing rather than manually corrected.

## 5.2 Add Previous Ongoing Relationship History

Previous ongoing support is summarised using only commitment information available before the supporter's first qualifying one-off gift.

### 5.2.1 Summarise Previous Ongoing Commitments

The number of completed ongoing commitments before the first gift is calculated for each supporter.

This provides additional information about the strength of the supporter's previous relationship with WVA without using future activity.

In [ ]:
# Use only ongoing records with reliable date sequences
ongoing_history = ongoing[
    ongoing["Date_Consistency_Flag"] == "Valid"
].copy()

# Match ongoing records with each supporter's first gift date
prior_ongoing = ongoing_history.merge(
    analysis_data[
        ["Deakin_Supporter_ID", "First_Gift_Date"]
    ],
    on="Deakin_Supporter_ID",
    how="inner"
)

# A commitment counts as previous if:
# 1. it started before or on the first gift date
# 2. it had already ended/closed before the first gift
prior_ongoing["Ended_Before_First_Gift"] = (
    (
        prior_ongoing["Fulfilment_Ended_Date"].notna() &
        (
            prior_ongoing["Fulfilment_Ended_Date"] <
            prior_ongoing["First_Gift_Date"]
        )
    )
    |
    (
        prior_ongoing["Closure_Date"].notna() &
        (
            prior_ongoing["Closure_Date"] <
            prior_ongoing["First_Gift_Date"]
        )
    )
)

prior_ongoing = prior_ongoing[
    (prior_ongoing["Fulfilment_Date"] <= prior_ongoing["First_Gift_Date"]) &
    (prior_ongoing["Ended_Before_First_Gift"])
]

# Count previous ongoing commitments per supporter
prior_ongoing_summary = (
    prior_ongoing
    .groupby("Deakin_Supporter_ID", as_index=False)
    .agg(
        Previous_Ongoing_Commitment_Count=(
            "DMRT_Commitment_Key", "nunique"
        )
    )
)

# Add to analytical dataset
analysis_data = analysis_data.merge(
    prior_ongoing_summary,
    on="Deakin_Supporter_ID",
    how="left",
    validate="one_to_one"
)

# Supporters without previous commitments receive 0
analysis_data["Previous_Ongoing_Commitment_Count"] = (
    analysis_data["Previous_Ongoing_Commitment_Count"]
    .fillna(0)
    .astype(int)
)

print(
    analysis_data["Previous_Ongoing_Commitment_Count"]
    .value_counts()
    .sort_index()
    .head(15)
)

print(
    "\nSupporters with at least one previous ongoing commitment:",
    (analysis_data["Previous_Ongoing_Commitment_Count"] > 0).sum()
)

**Observation:** Most supporters (18,795) had no previous completed ongoing commitment before their first qualifying one-off gift. However, 399 supporters had at least one previous ongoing commitment, indicating a smaller group with an established prior giving relationship with WVA.

### 5.2.2 Validate Previous Ongoing Relationship

The derived previous commitment count is compared with the previously assigned relationship status at the first gift.

This confirms that former ongoing supporters are identified consistently before ongoing-history variables are used in exploratory analysis.

In [ ]:
# Compare relationship status with previous commitment history

relationship_check = pd.crosstab(
    analysis_data["Relationship_At_First_Gift"],
    analysis_data["Previous_Ongoing_Commitment_Count"] > 0
)

relationship_check.columns = [
    "No Previous Commitment",
    "Has Previous Commitment"
]

display(relationship_check)

print(
    "\nFormer Ongoing supporters:",
    (analysis_data["Relationship_At_First_Gift"] == "Former Ongoing").sum()
)

print(
    "Supporters with previous ongoing commitments:",
    (analysis_data["Previous_Ongoing_Commitment_Count"] > 0).sum()
)

**Observation:** The previous ongoing relationship variables are fully consistent. All 399 supporters classified as Former Ongoing had at least one previous completed ongoing commitment, while all 18,795 supporters classified as No Prior Ongoing had none.

## 5.3 Add Communication History

Communication activity is considered only when it occurred before the supporter's first qualifying one-off gift.

This prevents future communication activity from leaking information about the repeat-giving outcome into the analysis.

### 5.3.1 Check Communication Coverage Before the First Gift

The sent and clicked communication datasets are reviewed to determine how much historical communication information is available for the analytical population before their first gift.

In [ ]:
# Match sent communications with supporters in the analytical population
sent_history = comms_sent.merge(
    analysis_data[
        ["Deakin_Supporter_ID", "First_Gift_Date"]
    ],
    on="Deakin_Supporter_ID",
    how="inner"
)

# Keep only communications sent before the first gift
sent_before_first_gift = sent_history[
    sent_history["Comm_Sent_Date"] < sent_history["First_Gift_Date"]
]

# Match clicked communications
clicked_history = comms_clicked.merge(
    analysis_data[
        ["Deakin_Supporter_ID", "First_Gift_Date"]
    ],
    on="Deakin_Supporter_ID",
    how="inner"
)

# Keep only clicks before the first gift
clicked_before_first_gift = clicked_history[
    clicked_history["Comm_Clicked_Date"] < clicked_history["First_Gift_Date"]
]

print(
    "Supporters with sent communications before first gift:",
    f"{sent_before_first_gift['Deakin_Supporter_ID'].nunique():,}"
)

print(
    "Supporters with clicked communications before first gift:",
    f"{clicked_before_first_gift['Deakin_Supporter_ID'].nunique():,}"
)

print("\nSent communication date range before first gift:")
print(
    sent_before_first_gift["Comm_Sent_Date"].min(),
    "to",
    sent_before_first_gift["Comm_Sent_Date"].max()
)

print("\nClicked communication date range before first gift:")
print(
    clicked_before_first_gift["Comm_Clicked_Date"].min(),
    "to",
    clicked_before_first_gift["Comm_Clicked_Date"].max()
)

**Finding:** Pre-first-gift communication coverage is very limited within the analytical population. Only 521 supporters have recorded sent communications and 204 have recorded clicks before their first gift. Click records also end in 2019, before the modelling cohort begins.

Therefore, communication activity is retained as a data-coverage finding but is not used as a core predictive feature because absence of a record cannot reliably be interpreted as absence of engagement.

## 5.4 Add Portal Engagement History

Portal engagement is considered only when activity occurred before the supporter's first qualifying one-off gift.

### 5.4.1 Check Portal Coverage Before the First Gift

Portal records are reviewed to determine how much historical engagement information is available for the analytical population before their first gift.

The coverage is assessed before deciding whether portal activity should be included in the analytical dataset.

In [ ]:
# Match portal activity with supporters in the analytical population
portal_history = portal.merge(
    analysis_data[
        ["Deakin_Supporter_ID", "First_Gift_Date"]
    ],
    on="Deakin_Supporter_ID",
    how="inner"
)

# Keep portal activity that occurred before the first gift
portal_before_first_gift = portal_history[
    portal_history["LoggedIn_Datetime"] <
    portal_history["First_Gift_Date"]
].copy()

print(
    "Supporters with portal activity before first gift:",
    f"{portal_before_first_gift['Deakin_Supporter_ID'].nunique():,}"
)

print(
    "Total pre-first-gift portal records:",
    f"{len(portal_before_first_gift):,}"
)

print("\nPortal activity date range before first gift:")
print(
    portal_before_first_gift["LoggedIn_Datetime"].min(),
    "to",
    portal_before_first_gift["LoggedIn_Datetime"].max()
)

**Finding:** Only 172 supporters had recorded portal activity before their first qualifying gift, representing less than 1% of the analytical population.

Because this coverage is too limited to represent the wider supporter population reliably, portal activity is not used as a core predictive feature or detailed EDA variable. The limited coverage also highlights an opportunity for more complete digital-engagement tracking in future predictive work.

## 5.5 Add First-Gift Acquisition Information

Acquisition segment information recorded at the first qualifying gift is added to the supporter-level analytical dataset.

These variables describe the WVA business area associated with the initial donation and may help explain differences in repeat-giving behaviour.

In [ ]:
# Get acquisition information from the largest transaction
# on the supporter's first gift date
first_gift_acquisition = (
    first_gift_transactions_sorted
    .drop_duplicates("Deakin_Supporter_ID")
    [
        [
            "Deakin_Supporter_ID",
            "Acquisition_Segment_Business_Unit",
            "Acquisition_Segment_Team",
            "Acquisition_Segment_Sub_Team"
        ]
    ]
    .rename(
        columns={
            "Acquisition_Segment_Business_Unit":
                "First_Gift_Acquisition_Business_Unit",
            "Acquisition_Segment_Team":
                "First_Gift_Acquisition_Team",
            "Acquisition_Segment_Sub_Team":
                "First_Gift_Acquisition_Sub_Team"
        }
    )
)

# Add to analytical dataset
analysis_data = analysis_data.merge(
    first_gift_acquisition,
    on="Deakin_Supporter_ID",
    how="left",
    validate="one_to_one"
)

print("Analytical records:", f"{len(analysis_data):,}")

print("\nMissing acquisition information:")
print(
    analysis_data[
        [
            "First_Gift_Acquisition_Business_Unit",
            "First_Gift_Acquisition_Team",
            "First_Gift_Acquisition_Sub_Team"
        ]
    ].isna().sum()
)

**Observation:** First-gift acquisition information is available for all 19,194 supporters, with no missing values across business unit, team or sub-team. These variables can therefore be explored further as potential drivers of repeat giving.

### 5.5.1 Validate First-Gift Acquisition Consistency

For supporters with multiple transactions on their first gift date, acquisition segment information is checked for consistency.

This confirms whether the acquisition details attached to the largest first-day transaction provide a reliable representation of the initial giving occasion.

In [ ]:
# Check acquisition consistency across multiple first-day transactions

acquisition_consistency = (
    multi_index_transactions
    .groupby("Deakin_Supporter_ID")
    .agg(
        Unique_Business_Units=(
            "Acquisition_Segment_Business_Unit", "nunique"
        ),
        Unique_Teams=(
            "Acquisition_Segment_Team", "nunique"
        ),
        Unique_Sub_Teams=(
            "Acquisition_Segment_Sub_Team", "nunique"
        )
    )
)

print(
    "Different business units on first gift date:",
    (acquisition_consistency["Unique_Business_Units"] > 1).sum()
)

print(
    "Different teams on first gift date:",
    (acquisition_consistency["Unique_Teams"] > 1).sum()
)

print(
    "Different sub-teams on first gift date:",
    (acquisition_consistency["Unique_Sub_Teams"] > 1).sum()
)

**Observation:** Acquisition information was fully consistent across multiple transactions occurring on the same first-gift date. No supporter had conflicting business unit, team or sub-team information, confirming that the selected first-gift acquisition variables reliably represent the initial giving occasion.

## 5.6 Validate the Final Analytical Dataset

The combined supporter-level dataset is validated before exploratory analysis.

The checks confirm that supporter records remain unique, the repeat-giving target is complete, key first-gift variables are available, and no duplicate records were introduced during data integration.

In [ ]:
# ---------------------------------------------------------
# Final Validation Before EDA
# ---------------------------------------------------------

# Check dataset structure
print("DATASET STRUCTURE")

print("Total records:", f"{len(analysis_data):,}")

print(
    "Unique supporters:",
    f"{analysis_data['Deakin_Supporter_ID'].nunique():,}"
)

print(
    "Duplicate supporter rows:",
    analysis_data["Deakin_Supporter_ID"].duplicated().sum()
)


# ---------------------------------------------------------
# Check important analytical variables
# ---------------------------------------------------------

print("\nKEY VARIABLE CHECKS")

key_variables = [
    "First_Gift_Date",
    "First_Gift_Total_Amount",
    "First_Gift_Channel",
    "First_Gift_Primary_Product",
    "Relationship_At_First_Gift",
    "Previous_Ongoing_Commitment_Count",
    "First_Gift_Acquisition_Business_Unit",
    "First_Gift_Acquisition_Team",
    "First_Gift_Acquisition_Sub_Team",
    "Repeat_Donor"
]

for column in key_variables:
    print(
        f"Missing {column}:",
        analysis_data[column].isna().sum()
    )


# ---------------------------------------------------------
# Check target distribution
# ---------------------------------------------------------

print("\nTARGET DISTRIBUTION")

print(
    analysis_data["Repeat_Donor"]
    .value_counts()
)


# ---------------------------------------------------------
# Review missing values in additional supporter information
# ---------------------------------------------------------

print("\nMAIN MISSING VALUES")

main_variables = [
    "Age_At_First_Gift",
    "Age_At_Joining_WVA",
    "Address_State",
    "Address_Country",
    "Helix_Community",
    "Helix_Persona",
    "Lewers_Segment_Description",
    "Days_With_WVA_Before_First_Gift"
]

missing_summary = (
    analysis_data[main_variables]
    .isna()
    .sum()
    .to_frame("Missing")
)

missing_summary["Missing_%"] = (
    missing_summary["Missing"] /
    len(analysis_data) * 100
).round(1)

display(missing_summary)

## 5.7 View the Final Analytical Dataset

The final supporter-level analytical dataset is reviewed before exploratory analysis.

A sample of supporter records is displayed to confirm how the available information is represented, followed by a variable summary showing data type, missing values and the number of unique values for each field.

In [ ]:
# ---------------------------------------------------------
# View Final Analytical Dataset
# ---------------------------------------------------------

# Show the main supporter-level information entering EDA
preview_columns = [
    "Deakin_Supporter_ID",
    "First_Gift_Date",
    "First_Gift_Total_Amount",
    "First_Gift_Transaction_Count",
    "First_Gift_Channel",
    "First_Gift_Primary_Product",
    "First_Gift_Campaign",
    "First_Gift_Activity",
    "Relationship_At_First_Gift",
    "Previous_Ongoing_Commitment_Count",
    "WVA_History_Before_First_Gift",
    "Address_State",
    "Helix_Persona",
    "First_Gift_Acquisition_Business_Unit",
    "First_Gift_Acquisition_Team",
    "Repeat_Donor"
]

print("SAMPLE OF FINAL ANALYTICAL DATASET")
# Portfolio privacy edit: supporter-level preview suppressed.


# ---------------------------------------------------------
# Create a summary of every variable in the final dataset
# ---------------------------------------------------------

variable_summary = pd.DataFrame({
    "Variable": analysis_data.columns,
    "Data_Type": analysis_data.dtypes.astype(str).values,
    "Missing": analysis_data.isna().sum().values,
    "Missing_%": (
        analysis_data.isna().mean().values * 100
    ).round(1),
    "Unique_Values": [
        analysis_data[col].nunique(dropna=True)
        for col in analysis_data.columns
    ]
})

print("\nFINAL DATASET VARIABLE SUMMARY")
# Portfolio privacy edit: supporter-level preview suppressed.

# 6. Exploratory Data Analysis

Exploratory Data Analysis (EDA) is used to understand the patterns associated with repeat one-off giving and translate them into useful insights for WVA.

The analysis focuses on three practical questions:

- **Who** shows stronger repeat-giving behaviour?
- **Why** might repeat giving differ across first-gift, acquisition and supporter relationship characteristics?
- **When** do repeat donors typically make their next qualifying donation?

Each analysis considers both the observed pattern and its practical relevance to WVA. The findings will be used to identify potential drivers of repeat giving, develop actionable retention recommendations and guide feature selection for predictive modelling.

## 6.1 Repeat-Giving Overview

### 6.1.1 Repeat vs Non-Repeat Supporters

The overall level of repeat giving is examined first to establish a baseline for the analytical cohort.

A repeat donor is defined as a supporter who made another qualifying one-off donation on a later date within 24 months of their first qualifying gift.

This overall repeat rate will be used as a benchmark when evaluating later supporter, donation and acquisition segments.

In [ ]:
# ---------------------------------------------------------
# 6.1.1 Repeat vs Non-Repeat Supporters
# ---------------------------------------------------------

# Calculate the number of repeat and non-repeat supporters
repeat_summary = (
    analysis_data["Repeat_Donor"]
    .value_counts()
    .sort_index()
    .rename_axis("Repeat_Donor")
    .reset_index(name="Supporters")
)

# Give the target values clear business labels
repeat_summary["Group"] = repeat_summary["Repeat_Donor"].map({
    0: "Non-Repeat",
    1: "Repeat"
})

# Calculate percentage of the analytical population
repeat_summary["Percentage"] = (
    repeat_summary["Supporters"] /
    len(analysis_data) * 100
).round(1)

# Keep a simple presentation table
repeat_summary = repeat_summary[
    ["Group", "Supporters", "Percentage"]
]

print("REPEAT-GIVING OVERVIEW")
display(repeat_summary)


# Overall repeat rate - this becomes our benchmark
overall_repeat_rate = (
    analysis_data["Repeat_Donor"].mean() * 100
)

print(
    f"\nOverall repeat-rate baseline: "
    f"{overall_repeat_rate:.1f}%"
)


# ---------------------------------------------------------
# Visualise repeat vs non-repeat supporters
# ---------------------------------------------------------

fig, ax = plt.subplots(figsize=(8, 5.5))

bars = ax.bar(
    repeat_summary["Group"],
    repeat_summary["Supporters"],
    width=0.55
)

# Add count and percentage labels
for bar, count, percentage in zip(
    bars,
    repeat_summary["Supporters"],
    repeat_summary["Percentage"]
):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 300,
        f"{count:,}\n{percentage:.1f}%",
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold"
    )

ax.set_title(
    "Repeat Giving Within 24 Months",
    fontsize=15,
    fontweight="bold",
    pad=15
)

ax.set_ylabel("Number of Supporters")
ax.set_xlabel("")

ax.set_ylim(
    0,
    repeat_summary["Supporters"].max() * 1.15
)

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.25
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

**Observation:** Of the 19,194 supporters in the analytical cohort, 3,194 (16.6%) made another qualifying one-off donation within 24 months, while 16,000 (83.4%) did not. The 16.6% repeat rate provides the baseline for later analysis, where supporter segments with substantially higher repeat rates may represent stronger opportunities for targeted retention activity.

### 6.1.2 Repeat Giving Across First-Gift Years

Repeat rates are compared across first-gift years to determine whether repeat-giving behaviour has remained stable over time.

This analysis is primarily a stability check rather than a targeting recommendation. Meaningful changes across years may indicate changing supporter behaviour and will help guide the later modelling and validation strategy.

In [ ]:
# ---------------------------------------------------------
# 6.1.2 Repeat Giving Across First-Gift Years
# ---------------------------------------------------------

# Overall repeat-rate benchmark from 6.1.1
overall_repeat_rate = (
    analysis_data["Repeat_Donor"].mean() * 100
)

# Summarise supporter volume and repeat behaviour by first-gift year
year_summary = (
    analysis_data
    .groupby("First_Gift_Year")
    .agg(
        Supporters=("Deakin_Supporter_ID", "count"),
        Repeat_Donors=("Repeat_Donor", "sum"),
        Repeat_Rate=("Repeat_Donor", "mean")
    )
    .reset_index()
)

# Convert repeat rate to percentage
year_summary["Repeat_Rate"] = (
    year_summary["Repeat_Rate"] * 100
).round(1)

# Compare each year with the overall 16.6% baseline
year_summary["Difference_vs_Overall"] = (
    year_summary["Repeat_Rate"] - overall_repeat_rate
).round(1)

# ---------------------------------------------------------
# Display summary table
# ---------------------------------------------------------

print("REPEAT GIVING BY FIRST-GIFT YEAR")

display(
    year_summary.rename(
        columns={
            "First_Gift_Year": "First Gift Year",
            "Repeat_Donors": "Repeat Donors",
            "Repeat_Rate": "Repeat Rate (%)",
            "Difference_vs_Overall": "Difference vs Overall (pp)"
        }
    )
)

# ---------------------------------------------------------
# Visualise repeat rate over time
# ---------------------------------------------------------

fig, ax = plt.subplots(figsize=(8.5, 5.5))

ax.plot(
    year_summary["First_Gift_Year"],
    year_summary["Repeat_Rate"],
    marker="o",
    linewidth=2.5,
    markersize=8
)

# Add repeat-rate labels
for x, y in zip(
    year_summary["First_Gift_Year"],
    year_summary["Repeat_Rate"]
):
    ax.text(
        x,
        y + 0.5,
        f"{y:.1f}%",
        ha="center",
        fontsize=10.5,
        fontweight="bold"
    )

# Add overall repeat-rate benchmark
ax.axhline(
    overall_repeat_rate,
    linestyle="--",
    linewidth=1.5,
    label=f"Overall Repeat Rate ({overall_repeat_rate:.1f}%)"
)

ax.set_title(
    "Repeat Giving Across First-Gift Years",
    fontsize=15,
    fontweight="bold",
    pad=15
)

ax.set_xlabel("First-Gift Year")
ax.set_ylabel("Repeat Rate (%)")
ax.set_xticks(year_summary["First_Gift_Year"])

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.25
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(frameon=False)

plt.tight_layout()
plt.show()

**Observation:** Repeat giving declined across the main full-year cohorts, from 19.6% in 2021 to 17.0% in 2022 and 13.9% in 2023. This indicates that repeat-giving behaviour has changed over time and supports the use of later-period hold-out data when evaluating the predictive model. The 2024 cohort contains only 612 supporters because it represents a partial year, so its 15.5% repeat rate should be interpreted cautiously.

## 6.2 First-Gift Behaviour

The first giving occasion is examined to understand whether initial donation behaviour is associated with future repeat giving.

### 6.2.1 First-Gift Amount Profile

First-gift amounts are grouped into practical value bands to understand the typical size of the initial giving occasion.

The raw amount distribution is highly skewed by a small number of very large donations, so value bands provide a clearer and more business-relevant view of where most supporters are concentrated.

In [ ]:
# ---------------------------------------------------------
# 6.2.1 First-Gift Amount Profile
# ---------------------------------------------------------

# Create practical and business-readable first-gift amount bands
amount_bins = [
    0,
    50,
    100,
    250,
    500,
    1000,
    5000,
    float("inf")
]

amount_labels = [
    "$0–50",
    "$51–100",
    "$101–250",
    "$251–500",
    "$501–1,000",
    "$1,001–5,000",
    "$5,000+"
]

analysis_data["First_Gift_Amount_Band"] = pd.cut(
    analysis_data["First_Gift_Total_Amount"],
    bins=amount_bins,
    labels=amount_labels,
    include_lowest=True
)


# ---------------------------------------------------------
# Summarise supporter distribution
# ---------------------------------------------------------

amount_distribution = (
    analysis_data["First_Gift_Amount_Band"]
    .value_counts(sort=False)
    .rename_axis("First Gift Amount")
    .reset_index(name="Supporters")
)

amount_distribution["Percentage"] = (
    amount_distribution["Supporters"] /
    len(analysis_data) * 100
).round(1)

print("FIRST-GIFT AMOUNT PROFILE")
display(amount_distribution)


# ---------------------------------------------------------
# Visualise supporter distribution
# ---------------------------------------------------------

fig, ax = plt.subplots(figsize=(10, 5.8))

bars = ax.bar(
    amount_distribution["First Gift Amount"].astype(str),
    amount_distribution["Supporters"],
    width=0.65
)

# Add supporter count and percentage
for bar, count, percentage in zip(
    bars,
    amount_distribution["Supporters"],
    amount_distribution["Percentage"]
):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 100,
        f"{count:,}\n{percentage:.1f}%",
        ha="center",
        fontsize=9.5,
        fontweight="bold"
    )

ax.set_title(
    "Distribution of First-Gift Amounts",
    fontsize=15,
    fontweight="bold",
    pad=15
)

ax.set_xlabel("First-Gift Amount")
ax.set_ylabel("Number of Supporters")

ax.set_ylim(
    0,
    amount_distribution["Supporters"].max() * 1.18
)

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.25
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

**Observation:** First-gift values are strongly concentrated in the lower donation bands. Almost half of supporters (47.4%) gave \$50 or less, while 86.2% gave \$250 or less. First gifts above \$1,000 were relatively uncommon. This indicates that any scalable repeat-giving strategy will need to work primarily for lower- and mid-value supporters rather than focusing only on high-value donors.

### 6.2.2 Repeat Rate by First-Gift Amount

Repeat rates are compared across first-gift amount bands to determine whether initial donation value is associated with future repeat giving.

Both repeat rate and supporter volume are considered, because very high repeat rates in small donor groups may have limited value for large-scale retention activity.

In [ ]:
# ---------------------------------------------------------
# 6.2.2 Repeat Rate by First-Gift Amount
# ---------------------------------------------------------

# Summarise repeat behaviour within each amount band
amount_repeat_summary = (
    analysis_data
    .groupby(
        "First_Gift_Amount_Band",
        observed=False
    )
    .agg(
        Supporters=("Deakin_Supporter_ID", "count"),
        Repeat_Donors=("Repeat_Donor", "sum"),
        Repeat_Rate=("Repeat_Donor", "mean")
    )
    .reset_index()
)

# Convert repeat rate to percentage
amount_repeat_summary["Repeat_Rate"] = (
    amount_repeat_summary["Repeat_Rate"] * 100
).round(1)

# Compare each group with the overall 16.6% baseline
amount_repeat_summary["Difference_vs_Overall"] = (
    amount_repeat_summary["Repeat_Rate"]
    - overall_repeat_rate
).round(1)


# ---------------------------------------------------------
# Display business summary table
# ---------------------------------------------------------

print("REPEAT GIVING BY FIRST-GIFT AMOUNT")

display(
    amount_repeat_summary.rename(
        columns={
            "First_Gift_Amount_Band": "First Gift Amount",
            "Repeat_Donors": "Repeat Donors",
            "Repeat_Rate": "Repeat Rate (%)",
            "Difference_vs_Overall": "Difference vs Overall (pp)"
        }
    )
)


# ---------------------------------------------------------
# Visualise repeat rate by amount band
# ---------------------------------------------------------

fig, ax = plt.subplots(figsize=(10, 5.8))

bars = ax.bar(
    amount_repeat_summary[
        "First_Gift_Amount_Band"
    ].astype(str),
    amount_repeat_summary["Repeat_Rate"],
    width=0.65
)

# Add repeat-rate labels
for bar, rate in zip(
    bars,
    amount_repeat_summary["Repeat_Rate"]
):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.6,
        f"{rate:.1f}%",
        ha="center",
        fontsize=10,
        fontweight="bold"
    )

# Overall repeat-rate benchmark
ax.axhline(
    overall_repeat_rate,
    linestyle="--",
    linewidth=1.5,
    label=f"Overall Repeat Rate ({overall_repeat_rate:.1f}%)"
)

ax.set_title(
    "Repeat Rate by First-Gift Amount",
    fontsize=15,
    fontweight="bold",
    pad=15
)

ax.set_xlabel("First-Gift Amount")
ax.set_ylabel("Repeat Rate (%)")

ax.set_ylim(
    0,
    amount_repeat_summary["Repeat_Rate"].max() + 7
)

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.25
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(frameon=False)

plt.tight_layout()
plt.show()

**Observation:** Repeat giving shows a clear positive relationship with first-gift value. Supporters giving \$50 or less recorded a 13.2% repeat rate, below the overall 16.6% baseline, while repeat rates increased to 23.3% for \$251–500 and approximately 26% for gifts between \$501 and \$5,000. The \$5,000+ group recorded the highest repeat rate at 38.0%, although this group contained only 150 supporters.

**Business implication:** First-gift value appears to be a useful prioritisation signal, but it should not be used as an exclusion rule. Higher-value first gifts may justify testing more intensive or personalised retention activity, while lower-value supporters still represent a large share of the total repeat-donor opportunity and require scalable retention journeys.

### 6.2.3 Single vs Multiple Products on the First Giving Occasion

Repeat behaviour is compared between supporters who contributed to a single WVA product and those who contributed to multiple products during their first giving occasion.

This tests whether broader initial involvement with WVA is associated with stronger subsequent repeat giving. Both supporter volume and repeat rate are considered when interpreting the result.

In [ ]:
# ---------------------------------------------------------
# 6.2.3 Single vs Multiple Products on First Giving Occasion
# ---------------------------------------------------------

# Summarise repeat behaviour by first-gift product pattern
product_pattern_summary = (
    analysis_data
    .groupby("First_Gift_Multiple_Products")
    .agg(
        Supporters=("Deakin_Supporter_ID", "count"),
        Repeat_Donors=("Repeat_Donor", "sum"),
        Repeat_Rate=("Repeat_Donor", "mean")
    )
    .reset_index()
)

# Give the groups clear business labels
product_pattern_summary["Product_Pattern"] = (
    product_pattern_summary["First_Gift_Multiple_Products"]
    .map({
        0: "Single Product",
        1: "Multiple Products"
    })
)

# Convert repeat rate to percentage
product_pattern_summary["Repeat_Rate"] = (
    product_pattern_summary["Repeat_Rate"] * 100
).round(1)

# Compare with overall baseline
product_pattern_summary["Difference_vs_Overall"] = (
    product_pattern_summary["Repeat_Rate"]
    - overall_repeat_rate
).round(1)


# ---------------------------------------------------------
# Display summary table
# ---------------------------------------------------------

print("REPEAT GIVING BY FIRST-GIFT PRODUCT PATTERN")

display(
    product_pattern_summary[
        [
            "Product_Pattern",
            "Supporters",
            "Repeat_Donors",
            "Repeat_Rate",
            "Difference_vs_Overall"
        ]
    ]
    .rename(
        columns={
            "Product_Pattern": "First-Gift Product Pattern",
            "Repeat_Donors": "Repeat Donors",
            "Repeat_Rate": "Repeat Rate (%)",
            "Difference_vs_Overall": "Difference vs Overall (pp)"
        }
    )
)


# ---------------------------------------------------------
# Visualise repeat rate
# ---------------------------------------------------------

fig, ax = plt.subplots(figsize=(7.5, 5.5))

bars = ax.bar(
    product_pattern_summary["Product_Pattern"],
    product_pattern_summary["Repeat_Rate"],
    width=0.55
)

for bar, rate in zip(
    bars,
    product_pattern_summary["Repeat_Rate"]
):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f"{rate:.1f}%",
        ha="center",
        fontsize=11,
        fontweight="bold"
    )

# Overall repeat-rate benchmark
ax.axhline(
    overall_repeat_rate,
    linestyle="--",
    linewidth=1.5,
    label=f"Overall Repeat Rate ({overall_repeat_rate:.1f}%)"
)

ax.set_title(
    "Repeat Rate by First-Gift Product Pattern",
    fontsize=15,
    fontweight="bold",
    pad=15
)

ax.set_ylabel("Repeat Rate (%)")
ax.set_xlabel("")

ax.set_ylim(
    0,
    product_pattern_summary["Repeat_Rate"].max() + 6
)

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.25
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(frameon=False)

plt.tight_layout()
plt.show()

**Observation:** Supporters who contributed to multiple products during their first giving occasion recorded a higher repeat rate (19.7%) than those supporting a single product (16.4%). The 3.3 percentage-point difference suggests that broader initial involvement with WVA is associated with stronger repeat-giving behaviour.

**Business implication:** Multiple-product support may be a useful prioritisation signal for retention activity and a potential predictive feature. However, this is an association rather than evidence that encouraging multiple products directly causes repeat giving.

### 6.2.4 Repeat Giving by Primary First-Gift Product

The primary product associated with each supporter's first giving occasion is examined to determine whether different forms of initial support are associated with different repeat-giving behaviour.

Both supporter volume and repeat rate are considered because very small product groups can produce unstable percentages and should not drive business recommendations.

In [ ]:
# ---------------------------------------------------------
# 6.2.4 Repeat Giving by Primary First-Gift Product
# ---------------------------------------------------------

# Summarise repeat behaviour by primary first-gift product
product_summary = (
    analysis_data
    .groupby("First_Gift_Primary_Product")
    .agg(
        Supporters=("Deakin_Supporter_ID", "count"),
        Repeat_Donors=("Repeat_Donor", "sum"),
        Repeat_Rate=("Repeat_Donor", "mean")
    )
    .reset_index()
)

# Convert repeat rate to percentage
product_summary["Repeat_Rate"] = (
    product_summary["Repeat_Rate"] * 100
).round(1)

# Compare each product with the overall baseline
product_summary["Difference_vs_Overall"] = (
    product_summary["Repeat_Rate"]
    - overall_repeat_rate
).round(1)

# Calculate lift relative to the overall repeat rate
product_summary["Lift_vs_Overall"] = (
    product_summary["Repeat_Rate"]
    / overall_repeat_rate
).round(2)

# Sort by supporter volume
product_summary = product_summary.sort_values(
    "Supporters",
    ascending=False
)

print(
    "Number of primary first-gift products:",
    product_summary["First_Gift_Primary_Product"].nunique()
)

print("\nREPEAT GIVING BY PRIMARY FIRST-GIFT PRODUCT")

display(
    product_summary.rename(
        columns={
            "First_Gift_Primary_Product": "Primary First-Gift Product",
            "Repeat_Donors": "Repeat Donors",
            "Repeat_Rate": "Repeat Rate (%)",
            "Difference_vs_Overall": "Difference vs Overall (pp)",
            "Lift_vs_Overall": "Lift vs Overall"
        }
    )
)

**Observation:** Repeat-giving behaviour varies substantially across primary first-gift products. Multiplying Gift Appeal recorded the strongest meaningful result, with 602 supporters and a 36.2% repeat rate, more than twice the overall 16.6% baseline. WV Regular Giving also performed above baseline at 19.4% across a much larger population of 5,602 supporters.

In contrast, the two largest groups, World Vision Gifts (14.0%) and Emergency Relief (15.3%), recorded repeat rates below the overall baseline. These large supporter populations may therefore represent important opportunities for improving retention activity.

**Business implication:** First-gift product appears to be a useful segmentation and predictive signal. High-performing products such as Multiplying Gift Appeal should be investigated further to understand what is different about their supporter journey, while large lower-performing groups may benefit from targeted retention testing. These differences are associations and should be examined alongside gift value, channel and acquisition source before making product-specific recommendations.

In [ ]:
# ---------------------------------------------------------
# Visualise Reliable Product Groups
# ---------------------------------------------------------

# Keep products with at least 100 supporters
# so very small groups do not distort interpretation
product_chart = (
    product_summary[
        product_summary["Supporters"] >= 100
    ]
    .sort_values(
        "Repeat_Rate",
        ascending=True
    )
    .copy()
)

fig, ax = plt.subplots(figsize=(10, 6.5))

bars = ax.barh(
    product_chart["First_Gift_Primary_Product"],
    product_chart["Repeat_Rate"]
)

# Add repeat-rate labels
for bar, rate in zip(
    bars,
    product_chart["Repeat_Rate"]
):
    ax.text(
        bar.get_width() + 0.5,
        bar.get_y() + bar.get_height() / 2,
        f"{rate:.1f}%",
        va="center",
        fontsize=10,
        fontweight="bold"
    )

# Overall baseline
ax.axvline(
    overall_repeat_rate,
    linestyle="--",
    linewidth=1.5,
    label=f"Overall Repeat Rate ({overall_repeat_rate:.1f}%)"
)

ax.set_title(
    "Repeat Rate by Primary First-Gift Product",
    fontsize=15,
    fontweight="bold",
    pad=15
)

ax.set_xlabel("Repeat Rate (%)")
ax.set_ylabel("")

ax.grid(
    axis="x",
    linestyle="--",
    alpha=0.25
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(frameon=False)

plt.tight_layout()
plt.show()

## 6.3 Acquisition Behaviour

Acquisition behaviour is examined to understand how supporters first entered WVA and whether different acquisition sources are associated with repeat giving.

### 6.3.1 First-Gift Channel Profile

The distribution of first-gift channels is examined first to understand which acquisition sources contribute the largest number of supporters.

This provides context for later comparison of repeat rates, because very small channels can produce unstable percentages.

In [ ]:
# ---------------------------------------------------------
# 6.3.1 First-Gift Channel Profile
# ---------------------------------------------------------

# Count supporters by first-gift channel
channel_profile = (
    analysis_data["First_Gift_Channel"]
    .value_counts()
    .reset_index()
)

channel_profile.columns = [
    "First_Gift_Channel",
    "Supporters"
]


# Calculate percentage of total supporters
channel_profile["Percentage"] = (
    channel_profile["Supporters"]
    / channel_profile["Supporters"].sum()
    * 100
).round(1)


# ---------------------------------------------------------
# Display full channel profile table
# ---------------------------------------------------------

print("FIRST-GIFT CHANNEL PROFILE")

display(
    channel_profile.rename(
        columns={
            "First_Gift_Channel": "First-Gift Channel"
        }
    )
)


# ---------------------------------------------------------
# Keep channels with at least 50 supporters for the chart
# Small channels remain in the dataset but are not plotted
# ---------------------------------------------------------

major_channel_profile = channel_profile[
    channel_profile["Supporters"] >= 50
].copy()


# Sort from smallest to largest for horizontal chart
major_channel_profile = major_channel_profile.sort_values(
    "Supporters",
    ascending=True
)


# ---------------------------------------------------------
# Visualise supporter volume by channel
# ---------------------------------------------------------

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.barh(
    major_channel_profile["First_Gift_Channel"],
    major_channel_profile["Supporters"]
)


# Add supporter count and percentage labels
for bar, count, percentage in zip(
    bars,
    major_channel_profile["Supporters"],
    major_channel_profile["Percentage"]
):
    ax.text(
        bar.get_width() + 100,
        bar.get_y() + bar.get_height() / 2,
        f"{count:,} ({percentage:.1f}%)",
        va="center",
        fontsize=9.5,
        fontweight="bold"
    )


ax.set_title(
    "Supporters by First-Gift Channel",
    fontsize=15,
    fontweight="bold",
    pad=15
)

ax.set_xlabel("Number of Supporters")
ax.set_ylabel("")

ax.grid(
    axis="x",
    linestyle="--",
    alpha=0.25
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

**Observation:** First-gift acquisition is heavily concentrated through the Website channel, which accounts for 17,427 supporters (90.8%) of the analytical population. Retail Stand Internal contributes a further 4.5%, while all remaining channels individually represent only a small share of supporters.

**Business implication:** Because most one-off supporters enter through the Website channel, scalable digital retention activity will be particularly important for WVA. However, channel volume alone does not indicate retention quality, so repeat rates are compared next.

### 6.3.2 Repeat Rate by First-Gift Channel

Repeat rates are compared across the main first-gift channels to determine whether acquisition source is associated with future repeat giving.

Both supporter volume and repeat rate are considered. Channels with fewer than 50 supporters are excluded from the main comparison because percentages based on very small groups may be unstable.

In [ ]:
# ---------------------------------------------------------
# 6.3.2 Repeat Rate by First-Gift Channel
# ---------------------------------------------------------

# Summarise repeat behaviour by first-gift channel
channel_repeat_summary = (
    analysis_data
    .groupby("First_Gift_Channel")
    .agg(
        Supporters=("Deakin_Supporter_ID", "count"),
        Repeat_Donors=("Repeat_Donor", "sum"),
        Repeat_Rate=("Repeat_Donor", "mean")
    )
    .reset_index()
)

# Convert repeat rate to percentage
channel_repeat_summary["Repeat_Rate"] = (
    channel_repeat_summary["Repeat_Rate"] * 100
).round(1)

# Compare each channel with the overall repeat-rate baseline
channel_repeat_summary["Difference_vs_Overall"] = (
    channel_repeat_summary["Repeat_Rate"]
    - overall_repeat_rate
).round(1)

# Calculate lift relative to the overall repeat rate
channel_repeat_summary["Lift_vs_Overall"] = (
    channel_repeat_summary["Repeat_Rate"]
    / overall_repeat_rate
).round(2)


# ---------------------------------------------------------
# Keep channels with enough supporters for interpretation
# ---------------------------------------------------------

major_channels = (
    channel_repeat_summary[
        channel_repeat_summary["Supporters"] >= 50
    ]
    .copy()
)

# Sort by repeat rate
major_channels = major_channels.sort_values(
    "Repeat_Rate",
    ascending=False
)


# ---------------------------------------------------------
# Display summary table
# ---------------------------------------------------------

print("REPEAT GIVING BY FIRST-GIFT CHANNEL")

display(
    major_channels.rename(
        columns={
            "First_Gift_Channel": "First-Gift Channel",
            "Repeat_Donors": "Repeat Donors",
            "Repeat_Rate": "Repeat Rate (%)",
            "Difference_vs_Overall": "Difference vs Overall (pp)",
            "Lift_vs_Overall": "Lift vs Overall"
        }
    )
)


# ---------------------------------------------------------
# Visualise repeat rate by channel
# ---------------------------------------------------------

channel_chart = major_channels.sort_values(
    "Repeat_Rate",
    ascending=True
)

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.barh(
    channel_chart["First_Gift_Channel"],
    channel_chart["Repeat_Rate"]
)

# Add repeat-rate labels
for bar, rate in zip(
    bars,
    channel_chart["Repeat_Rate"]
):
    ax.text(
        bar.get_width() + 0.4,
        bar.get_y() + bar.get_height() / 2,
        f"{rate:.1f}%",
        va="center",
        fontsize=10,
        fontweight="bold"
    )

# Add overall repeat-rate baseline
ax.axvline(
    overall_repeat_rate,
    linestyle="--",
    linewidth=1.5,
    label=f"Overall Repeat Rate ({overall_repeat_rate:.1f}%)"
)

ax.set_title(
    "Repeat Rate by First-Gift Channel",
    fontsize=15,
    fontweight="bold",
    pad=15
)

ax.set_xlabel("Repeat Rate (%)")
ax.set_ylabel("")

ax.grid(
    axis="x",
    linestyle="--",
    alpha=0.25
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(frameon=False)

plt.tight_layout()
plt.show()

**Observation:** Repeat-giving behaviour differs substantially across first-gift channels. Events Internal recorded the highest meaningful repeat rate at 43.2% across 199 supporters, around 2.6 times the overall 16.6% baseline. CC Inbound Internal and Legacy also performed above baseline, while Website, which accounts for most supporters, recorded a repeat rate of 17.0%.

Retail Stand Internal stands out in the opposite direction. Despite acquiring 860 supporters, only 3.7% repeated within 24 months.

**Business implication:** Acquisition channel appears to be an important retention signal. WVA should investigate what is different about the supporter experience associated with high-performing channels such as Events Internal, while the Retail Stand journey may warrant particular attention because it produces a meaningful number of supporters but very low repeat giving. Website retention also remains strategically important because of its very large supporter volume.

### 6.3.3 Channel and Product Combination Check

Channel and primary first-gift product are examined together to determine whether strong channel-level repeat rates may partly reflect the types of products acquired through those channels.

Only combinations with a meaningful number of supporters are interpreted to reduce the risk of drawing conclusions from very small groups.

In [ ]:
# ---------------------------------------------------------
# 6.3.3 Channel and Product Combination Check
# ---------------------------------------------------------

# Summarise supporter behaviour for each
# channel + primary product combination
channel_product_summary = (
    analysis_data
    .groupby(
        [
            "First_Gift_Channel",
            "First_Gift_Primary_Product"
        ]
    )
    .agg(
        Supporters=("Deakin_Supporter_ID", "count"),
        Repeat_Donors=("Repeat_Donor", "sum"),
        Repeat_Rate=("Repeat_Donor", "mean")
    )
    .reset_index()
)

# Convert repeat rate to percentage
channel_product_summary["Repeat_Rate"] = (
    channel_product_summary["Repeat_Rate"] * 100
).round(1)

# Compare with overall baseline
channel_product_summary["Difference_vs_Overall"] = (
    channel_product_summary["Repeat_Rate"]
    - overall_repeat_rate
).round(1)


# ---------------------------------------------------------
# Keep combinations with at least 50 supporters
# ---------------------------------------------------------

reliable_channel_product = (
    channel_product_summary[
        channel_product_summary["Supporters"] >= 50
    ]
    .sort_values(
        ["Repeat_Rate", "Supporters"],
        ascending=[False, False]
    )
    .copy()
)


print("CHANNEL + PRODUCT COMBINATIONS")

display(
    reliable_channel_product.rename(
        columns={
            "First_Gift_Channel": "First-Gift Channel",
            "First_Gift_Primary_Product": "Primary First-Gift Product",
            "Repeat_Donors": "Repeat Donors",
            "Repeat_Rate": "Repeat Rate (%)",
            "Difference_vs_Overall": "Difference vs Overall (pp)"
        }
    )
)

**Observation:** The channel differences remain substantial even when supporters are compared within the same first-gift products. For example, WV Regular Giving supporters acquired through Website recorded a 19.9% repeat rate compared with only 3.3% through Retail Stand Internal. Similar gaps were observed for Child Rescue, World Vision Gifts and Emergency Relief.

The strong Multiplying Gift Appeal result also remained visible within Website supporters, where the repeat rate was 35.9%, indicating that its higher repeat behaviour is not explained only by channel.

**Business implication:** The very low Retail Stand repeat rates appear to extend across several different products, suggesting that the acquisition or early retention journey associated with Retail Stand supporters warrants further investigation. Rather than simply increasing Retail Stand acquisition volume, WVA could test a tailored post-acquisition journey for these supporters and measure whether repeat giving improves.

### 6.3.4 Acquisition Business Unit

The acquisition business unit associated with the first giving occasion is examined to determine whether broad supporter groups demonstrate different repeat-giving behaviour.

This analysis is used mainly to identify structurally different supporter populations. Both supporter volume and repeat rate are considered before deciding whether all business units should be combined in later predictive modelling.

In [ ]:
# ---------------------------------------------------------
# 6.3.4 Acquisition Business Unit
# ---------------------------------------------------------

# Summarise repeat behaviour by acquisition business unit
business_unit_summary = (
    analysis_data
    .groupby("First_Gift_Acquisition_Business_Unit")
    .agg(
        Supporters=("Deakin_Supporter_ID", "count"),
        Repeat_Donors=("Repeat_Donor", "sum"),
        Repeat_Rate=("Repeat_Donor", "mean")
    )
    .reset_index()
)

# Convert repeat rate to percentage
business_unit_summary["Repeat_Rate"] = (
    business_unit_summary["Repeat_Rate"] * 100
).round(1)

# Compare with the overall repeat-rate baseline
business_unit_summary["Difference_vs_Overall"] = (
    business_unit_summary["Repeat_Rate"]
    - overall_repeat_rate
).round(1)

# Calculate lift relative to the overall baseline
business_unit_summary["Lift_vs_Overall"] = (
    business_unit_summary["Repeat_Rate"]
    / overall_repeat_rate
).round(2)

# Sort by supporter volume
business_unit_summary = business_unit_summary.sort_values(
    "Supporters",
    ascending=False
)

print(
    "Number of acquisition business units:",
    business_unit_summary[
        "First_Gift_Acquisition_Business_Unit"
    ].nunique()
)

print("\nREPEAT GIVING BY ACQUISITION BUSINESS UNIT")

display(
    business_unit_summary.rename(
        columns={
            "First_Gift_Acquisition_Business_Unit": "Acquisition Business Unit",
            "Repeat_Donors": "Repeat Donors",
            "Repeat_Rate": "Repeat Rate (%)",
            "Difference_vs_Overall": "Difference vs Overall (pp)",
            "Lift_vs_Overall": "Lift vs Overall"
        }
    )
)

**Observation:** The analytical population is overwhelmingly concentrated in the Mass business unit, with 18,921 supporters and a 16.4% repeat rate. Private Funding contains only 199 supporters but records a substantially higher repeat rate of 35.7%.

**Business implication:** Private Funding represents a small, relationship-managed supporter group with behaviour that differs substantially from the general Mass population. To develop a predictive model that is relevant to WVA's scalable one-off donor retention activity, the main model should therefore focus on Mass supporters rather than combining structurally different supporter groups.

### 6.3.5 Acquisition Team

Acquisition teams are examined to determine whether they provide additional information beyond the broader business-unit classification.

Supporter volume is considered alongside repeat rate because some relationship-managed teams contain relatively few supporters and may not be appropriate for generalised retention recommendations or predictive modelling.

In [ ]:
# ---------------------------------------------------------
# 6.3.5 Acquisition Team
# ---------------------------------------------------------

# Summarise repeat behaviour by acquisition team
team_summary = (
    analysis_data
    .groupby("First_Gift_Acquisition_Team")
    .agg(
        Supporters=("Deakin_Supporter_ID", "count"),
        Repeat_Donors=("Repeat_Donor", "sum"),
        Repeat_Rate=("Repeat_Donor", "mean")
    )
    .reset_index()
)

# Convert repeat rate to percentage
team_summary["Repeat_Rate"] = (
    team_summary["Repeat_Rate"] * 100
).round(1)

# Compare with overall baseline
team_summary["Difference_vs_Overall"] = (
    team_summary["Repeat_Rate"]
    - overall_repeat_rate
).round(1)

# Calculate lift
team_summary["Lift_vs_Overall"] = (
    team_summary["Repeat_Rate"]
    / overall_repeat_rate
).round(2)

# Sort by supporter volume
team_summary = team_summary.sort_values(
    "Supporters",
    ascending=False
)

print(
    "Number of acquisition teams:",
    team_summary[
        "First_Gift_Acquisition_Team"
    ].nunique()
)

print("\nREPEAT GIVING BY ACQUISITION TEAM")

display(
    team_summary.rename(
        columns={
            "First_Gift_Acquisition_Team": "Acquisition Team",
            "Repeat_Donors": "Repeat Donors",
            "Repeat_Rate": "Repeat Rate (%)",
            "Difference_vs_Overall": "Difference vs Overall (pp)",
            "Lift_vs_Overall": "Lift vs Overall"
        }
    )
)

**Observation:** The acquisition-team results mainly provide additional detail within the small relationship-managed supporter groups. Mass accounts for 18,921 supporters, while Philanthropy, Middle and Bequest contain relatively small populations with higher repeat rates.

**Business implication:** Acquisition Team does not provide meaningful additional segmentation for the main Mass supporter population. It is therefore useful for understanding population structure but is unlikely to add predictive value once the main model is restricted to Mass supporters.

## 6.4 Previous WVA Relationship

Previous relationship with WVA is examined to determine whether existing supporter history is associated with future repeat giving.

### 6.4.1 Repeat Rate by Previous WVA Relationship

Supporters with no prior ongoing relationship are compared with supporters whose previous ongoing commitment had already ended before their first qualifying one-off gift.

This helps assess whether previous ongoing involvement with WVA provides a useful signal for repeat-giving propensity.

In [ ]:
# ---------------------------------------------------------
# 6.4.1 Repeat Rate by Previous WVA Relationship
# ---------------------------------------------------------

# Summarise repeat behaviour by previous WVA relationship
relationship_summary = (
    analysis_data
    .groupby("Relationship_At_First_Gift")
    .agg(
        Supporters=("Deakin_Supporter_ID", "count"),
        Repeat_Donors=("Repeat_Donor", "sum"),
        Repeat_Rate=("Repeat_Donor", "mean")
    )
    .reset_index()
)

# Convert repeat rate to percentage
relationship_summary["Repeat_Rate"] = (
    relationship_summary["Repeat_Rate"] * 100
).round(1)

# Compare each group with the overall baseline
relationship_summary["Difference_vs_Overall"] = (
    relationship_summary["Repeat_Rate"]
    - overall_repeat_rate
).round(1)

# Calculate lift relative to the overall baseline
relationship_summary["Lift_vs_Overall"] = (
    relationship_summary["Repeat_Rate"]
    / overall_repeat_rate
).round(2)


# Set a clear business order
relationship_order = [
    "No Prior Ongoing",
    "Former Ongoing"
]

relationship_summary["Relationship_At_First_Gift"] = pd.Categorical(
    relationship_summary["Relationship_At_First_Gift"],
    categories=relationship_order,
    ordered=True
)

relationship_summary = relationship_summary.sort_values(
    "Relationship_At_First_Gift"
)


# ---------------------------------------------------------
# Display summary table
# ---------------------------------------------------------

print("REPEAT GIVING BY PREVIOUS WVA RELATIONSHIP")

display(
    relationship_summary.rename(
        columns={
            "Relationship_At_First_Gift": "Previous WVA Relationship",
            "Repeat_Donors": "Repeat Donors",
            "Repeat_Rate": "Repeat Rate (%)",
            "Difference_vs_Overall": "Difference vs Overall (pp)",
            "Lift_vs_Overall": "Lift vs Overall"
        }
    )
)


# ---------------------------------------------------------
# Visualise repeat rate
# ---------------------------------------------------------

fig, ax = plt.subplots(figsize=(7.5, 5.5))

bars = ax.bar(
    relationship_summary[
        "Relationship_At_First_Gift"
    ].astype(str),
    relationship_summary["Repeat_Rate"],
    width=0.55
)

# Add repeat-rate labels
for bar, rate in zip(
    bars,
    relationship_summary["Repeat_Rate"]
):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f"{rate:.1f}%",
        ha="center",
        fontsize=11,
        fontweight="bold"
    )

# Add overall baseline
ax.axhline(
    overall_repeat_rate,
    linestyle="--",
    linewidth=1.5,
    label=f"Overall Repeat Rate ({overall_repeat_rate:.1f}%)"
)

ax.set_title(
    "Repeat Rate by Previous WVA Relationship",
    fontsize=15,
    fontweight="bold",
    pad=15
)

ax.set_ylabel("Repeat Rate (%)")
ax.set_xlabel("")

ax.set_ylim(
    0,
    relationship_summary["Repeat_Rate"].max() + 6
)

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.25
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(frameon=False)

plt.tight_layout()
plt.show()

**Observation:** Supporters with a former ongoing relationship with WVA recorded a higher repeat rate (21.3%) than supporters with no prior ongoing relationship (16.5%). Although Former Ongoing supporters represent a relatively small group of 399 supporters, their repeat rate is 4.8 percentage points higher than the no-prior-ongoing group.

**Business implication:** Previous ongoing involvement appears to be a useful retention signal. Former ongoing supporters could be recognised within the post-gift journey as returning supporters rather than treated identically to completely new supporters. Previous relationship status should also be retained as a candidate predictive feature.

### 6.4.2 WVA History Before the First Gift

Supporters are compared according to whether they were first recorded in the WVA database on their first qualifying one-off gift date or were already known to WVA beforehand.

This provides a broader view of previous WVA familiarity than ongoing-commitment history alone and helps assess whether existing supporter history is associated with stronger repeat giving.

In [ ]:
# ---------------------------------------------------------
# 6.4.2 Repeat Rate by WVA History Before First Gift
# ---------------------------------------------------------

# Summarise repeat behaviour by previous WVA history
history_summary = (
    analysis_data
    .groupby("WVA_History_Before_First_Gift")
    .agg(
        Supporters=("Deakin_Supporter_ID", "count"),
        Repeat_Donors=("Repeat_Donor", "sum"),
        Repeat_Rate=("Repeat_Donor", "mean")
    )
    .reset_index()
)

# Convert repeat rate to percentage
history_summary["Repeat_Rate"] = (
    history_summary["Repeat_Rate"] * 100
).round(1)

# Compare each group with the overall baseline
history_summary["Difference_vs_Overall"] = (
    history_summary["Repeat_Rate"]
    - overall_repeat_rate
).round(1)

# Calculate lift relative to the overall baseline
history_summary["Lift_vs_Overall"] = (
    history_summary["Repeat_Rate"]
    / overall_repeat_rate
).round(2)


# ---------------------------------------------------------
# Set a clear business order
# ---------------------------------------------------------

history_order = [
    "Joined on First Gift Date",
    "Known to WVA Before First Gift",
    "Unknown / Inconsistent"
]

history_summary["WVA_History_Before_First_Gift"] = pd.Categorical(
    history_summary["WVA_History_Before_First_Gift"],
    categories=history_order,
    ordered=True
)

history_summary = history_summary.sort_values(
    "WVA_History_Before_First_Gift"
)


# ---------------------------------------------------------
# Display summary table
# ---------------------------------------------------------

print("REPEAT GIVING BY WVA HISTORY BEFORE FIRST GIFT")

display(
    history_summary.rename(
        columns={
            "WVA_History_Before_First_Gift": "WVA History Before First Gift",
            "Repeat_Donors": "Repeat Donors",
            "Repeat_Rate": "Repeat Rate (%)",
            "Difference_vs_Overall": "Difference vs Overall (pp)",
            "Lift_vs_Overall": "Lift vs Overall"
        }
    )
)


# ---------------------------------------------------------
# Visualise repeat rate
# ---------------------------------------------------------

fig, ax = plt.subplots(figsize=(9, 5.5))

bars = ax.bar(
    history_summary[
        "WVA_History_Before_First_Gift"
    ].astype(str),
    history_summary["Repeat_Rate"],
    width=0.6
)

# Add repeat-rate labels
for bar, rate in zip(
    bars,
    history_summary["Repeat_Rate"]
):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f"{rate:.1f}%",
        ha="center",
        fontsize=10.5,
        fontweight="bold"
    )

# Add overall baseline
ax.axhline(
    overall_repeat_rate,
    linestyle="--",
    linewidth=1.5,
    label=f"Overall Repeat Rate ({overall_repeat_rate:.1f}%)"
)

ax.set_title(
    "Repeat Rate by WVA History Before First Gift",
    fontsize=15,
    fontweight="bold",
    pad=15
)

ax.set_ylabel("Repeat Rate (%)")
ax.set_xlabel("")

ax.set_ylim(
    0,
    history_summary["Repeat_Rate"].max() + 6
)

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.25
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(frameon=False)

plt.xticks(rotation=8)

plt.tight_layout()
plt.show()

**Observation:** Supporters who were already known to WVA before their first qualifying one-off gift recorded a substantially higher repeat rate (22.4%) than supporters recorded as joining WVA on their first-gift date (16.0%). This represents a 6.4 percentage-point difference and suggests that previous familiarity with WVA is associated with stronger repeat-giving behaviour.

**Business implication:** A first qualifying one-off gift should not automatically be treated as the start of a completely new supporter relationship. WVA could distinguish previously known supporters from genuinely new supporters when prioritising and tailoring post-gift retention activity. Previous WVA history should also be retained as a candidate predictive signal.

### 6.4.3 Length of Previous WVA History

Supporters who were already known to WVA before their first qualifying one-off gift are further grouped by the length of their recorded relationship with WVA.

This helps determine whether longer previous familiarity with WVA is associated with stronger repeat-giving behaviour, rather than treating all previously known supporters as one group.

In [ ]:
# ---------------------------------------------------------
# 6.4.3 Length of Previous WVA History
# ---------------------------------------------------------

# Create simple and business-readable WVA history bands
analysis_data["WVA_History_Length_Band"] = np.select(
    [
        analysis_data["Days_With_WVA_Before_First_Gift"] == 0,

        (
            analysis_data["Days_With_WVA_Before_First_Gift"] > 0
        ) & (
            analysis_data["Days_With_WVA_Before_First_Gift"] < 365 * 3
        ),

        (
            analysis_data["Days_With_WVA_Before_First_Gift"] >= 365 * 3
        ) & (
            analysis_data["Days_With_WVA_Before_First_Gift"] < 365 * 7
        ),

        analysis_data["Days_With_WVA_Before_First_Gift"] >= 365 * 7
    ],
    [
        "Joined on First Gift Date",
        "Known < 3 Years",
        "Known 3–7 Years",
        "Known 7+ Years"
    ],
    default="Unknown / Inconsistent"
)


# ---------------------------------------------------------
# Summarise repeat behaviour
# ---------------------------------------------------------

history_length_summary = (
    analysis_data
    .groupby("WVA_History_Length_Band")
    .agg(
        Supporters=("Deakin_Supporter_ID", "count"),
        Repeat_Donors=("Repeat_Donor", "sum"),
        Repeat_Rate=("Repeat_Donor", "mean")
    )
    .reset_index()
)

history_length_summary["Repeat_Rate"] = (
    history_length_summary["Repeat_Rate"] * 100
).round(1)

history_length_summary["Difference_vs_Overall"] = (
    history_length_summary["Repeat_Rate"]
    - overall_repeat_rate
).round(1)


# Set logical order
history_length_order = [
    "Joined on First Gift Date",
    "Known < 3 Years",
    "Known 3–7 Years",
    "Known 7+ Years",
    "Unknown / Inconsistent"
]

history_length_summary["WVA_History_Length_Band"] = pd.Categorical(
    history_length_summary["WVA_History_Length_Band"],
    categories=history_length_order,
    ordered=True
)

history_length_summary = history_length_summary.sort_values(
    "WVA_History_Length_Band"
)


print("REPEAT GIVING BY LENGTH OF WVA HISTORY")

display(
    history_length_summary.rename(
        columns={
            "WVA_History_Length_Band": "WVA History Length",
            "Repeat_Donors": "Repeat Donors",
            "Repeat_Rate": "Repeat Rate (%)",
            "Difference_vs_Overall": "Difference vs Overall (pp)"
        }
    )
)

**Observation:** Repeat giving does not increase steadily with longer recorded WVA history. The strongest group was supporters who had been known to WVA for less than three years before their first qualifying one-off gift, with a 29.1% repeat rate compared with the overall 16.6% baseline. Supporters known for three to seven years and seven years or more recorded more moderate repeat rates of 19.0% and 20.1%.

**Business implication:** Recorded WVA tenure may provide more useful information than a simple known/not-known flag. In particular, supporters who were already known to WVA for a relatively short period before their first qualifying one-off gift appear to represent a promising retention segment. This pattern should be checked against previous ongoing relationship and acquisition characteristics before being used as a targeting recommendation or predictive feature.

## 6.5 Supporter Characteristics

Supporter characteristics are explored selectively to identify whether geographic or demographic differences provide meaningful information about repeat giving.

Variables are included only where they provide sufficient coverage and practical value for WVA's retention decisions.

### 6.5.1 Geographic Coverage

Supporter country is reviewed first to understand the geographic composition of the analytical cohort.

This is primarily a coverage check. If the population is overwhelmingly Australian, country-level differences will not be treated as a major retention or predictive signal.

In [ ]:
# ---------------------------------------------------------
# 6.5.1 Geographic Coverage
# ---------------------------------------------------------

# Summarise supporter country
country_summary = (
    analysis_data
    .assign(
        Country=analysis_data["Address_Country"].fillna("Missing")
    )
    .groupby("Country")
    .agg(
        Supporters=("Deakin_Supporter_ID", "count"),
        Repeat_Donors=("Repeat_Donor", "sum"),
        Repeat_Rate=("Repeat_Donor", "mean")
    )
    .reset_index()
)

# Convert repeat rate to percentage
country_summary["Repeat_Rate"] = (
    country_summary["Repeat_Rate"] * 100
).round(1)

# Calculate share of analytical population
country_summary["Supporter_Share"] = (
    country_summary["Supporters"]
    / len(analysis_data)
    * 100
).round(1)

# Sort by supporter volume
country_summary = country_summary.sort_values(
    "Supporters",
    ascending=False
)

print("GEOGRAPHIC COVERAGE")

display(
    country_summary.rename(
        columns={
            "Repeat_Donors": "Repeat Donors",
            "Repeat_Rate": "Repeat Rate (%)",
            "Supporter_Share": "Supporter Share (%)"
        }
    )
)

**Observation:** The analytical cohort is overwhelmingly Australian, with 19,004 supporters (99.0%) recorded in Australia. Overseas supporter groups contain very few observations and therefore do not provide meaningful evidence for geographic retention differences.

**Business implication:** Country does not provide useful differentiation for this analysis and will not be treated as a core retention or predictive signal. Geographic analysis therefore focuses on Australian state, where sufficient supporter volume is available for comparison.

### 6.5.2 Repeat Giving by Australian State

Repeat-giving behaviour is compared across Australian states to determine whether meaningful geographic differences exist within the main supporter population.

Only supporters recorded in Australia with a valid state are included, and supporter volume is considered alongside repeat rate to avoid over-interpreting small geographic groups.

In [ ]:
# ---------------------------------------------------------
# 6.5.2 Repeat Giving by Australian State
# ---------------------------------------------------------

# Keep Australian supporters with a valid state
state_data = analysis_data[
    (analysis_data["Address_Country"] == "AUSTRALIA") &
    (analysis_data["Address_State"].notna())
].copy()


# ---------------------------------------------------------
# Summarise repeat behaviour by state
# ---------------------------------------------------------

state_summary = (
    state_data
    .groupby("Address_State")
    .agg(
        Supporters=("Deakin_Supporter_ID", "count"),
        Repeat_Donors=("Repeat_Donor", "sum"),
        Repeat_Rate=("Repeat_Donor", "mean")
    )
    .reset_index()
)

# Convert repeat rate to percentage
state_summary["Repeat_Rate"] = (
    state_summary["Repeat_Rate"] * 100
).round(1)

# Compare with overall baseline
state_summary["Difference_vs_Overall"] = (
    state_summary["Repeat_Rate"]
    - overall_repeat_rate
).round(1)

# Calculate lift relative to baseline
state_summary["Lift_vs_Overall"] = (
    state_summary["Repeat_Rate"]
    / overall_repeat_rate
).round(2)

# Sort table by supporter volume
state_summary = state_summary.sort_values(
    "Supporters",
    ascending=False
)

print("REPEAT GIVING BY AUSTRALIAN STATE")

display(
    state_summary.rename(
        columns={
            "Address_State": "State",
            "Repeat_Donors": "Repeat Donors",
            "Repeat_Rate": "Repeat Rate (%)",
            "Difference_vs_Overall": "Difference vs Overall (pp)",
            "Lift_vs_Overall": "Lift vs Overall"
        }
    )
)


# ---------------------------------------------------------
# Create clean dot plot
# ---------------------------------------------------------

# Sort states by repeat rate
plot_state = state_summary.sort_values(
    "Repeat_Rate",
    ascending=True
).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(9.5, 5.8))

y_positions = np.arange(len(plot_state))


# Draw subtle guide lines between baseline and each state
for y, rate in zip(
    y_positions,
    plot_state["Repeat_Rate"]
):
    ax.plot(
        [overall_repeat_rate, rate],
        [y, y],
        color="lightgray",
        linewidth=2,
        zorder=1
    )


# Plot state repeat rates
ax.scatter(
    plot_state["Repeat_Rate"],
    y_positions,
    s=110,
    zorder=3
)


# Overall repeat-rate baseline
ax.axvline(
    overall_repeat_rate,
    linestyle="--",
    linewidth=1.5,
    zorder=2,
    label=f"Overall Repeat Rate ({overall_repeat_rate:.1f}%)"
)


# ---------------------------------------------------------
# Add labels without overlap
# ---------------------------------------------------------

for y, rate in zip(
    y_positions,
    plot_state["Repeat_Rate"]
):

    # States below the baseline:
    # place label to the left of the point
    if rate < overall_repeat_rate:
        ax.annotate(
            f"{rate:.1f}%",
            xy=(rate, y),
            xytext=(-12, 0),
            textcoords="offset points",
            ha="right",
            va="center",
            fontsize=10,
            fontweight="bold"
        )

    # States at or above the baseline:
    # place label to the right of the point
    else:
        ax.annotate(
            f"{rate:.1f}%",
            xy=(rate, y),
            xytext=(12, 0),
            textcoords="offset points",
            ha="left",
            va="center",
            fontsize=10,
            fontweight="bold"
        )


# ---------------------------------------------------------
# Chart formatting
# ---------------------------------------------------------

ax.set_yticks(y_positions)
ax.set_yticklabels(plot_state["Address_State"])

# Give labels enough room on both sides
ax.set_xlim(
    plot_state["Repeat_Rate"].min() - 1.5,
    plot_state["Repeat_Rate"].max() + 1.8
)

ax.set_title(
    "Repeat Rate by Australian State",
    fontsize=15,
    fontweight="bold",
    pad=15
)

ax.set_xlabel("Repeat Rate (%)")
ax.set_ylabel("")

ax.grid(
    axis="x",
    linestyle="--",
    alpha=0.20
)

# Remove unnecessary borders
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)

ax.tick_params(
    axis="y",
    length=0
)

# Keep legend away from data points
ax.legend(
    frameon=False,
    loc="lower right"
)

plt.tight_layout()
plt.show()

**Observation:** Repeat-giving rates are relatively similar across most Australian states. WA records the highest rate at 18.7%, followed by TAS at 17.8%, while the largest supporter populations in NSW, VIC and QLD remain close to the overall 16.6% baseline.

**Business implication:** Geographic location does not appear to be a major driver of repeat giving within Australia. State may provide some secondary information, but the differences are much weaker than those observed for first-gift value, product and acquisition channel. A state-specific retention strategy is therefore not supported by this analysis.

### 6.5.3 Repeat Giving by Helix Community

Helix Community provides a broad demographic and lifestyle segmentation of supporters.

The standard Helix communities are compared to assess whether supporter profile is associated with repeat giving. Missing values are retained in the summary for transparency but excluded from the visual comparison because they do not represent a meaningful supporter segment.

In [ ]:
# ---------------------------------------------------------
# 6.5.3 Repeat Giving by Helix Community
# ---------------------------------------------------------

# Create a clear category for missing Helix information
analysis_data["Helix_Community_Analysis"] = (
    analysis_data["Helix_Community"]
    .fillna("Missing")
)


# ---------------------------------------------------------
# Summarise repeat behaviour
# ---------------------------------------------------------

helix_summary = (
    analysis_data
    .groupby("Helix_Community_Analysis")
    .agg(
        Supporters=("Deakin_Supporter_ID", "count"),
        Repeat_Donors=("Repeat_Donor", "sum"),
        Repeat_Rate=("Repeat_Donor", "mean")
    )
    .reset_index()
)

# Convert repeat rate to percentage
helix_summary["Repeat_Rate"] = (
    helix_summary["Repeat_Rate"] * 100
).round(1)

# Compare with overall baseline
helix_summary["Difference_vs_Overall"] = (
    helix_summary["Repeat_Rate"]
    - overall_repeat_rate
).round(1)

# Calculate supporter share
helix_summary["Supporter_Share"] = (
    helix_summary["Supporters"]
    / len(analysis_data)
    * 100
).round(1)

# Sort table by supporter volume
helix_summary = helix_summary.sort_values(
    "Supporters",
    ascending=False
)

print("REPEAT GIVING BY HELIX COMMUNITY")

display(
    helix_summary.rename(
        columns={
            "Helix_Community_Analysis": "Helix Community",
            "Repeat_Donors": "Repeat Donors",
            "Repeat_Rate": "Repeat Rate (%)",
            "Difference_vs_Overall": "Difference vs Overall (pp)",
            "Supporter_Share": "Supporter Share (%)"
        }
    )
)


# ---------------------------------------------------------
# Keep standard Helix communities for visual comparison
# ---------------------------------------------------------

standard_helix = [
    "100 Leading Lifestyles",
    "200 Metrotechs",
    "300 Aspirationals",
    "400 Hearth and Home",
    "500 Doing Fine",
    "600 Fair Go",
    "700 Other"
]

helix_plot = helix_summary[
    helix_summary["Helix_Community_Analysis"].isin(
        standard_helix
    )
].copy()

# Sort by repeat rate for easier comparison
helix_plot = helix_plot.sort_values(
    "Repeat_Rate",
    ascending=True
).reset_index(drop=True)


# ---------------------------------------------------------
# Create clean dot plot
# ---------------------------------------------------------

fig, ax = plt.subplots(figsize=(9.5, 6))

y_positions = np.arange(len(helix_plot))

# Draw guide lines from overall baseline to each community
for y, rate in zip(
    y_positions,
    helix_plot["Repeat_Rate"]
):
    ax.plot(
        [overall_repeat_rate, rate],
        [y, y],
        linewidth=2,
        alpha=0.35
    )

# Plot repeat rates
ax.scatter(
    helix_plot["Repeat_Rate"],
    y_positions,
    s=110,
    zorder=3
)

# Overall baseline
ax.axvline(
    overall_repeat_rate,
    linestyle="--",
    linewidth=1.5,
    label=f"Overall Repeat Rate ({overall_repeat_rate:.1f}%)"
)

# Add labels without overlap
for y, rate in zip(
    y_positions,
    helix_plot["Repeat_Rate"]
):

    if rate < overall_repeat_rate:
        ax.annotate(
            f"{rate:.1f}%",
            xy=(rate, y),
            xytext=(-12, 0),
            textcoords="offset points",
            ha="right",
            va="center",
            fontsize=10,
            fontweight="bold"
        )
    else:
        ax.annotate(
            f"{rate:.1f}%",
            xy=(rate, y),
            xytext=(12, 0),
            textcoords="offset points",
            ha="left",
            va="center",
            fontsize=10,
            fontweight="bold"
        )

# Community labels
ax.set_yticks(y_positions)
ax.set_yticklabels(
    helix_plot["Helix_Community_Analysis"]
)

# Focus chart on observed rates
ax.set_xlim(
    helix_plot["Repeat_Rate"].min() - 2,
    helix_plot["Repeat_Rate"].max() + 2
)

ax.set_title(
    "Repeat Rate by Helix Community",
    fontsize=15,
    fontweight="bold",
    pad=15
)

ax.set_xlabel("Repeat Rate (%)")
ax.set_ylabel("")

ax.grid(
    axis="x",
    linestyle="--",
    alpha=0.20
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)

ax.tick_params(
    axis="y",
    length=0
)

ax.legend(
    frameon=False,
    loc="lower right"
)

plt.tight_layout()
plt.show()

**Observation:** Repeat-giving behaviour is relatively similar across most major Helix Communities. The 500 Doing Fine group records the highest meaningful repeat rate at 18.6%, while 700 Other records a lower rate of 10.9%. However, most large communities remain close to the overall 16.6% baseline.

**Business implication:** Helix Community provides some descriptive supporter profiling but offers weaker differentiation than first-gift value, product, acquisition channel and previous WVA history. It may therefore be useful as supporting segmentation information rather than a primary retention signal. The 13.4% missing coverage should also be considered when interpreting these results.

### 6.5.4 Repeat Giving by Helix Persona

Helix Persona provides a more detailed demographic and lifestyle profile than the broader Helix Community grouping.

Because the dataset contains many persona categories, only personas with at least 300 supporters are compared in detail. This provides a supporting check of whether more granular supporter profiles show meaningful differences in repeat giving, while reducing the risk of over-interpreting very small groups.

In [ ]:
# ---------------------------------------------------------
# 6.5.4 Repeat Giving by Helix Persona
# ---------------------------------------------------------

# Create a clear category for missing persona information
analysis_data["Helix_Persona_Analysis"] = (
    analysis_data["Helix_Persona"]
    .fillna("Missing")
)


# ---------------------------------------------------------
# Summarise repeat behaviour by persona
# ---------------------------------------------------------

persona_summary = (
    analysis_data
    .groupby("Helix_Persona_Analysis")
    .agg(
        Supporters=("Deakin_Supporter_ID", "count"),
        Repeat_Donors=("Repeat_Donor", "sum"),
        Repeat_Rate=("Repeat_Donor", "mean")
    )
    .reset_index()
)

# Convert repeat rate to percentage
persona_summary["Repeat_Rate"] = (
    persona_summary["Repeat_Rate"] * 100
).round(1)

# Compare with overall baseline
persona_summary["Difference_vs_Overall"] = (
    persona_summary["Repeat_Rate"]
    - overall_repeat_rate
).round(1)


# ---------------------------------------------------------
# Keep sufficiently large personas for reliable comparison
# ---------------------------------------------------------

persona_plot = persona_summary[
    (persona_summary["Supporters"] >= 300) &
    (persona_summary["Helix_Persona_Analysis"] != "Missing")
].copy()

# Sort by repeat rate
persona_plot = persona_plot.sort_values(
    "Repeat_Rate",
    ascending=True
).reset_index(drop=True)

print(
    "Personas with at least 300 supporters:",
    len(persona_plot)
)

print("\nREPEAT GIVING BY MAJOR HELIX PERSONA")

display(
    persona_plot.rename(
        columns={
            "Helix_Persona_Analysis": "Helix Persona",
            "Repeat_Donors": "Repeat Donors",
            "Repeat_Rate": "Repeat Rate (%)",
            "Difference_vs_Overall": "Difference vs Overall (pp)"
        }
    )
)


# ---------------------------------------------------------
# Create clean dot plot
# ---------------------------------------------------------

fig, ax = plt.subplots(figsize=(10, 7))

y_positions = np.arange(len(persona_plot))

# Draw guide lines from baseline to each persona
for y, rate in zip(
    y_positions,
    persona_plot["Repeat_Rate"]
):
    ax.plot(
        [overall_repeat_rate, rate],
        [y, y],
        linewidth=2,
        alpha=0.35
    )

# Plot repeat rates
ax.scatter(
    persona_plot["Repeat_Rate"],
    y_positions,
    s=100,
    zorder=3
)

# Overall repeat-rate baseline
ax.axvline(
    overall_repeat_rate,
    linestyle="--",
    linewidth=1.5,
    label=f"Overall Repeat Rate ({overall_repeat_rate:.1f}%)"
)

# Add rate labels
for y, rate in zip(
    y_positions,
    persona_plot["Repeat_Rate"]
):

    if rate < overall_repeat_rate:
        ax.annotate(
            f"{rate:.1f}%",
            xy=(rate, y),
            xytext=(-10, 0),
            textcoords="offset points",
            ha="right",
            va="center",
            fontsize=9.5,
            fontweight="bold"
        )
    else:
        ax.annotate(
            f"{rate:.1f}%",
            xy=(rate, y),
            xytext=(10, 0),
            textcoords="offset points",
            ha="left",
            va="center",
            fontsize=9.5,
            fontweight="bold"
        )

# Persona labels
ax.set_yticks(y_positions)
ax.set_yticklabels(
    persona_plot["Helix_Persona_Analysis"],
    fontsize=9
)

# Keep chart focused on actual values
ax.set_xlim(
    persona_plot["Repeat_Rate"].min() - 2,
    persona_plot["Repeat_Rate"].max() + 2
)

ax.set_title(
    "Repeat Rate by Major Helix Persona",
    fontsize=15,
    fontweight="bold",
    pad=15
)

ax.set_xlabel("Repeat Rate (%)")
ax.set_ylabel("")

ax.grid(
    axis="x",
    linestyle="--",
    alpha=0.20
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)

ax.tick_params(
    axis="y",
    length=0
)

ax.legend(
    frameon=False,
    loc="lower right"
)

plt.tight_layout()
plt.show()

**Observation:** Helix Persona shows more variation in repeat giving than the broader Helix Community classification. Among the major personas, 206 Fit and Fab records the highest repeat rate at 20.8%, while 701 Other records the lowest at 10.9%. However, many personas remain relatively close to the overall 16.6% baseline.

**Business implication:** Detailed demographic profiling may provide some additional segmentation information, but the results are less consistent and less directly actionable than first-gift value, product, acquisition channel and previous WVA history. Helix Persona is therefore retained as supporting evidence rather than treated as a primary retention driver.

### 6.5.5 Repeat Giving by Lewers Segment

Lewers Segment provides WVA's internal supporter segmentation and is examined to determine whether existing supporter groups show meaningful differences in repeat giving.

The analysis is used as a supporting segmentation check and is compared with the overall repeat-giving baseline.

In [ ]:
# ---------------------------------------------------------
# 6.5.5 Repeat Giving by Lewers Segment
# ---------------------------------------------------------

# Create a clear category for missing Lewers information
analysis_data["Lewers_Analysis"] = (
    analysis_data["Lewers_Segment_Description"]
    .fillna("Missing")
)


# ---------------------------------------------------------
# Summarise repeat behaviour by Lewers segment
# ---------------------------------------------------------

lewers_summary = (
    analysis_data
    .groupby("Lewers_Analysis")
    .agg(
        Supporters=("Deakin_Supporter_ID", "count"),
        Repeat_Donors=("Repeat_Donor", "sum"),
        Repeat_Rate=("Repeat_Donor", "mean")
    )
    .reset_index()
)

# Convert repeat rate to percentage
lewers_summary["Repeat_Rate"] = (
    lewers_summary["Repeat_Rate"] * 100
).round(1)

# Compare with overall baseline
lewers_summary["Difference_vs_Overall"] = (
    lewers_summary["Repeat_Rate"]
    - overall_repeat_rate
).round(1)

# Calculate supporter share
lewers_summary["Supporter_Share"] = (
    lewers_summary["Supporters"]
    / len(analysis_data)
    * 100
).round(1)

# Sort summary table by supporter volume
lewers_summary = lewers_summary.sort_values(
    "Supporters",
    ascending=False
)

print("REPEAT GIVING BY LEWERS SEGMENT")

display(
    lewers_summary.rename(
        columns={
            "Lewers_Analysis": "Lewers Segment",
            "Repeat_Donors": "Repeat Donors",
            "Repeat_Rate": "Repeat Rate (%)",
            "Difference_vs_Overall": "Difference vs Overall (pp)",
            "Supporter_Share": "Supporter Share (%)"
        }
    )
)


# ---------------------------------------------------------
# Prepare segments for visual comparison
# ---------------------------------------------------------

lewers_plot = lewers_summary[
    lewers_summary["Lewers_Analysis"] != "Missing"
].copy()

# Shorter labels for the chart
lewers_plot["Chart_Label"] = (
    lewers_plot["Lewers_Analysis"]
    .str.replace("Segment 1 - ", "1 - ", regex=False)
    .str.replace("Segment 2 - ", "2 - ", regex=False)
    .str.replace("Segment 3 - ", "3 - ", regex=False)
    .str.replace("Segment 4 - ", "4 - ", regex=False)
    .str.replace("Segment 5 - ", "5 - ", regex=False)
    .str.replace("Segment 6 - ", "6 - ", regex=False)
)

# Sort by repeat rate
lewers_plot = lewers_plot.sort_values(
    "Repeat_Rate",
    ascending=True
).reset_index(drop=True)


# ---------------------------------------------------------
# Create clean dot plot
# ---------------------------------------------------------

fig, ax = plt.subplots(figsize=(9.5, 6))

y_positions = np.arange(len(lewers_plot))

# Draw guide lines from baseline to each segment
for y, rate in zip(
    y_positions,
    lewers_plot["Repeat_Rate"]
):
    ax.plot(
        [overall_repeat_rate, rate],
        [y, y],
        linewidth=2,
        alpha=0.35
    )

# Plot repeat rates
ax.scatter(
    lewers_plot["Repeat_Rate"],
    y_positions,
    s=110,
    zorder=3
)

# Overall baseline
ax.axvline(
    overall_repeat_rate,
    linestyle="--",
    linewidth=1.5,
    label=f"Overall Repeat Rate ({overall_repeat_rate:.1f}%)"
)

# Add labels without overlap
for y, rate in zip(
    y_positions,
    lewers_plot["Repeat_Rate"]
):

    if rate < overall_repeat_rate:
        ax.annotate(
            f"{rate:.1f}%",
            xy=(rate, y),
            xytext=(-12, 0),
            textcoords="offset points",
            ha="right",
            va="center",
            fontsize=10,
            fontweight="bold"
        )
    else:
        ax.annotate(
            f"{rate:.1f}%",
            xy=(rate, y),
            xytext=(12, 0),
            textcoords="offset points",
            ha="left",
            va="center",
            fontsize=10,
            fontweight="bold"
        )

# Segment labels
ax.set_yticks(y_positions)
ax.set_yticklabels(
    lewers_plot["Chart_Label"]
)

# Focus on actual rate range
ax.set_xlim(
    lewers_plot["Repeat_Rate"].min() - 2,
    lewers_plot["Repeat_Rate"].max() + 2
)

ax.set_title(
    "Repeat Rate by Lewers Segment",
    fontsize=15,
    fontweight="bold",
    pad=15
)

ax.set_xlabel("Repeat Rate (%)")
ax.set_ylabel("")

ax.grid(
    axis="x",
    linestyle="--",
    alpha=0.20
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)

ax.tick_params(
    axis="y",
    length=0
)

ax.legend(
    frameon=False,
    loc="lower right"
)

plt.tight_layout()
plt.show()

**Observation:** Repeat-giving rates vary only modestly across the main Lewers segments. Segment 2 - Pauline records the highest rate at 18.1%, while Segments 3 - Melissa and 5 - Waleed record the lowest at 16.1%. Most segments remain relatively close to the overall 16.6% baseline.

**Business implication:** WVA's existing Lewers segmentation provides useful descriptive context but does not strongly differentiate repeat-giving behaviour in this cohort. It is therefore better treated as supporting segmentation information rather than a primary retention or predictive signal.

## 6.6 Timing of Repeat Giving

The timing of the second qualifying donation is examined to understand when repeat donors return after their first gift.

This analysis helps identify important periods in the supporter journey where WVA may have stronger opportunities to maintain engagement and encourage another donation.

### 6.6.1 Time to Second Gift

The number of days between the first and second qualifying one-off gifts is summarised for repeat donors to understand the typical time required for supporters to return.

In [ ]:
# ---------------------------------------------------------
# 6.6.1 Time to Second Gift
# ---------------------------------------------------------

# Keep repeat donors only
repeat_timing = analysis_data[
    analysis_data["Repeat_Donor"] == 1
].copy()

# Key timing statistics
timing_summary = (
    repeat_timing["Days_to_Repeat"]
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75
        ]
    )
)

print(
    "Repeat donors:",
    f"{len(repeat_timing):,}"
)

print(
    "Minimum days to repeat:",
    int(
        repeat_timing[
            "Days_to_Repeat"
        ].min()
    )
)

print(
    "25th percentile:",
    int(
        repeat_timing[
            "Days_to_Repeat"
        ].quantile(0.25)
    ),
    "days"
)

print(
    "Median:",
    int(
        repeat_timing[
            "Days_to_Repeat"
        ].median()
    ),
    "days"
)

print(
    "75th percentile:",
    int(
        repeat_timing[
            "Days_to_Repeat"
        ].quantile(0.75)
    ),
    "days"
)

print(
    "Maximum days to repeat:",
    int(
        repeat_timing[
            "Days_to_Repeat"
        ].max()
    )
)

**Observation:** Repeat giving occurs across a broad period after the first gift. One-quarter of repeat donors returned within 69 days, the median second gift occurred after 240 days, and 75% of repeat donors returned within one year.

**Business implication:** The first year after a supporter’s initial qualifying gift appears to be the most important retention period. Rather than relying on a single follow-up point, WVA should maintain engagement across this period, with particular attention to both early post-gift engagement and later opportunities as the supporter approaches the first anniversary of their donation.

### 6.6.2 Cumulative Timing of Repeat Giving

Cumulative repeat behaviour is examined at key milestones after the first qualifying gift.

This shows how quickly second donations accumulate across the 24-month follow-up period and helps identify the periods in which continued supporter engagement may be most important.

In [ ]:
# ---------------------------------------------------------
# 6.6.2 Cumulative Timing of Repeat Giving
# ---------------------------------------------------------

# Key follow-up milestones
milestones = {
    "3 Months": 90,
    "6 Months": 180,
    "12 Months": 365,
    "18 Months": 548,
    "24 Months": 731
}


# ---------------------------------------------------------
# Calculate cumulative repeat activity
# ---------------------------------------------------------

milestone_results = []

for label, days in milestones.items():

    cumulative_count = (
        repeat_timing["Days_to_Repeat"] <= days
    ).sum()

    cumulative_percentage = (
        cumulative_count
        / len(repeat_timing)
        * 100
    )

    milestone_results.append(
        {
            "Milestone": label,
            "Days": days,
            "Repeat_Donors_Returned": cumulative_count,
            "Cumulative_Percentage": cumulative_percentage
        }
    )


milestone_data = pd.DataFrame(
    milestone_results
)

milestone_data["Cumulative_Percentage"] = (
    milestone_data["Cumulative_Percentage"]
    .round(1)
)


# ---------------------------------------------------------
# Display milestone table
# ---------------------------------------------------------

print("CUMULATIVE TIMING OF REPEAT GIVING")

display(
    milestone_data[
        [
            "Milestone",
            "Repeat_Donors_Returned",
            "Cumulative_Percentage"
        ]
    ].rename(
        columns={
            "Repeat_Donors_Returned": "Repeat Donors Returned",
            "Cumulative_Percentage": "Cumulative Repeat Donors (%)"
        }
    )
)


# ---------------------------------------------------------
# Create cumulative line chart
# ---------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(9, 5.8)
)

ax.plot(
    milestone_data["Milestone"],
    milestone_data["Cumulative_Percentage"],
    marker="o",
    linewidth=2.5,
    markersize=8
)

# Add percentage labels above each point
for x, y in zip(
    milestone_data["Milestone"],
    milestone_data["Cumulative_Percentage"]
):
    ax.annotate(
        f"{y:.1f}%",
        xy=(x, y),
        xytext=(0, 10),
        textcoords="offset points",
        ha="center",
        fontsize=10.5,
        fontweight="bold"
    )

ax.set_title(
    "Cumulative Timing of Repeat Giving",
    fontsize=15,
    fontweight="bold",
    pad=15
)

ax.set_ylabel(
    "Cumulative Repeat Donors (%)"
)

ax.set_xlabel("")

ax.set_ylim(
    0,
    108
)

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.25
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

**Observation:** Repeat donations accumulate strongly during the first year after the initial gift. Around 29.0% of repeat donors returned within three months, 42.9% within six months and 75.7% within 12 months. A further 15.6% returned between 12 and 18 months, with 91.3% of all repeat donors having returned by that point.

**Business implication:** WVA's retention journey should place particular emphasis on the first 12 months while continuing engagement beyond the first anniversary. The results suggest that repeat giving is not concentrated at only one point in time, supporting a sustained supporter journey rather than a single follow-up campaign.

### 6.6.3 Distribution of Second-Gift Timing

Repeat donors are grouped into approximately three-month intervals based on when their second qualifying donation occurred.

Using similar-length intervals provides a clearer comparison of where repeat-giving activity is concentrated across the 24-month follow-up period.

In [ ]:
# ---------------------------------------------------------
# 6.6.3 Distribution of Second-Gift Timing
# ---------------------------------------------------------

# Approximately equal three-month timing windows
timing_bins = [
    0,
    90,
    180,
    274,
    365,
    456,
    548,
    639,
    731
]

timing_labels = [
    "0–3 Months",
    "3–6 Months",
    "6–9 Months",
    "9–12 Months",
    "12–15 Months",
    "15–18 Months",
    "18–21 Months",
    "21–24 Months"
]


# ---------------------------------------------------------
# Assign each repeat donor to a timing window
# ---------------------------------------------------------

repeat_timing["Repeat_Timing_Window"] = pd.cut(
    repeat_timing["Days_to_Repeat"],
    bins=timing_bins,
    labels=timing_labels,
    include_lowest=True,
    right=True
)


# ---------------------------------------------------------
# Summarise repeat activity
# ---------------------------------------------------------

timing_window_summary = (
    repeat_timing
    .groupby(
        "Repeat_Timing_Window",
        observed=True
    )
    .agg(
        Repeat_Donors=("Deakin_Supporter_ID", "count")
    )
    .reset_index()
)

# Percentage of all repeat donors
timing_window_summary["Share_of_Repeat_Donors"] = (
    timing_window_summary["Repeat_Donors"]
    / len(repeat_timing)
    * 100
).round(1)


print("DISTRIBUTION OF SECOND-GIFT TIMING")

display(
    timing_window_summary.rename(
        columns={
            "Repeat_Timing_Window": "Timing Window",
            "Repeat_Donors": "Repeat Donors",
            "Share_of_Repeat_Donors": "Share of Repeat Donors (%)"
        }
    )
)


# ---------------------------------------------------------
# Create clean bar chart
# ---------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(10, 5.8)
)

bars = ax.bar(
    timing_window_summary["Repeat_Timing_Window"].astype(str),
    timing_window_summary["Share_of_Repeat_Donors"]
)

# Add percentage labels above bars
for bar, percentage in zip(
    bars,
    timing_window_summary["Share_of_Repeat_Donors"]
):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f"{percentage:.1f}%",
        ha="center",
        fontsize=10,
        fontweight="bold"
    )

ax.set_title(
    "Distribution of Second-Gift Timing",
    fontsize=15,
    fontweight="bold",
    pad=15
)

ax.set_ylabel(
    "Share of Repeat Donors (%)"
)

ax.set_xlabel(
    "Time After First Gift"
)

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.25
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.xticks(
    rotation=20,
    ha="right"
)

plt.tight_layout()
plt.show()

**Observation:** Second-gift timing is concentrated in two notable periods. The first three months account for the largest share of repeat donors (29.0%), while a second substantial concentration occurs between nine and twelve months (21.7%). Repeat activity becomes considerably lower after 15 months.

**Business implication:** The results support a retention journey with at least two important phases: strong early engagement after the first gift and renewed engagement as supporters approach the first anniversary of their donation. However, these patterns describe when repeat donations occurred and do not by themselves prove that contacting supporters at these times will cause additional donations.

### 6.6.4 Monthly Pattern of Second Gifts

Second-gift timing is examined at an approximate monthly level to provide a more detailed view of when repeat donors return.

This analysis helps determine whether the broader timing windows contain specific periods of concentrated repeat-giving activity, particularly during the early supporter journey and around the first anniversary of the initial gift.

In [ ]:
# ---------------------------------------------------------
# 6.6.4 Monthly Pattern of Second Gifts
# ---------------------------------------------------------

# Average number of days in a month
days_per_month = 365.25 / 12


# ---------------------------------------------------------
# Convert days to approximate months after first gift
# ---------------------------------------------------------

repeat_timing["Approx_Month_to_Repeat"] = np.ceil(
    repeat_timing["Days_to_Repeat"] / days_per_month
).astype(int)

# Keep values within the 24-month follow-up window
repeat_timing["Approx_Month_to_Repeat"] = (
    repeat_timing["Approx_Month_to_Repeat"]
    .clip(1, 24)
)


# ---------------------------------------------------------
# Summarise repeat donors by month
# ---------------------------------------------------------

monthly_repeat_summary = (
    repeat_timing
    .groupby("Approx_Month_to_Repeat")
    .agg(
        Repeat_Donors=("Deakin_Supporter_ID", "count")
    )
    .reset_index()
)

# Ensure all 24 months appear even if a month has no donors
all_months = pd.DataFrame({
    "Approx_Month_to_Repeat": range(1, 25)
})

monthly_repeat_summary = all_months.merge(
    monthly_repeat_summary,
    on="Approx_Month_to_Repeat",
    how="left"
)

monthly_repeat_summary["Repeat_Donors"] = (
    monthly_repeat_summary["Repeat_Donors"]
    .fillna(0)
    .astype(int)
)

# Calculate share of all repeat donors
monthly_repeat_summary["Share_of_Repeat_Donors"] = (
    monthly_repeat_summary["Repeat_Donors"]
    / len(repeat_timing)
    * 100
).round(1)


# ---------------------------------------------------------
# Display monthly results
# ---------------------------------------------------------

print("MONTHLY PATTERN OF SECOND GIFTS")

display(
    monthly_repeat_summary.rename(
        columns={
            "Approx_Month_to_Repeat": "Month After First Gift",
            "Repeat_Donors": "Repeat Donors",
            "Share_of_Repeat_Donors": "Share of Repeat Donors (%)"
        }
    )
)


# ---------------------------------------------------------
# Create monthly line chart
# ---------------------------------------------------------

fig, ax = plt.subplots(figsize=(11, 5.8))

ax.plot(
    monthly_repeat_summary["Approx_Month_to_Repeat"],
    monthly_repeat_summary["Share_of_Repeat_Donors"],
    marker="o",
    linewidth=2,
    markersize=5
)

# Highlight Month 12 with a reference line
ax.axvline(
    12,
    linestyle="--",
    linewidth=1.3,
    alpha=0.7,
    label="Approx. First Anniversary"
)

ax.set_title(
    "Monthly Pattern of Second Gifts",
    fontsize=15,
    fontweight="bold",
    pad=15
)

ax.set_xlabel("Approximate Month After First Gift")
ax.set_ylabel("Share of Repeat Donors (%)")

ax.set_xticks(range(1, 25))

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.25
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(
    frameon=False
)

plt.tight_layout()
plt.show()

**Observation:** The monthly timing pattern shows two clear peaks in repeat giving. The largest concentration occurs in the first month after the initial gift, accounting for 16.4% of repeat donors. A second strong concentration appears around the first anniversary, with 13.4% returning in Month 12 and a further 8.0% in Month 13.

**Business implication:** WVA could structure the supporter journey around two important phases: strong early stewardship immediately after the first gift, followed by renewed impact messaging and re-engagement as the supporter approaches the first anniversary. The timing pattern should be used to guide testing rather than interpreted as evidence that contact at these points will directly cause another donation.

# 6.7 Relationships Between Important Variables

Earlier EDA examined important supporter characteristics individually.

This section examines selected variables together to determine whether the strongest patterns remain visible when supporter characteristics overlap. This helps identify possible interactions between variables and provides additional evidence for feature selection during predictive modelling.

## 6.7.1 First-Gift Amount and Primary Product

First-gift amount and primary product were both associated with repeat giving in the earlier analysis.

These variables are examined together to determine whether higher first-gift values are still associated with stronger repeat behaviour within different major products.

Only product-and-amount combinations with sufficient supporter numbers are interpreted to avoid drawing conclusions from very small groups.

In [ ]:
# ---------------------------------------------------------
# 6.7.1 First-Gift Amount and Primary Product
# ---------------------------------------------------------

# Keep major products with at least 500 supporters
major_products = (
    product_summary[
        product_summary["Supporters"] >= 500
    ]["First_Gift_Primary_Product"]
    .tolist()
)

amount_product_data = analysis_data[
    analysis_data["First_Gift_Primary_Product"].isin(
        major_products
    )
].copy()


# ---------------------------------------------------------
# Summarise Product × Amount Band combinations
# ---------------------------------------------------------

amount_product_summary = (
    amount_product_data
    .groupby(
        [
            "First_Gift_Primary_Product",
            "First_Gift_Amount_Band"
        ],
        observed=False
    )
    .agg(
        Supporters=("Deakin_Supporter_ID", "count"),
        Repeat_Donors=("Repeat_Donor", "sum"),
        Repeat_Rate=("Repeat_Donor", "mean")
    )
    .reset_index()
)

amount_product_summary["Repeat_Rate"] = (
    amount_product_summary["Repeat_Rate"] * 100
).round(1)


# ---------------------------------------------------------
# Set amount-band order
# ---------------------------------------------------------

amount_order = [
    "$0–50",
    "$51–100",
    "$101–250",
    "$251–500",
    "$501–1,000",
    "$1,001–5,000",
    "$5,000+"
]

amount_product_summary["First_Gift_Amount_Band"] = pd.Categorical(
    amount_product_summary["First_Gift_Amount_Band"],
    categories=amount_order,
    ordered=True
)


# ---------------------------------------------------------
# Keep reliable combinations
# ---------------------------------------------------------

reliable_amount_product = amount_product_summary[
    amount_product_summary["Supporters"] >= 50
].copy()

reliable_amount_product = reliable_amount_product.sort_values(
    [
        "First_Gift_Primary_Product",
        "First_Gift_Amount_Band"
    ]
)

print("RELIABLE PRODUCT × FIRST-GIFT AMOUNT COMBINATIONS")

display(
    reliable_amount_product.rename(
        columns={
            "First_Gift_Primary_Product": "Primary Product",
            "First_Gift_Amount_Band": "First-Gift Amount",
            "Repeat_Donors": "Repeat Donors",
            "Repeat_Rate": "Repeat Rate (%)"
        }
    )
)


# ---------------------------------------------------------
# Create repeat-rate matrix for visual comparison
# ---------------------------------------------------------

rate_matrix = reliable_amount_product.pivot(
    index="First_Gift_Primary_Product",
    columns="First_Gift_Amount_Band",
    values="Repeat_Rate"
)

rate_matrix = rate_matrix.reindex(
    columns=amount_order
)

# Order products by overall supporter volume
product_order = (
    amount_product_data[
        "First_Gift_Primary_Product"
    ]
    .value_counts()
    .index
)

rate_matrix = rate_matrix.reindex(
    product_order
)


# ---------------------------------------------------------
# Create heatmap
# ---------------------------------------------------------

fig, ax = plt.subplots(figsize=(11.5, 6))

heatmap = ax.imshow(
    rate_matrix.values,
    aspect="auto"
)

ax.set_xticks(
    range(len(rate_matrix.columns))
)

ax.set_xticklabels(
    rate_matrix.columns,
    rotation=25,
    ha="right"
)

ax.set_yticks(
    range(len(rate_matrix.index))
)

ax.set_yticklabels(
    rate_matrix.index
)

# Add repeat-rate labels
for i in range(len(rate_matrix.index)):
    for j in range(len(rate_matrix.columns)):

        rate = rate_matrix.iloc[i, j]

        if pd.notna(rate):
            ax.text(
                j,
                i,
                f"{rate:.1f}%",
                ha="center",
                va="center",
                fontsize=9,
                fontweight="bold"
            )

ax.set_title(
    "Repeat Rate by First-Gift Amount and Product",
    fontsize=15,
    fontweight="bold",
    pad=15
)

ax.set_xlabel(
    "First-Gift Amount"
)

ax.set_ylabel(
    "Primary First-Gift Product"
)

colour_bar = plt.colorbar(
    heatmap,
    ax=ax,
    pad=0.02
)

colour_bar.set_label(
    "Repeat Rate (%)"
)

for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.show()

**Observation:** The relationship between first-gift value and repeat giving remains visible within several major products. For WV Regular Giving, Emergency Relief and Child Rescue, higher first-gift values are generally associated with stronger repeat behaviour. This indicates that the overall first-gift amount pattern is not explained only by differences in product mix.

Multiplying Gift Appeal behaves differently. Repeat rates remain very high across its reliable amount bands, including 38.2% among supporters giving \$50 or less. This suggests that the strong repeat behaviour associated with this product is not dependent on high initial gift value.

**Business implication:** First-gift value and primary product provide complementary retention information and should be considered together when prioritising supporters. Higher gift value may strengthen prioritisation within several products, while strong-performing products such as Multiplying Gift Appeal may identify promising supporters even when the initial donation value is relatively low.

## 6.7.2 First-Gift Amount and Previous WVA History

First-gift amount and previous WVA history were both associated with repeat giving in the earlier analysis.

These variables are examined together to determine whether the relationship between first-gift value and repeat giving differs between supporters who joined WVA on their first-gift date and supporters who were already known to WVA beforehand.

In [ ]:
# ---------------------------------------------------------
# 6.7.2 First-Gift Amount and Previous WVA History
# ---------------------------------------------------------

# Keep the two meaningful WVA history groups
relationship_amount_data = analysis_data[
    analysis_data["WVA_History_Before_First_Gift"].isin(
        [
            "Joined on First Gift Date",
            "Known to WVA Before First Gift"
        ]
    )
].copy()


# ---------------------------------------------------------
# Summarise repeat behaviour by amount and WVA history
# ---------------------------------------------------------

relationship_amount_summary = (
    relationship_amount_data
    .groupby(
        [
            "First_Gift_Amount_Band",
            "WVA_History_Before_First_Gift"
        ],
        observed=False
    )
    .agg(
        Supporters=("Deakin_Supporter_ID", "count"),
        Repeat_Donors=("Repeat_Donor", "sum"),
        Repeat_Rate=("Repeat_Donor", "mean")
    )
    .reset_index()
)

# Convert repeat rate to percentage
relationship_amount_summary["Repeat_Rate"] = (
    relationship_amount_summary["Repeat_Rate"] * 100
).round(1)


# ---------------------------------------------------------
# Set logical amount-band order
# ---------------------------------------------------------

amount_order = [
    "$0–50",
    "$51–100",
    "$101–250",
    "$251–500",
    "$501–1,000",
    "$1,001–5,000",
    "$5,000+"
]

relationship_amount_summary["First_Gift_Amount_Band"] = pd.Categorical(
    relationship_amount_summary["First_Gift_Amount_Band"],
    categories=amount_order,
    ordered=True
)

relationship_amount_summary = relationship_amount_summary.sort_values(
    [
        "First_Gift_Amount_Band",
        "WVA_History_Before_First_Gift"
    ]
)


# ---------------------------------------------------------
# Keep reliable combinations for comparison
# ---------------------------------------------------------

reliable_relationship_amount = relationship_amount_summary[
    relationship_amount_summary["Supporters"] >= 50
].copy()

print("RELIABLE FIRST-GIFT AMOUNT × WVA HISTORY COMBINATIONS")

display(
    reliable_relationship_amount.rename(
        columns={
            "First_Gift_Amount_Band": "First-Gift Amount",
            "WVA_History_Before_First_Gift": "WVA History",
            "Repeat_Donors": "Repeat Donors",
            "Repeat_Rate": "Repeat Rate (%)"
        }
    )
)


# ---------------------------------------------------------
# Prepare two groups for chart
# ---------------------------------------------------------

new_supporters = reliable_relationship_amount[
    reliable_relationship_amount[
        "WVA_History_Before_First_Gift"
    ] == "Joined on First Gift Date"
].copy()

known_supporters = reliable_relationship_amount[
    reliable_relationship_amount[
        "WVA_History_Before_First_Gift"
    ] == "Known to WVA Before First Gift"
].copy()


# ---------------------------------------------------------
# Create comparison chart
# ---------------------------------------------------------

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(
    new_supporters["First_Gift_Amount_Band"].astype(str),
    new_supporters["Repeat_Rate"],
    marker="o",
    linewidth=2.4,
    markersize=7,
    label="Joined on First Gift Date"
)

ax.plot(
    known_supporters["First_Gift_Amount_Band"].astype(str),
    known_supporters["Repeat_Rate"],
    marker="o",
    linewidth=2.4,
    markersize=7,
    label="Known to WVA Before First Gift"
)

# Overall repeat-rate baseline
ax.axhline(
    overall_repeat_rate,
    linestyle="--",
    linewidth=1.3,
    label=f"Overall Repeat Rate ({overall_repeat_rate:.1f}%)"
)

ax.set_title(
    "First-Gift Value and Previous WVA History",
    fontsize=15,
    fontweight="bold",
    pad=15
)

ax.set_xlabel("First-Gift Amount")
ax.set_ylabel("Repeat Rate (%)")

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.25
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(
    frameon=False,
    loc="upper left"
)

plt.xticks(
    rotation=20,
    ha="right"
)

plt.tight_layout()
plt.show()

**Observation:** First-gift value shows a clear and consistent relationship with repeat giving among supporters who joined WVA on their first-gift date, with repeat rates increasing from 12.6% in the \$0–50 band to 29.1% in the \$5,000+ band.

Among supporters already known to WVA, the pattern is less consistent. However, previous WVA history appears particularly important among lower-value donors. Supporters already known to WVA who gave \$50 or less recorded a 21.2% repeat rate, compared with 12.6% among supporters joining on their first-gift date.

**Business implication:** First-gift amount appears to be a particularly strong prioritisation signal for newly acquired supporters, while previous WVA familiarity provides additional information that can identify promising supporters even when the initial gift value is relatively low. WVA should therefore avoid using gift value alone to deprioritise previously known supporters.

## 6.7.3 Stability of First-Gift Amount Across Years

Repeat-giving rates changed across the 2021–2023 first-gift cohorts, so the relationship between first-gift amount and repeat giving is checked across individual years.

This helps determine whether first-gift value is a reasonably stable retention signal over time or whether the earlier relationship is mainly driven by one cohort.

In [ ]:
# ---------------------------------------------------------
# 6.7.3 Stability of First-Gift Amount Across Years
# ---------------------------------------------------------

# Use the three complete main yearly cohorts
year_amount_data = analysis_data[
    analysis_data["First_Gift_Year"].isin(
        [2021, 2022, 2023]
    )
].copy()


# ---------------------------------------------------------
# Summarise amount-band behaviour within each year
# ---------------------------------------------------------

year_amount_summary = (
    year_amount_data
    .groupby(
        [
            "First_Gift_Amount_Band",
            "First_Gift_Year"
        ],
        observed=False
    )
    .agg(
        Supporters=("Deakin_Supporter_ID", "count"),
        Repeat_Donors=("Repeat_Donor", "sum"),
        Repeat_Rate=("Repeat_Donor", "mean")
    )
    .reset_index()
)

# Convert repeat rate to percentage
year_amount_summary["Repeat_Rate"] = (
    year_amount_summary["Repeat_Rate"] * 100
).round(1)


# ---------------------------------------------------------
# Set logical amount-band order
# ---------------------------------------------------------

amount_order = [
    "$0–50",
    "$51–100",
    "$101–250",
    "$251–500",
    "$501–1,000",
    "$1,001–5,000",
    "$5,000+"
]

year_amount_summary["First_Gift_Amount_Band"] = pd.Categorical(
    year_amount_summary["First_Gift_Amount_Band"],
    categories=amount_order,
    ordered=True
)

year_amount_summary = year_amount_summary.sort_values(
    [
        "First_Gift_Amount_Band",
        "First_Gift_Year"
    ]
)


# ---------------------------------------------------------
# Keep reliable year × amount combinations
# ---------------------------------------------------------

reliable_year_amount = year_amount_summary[
    year_amount_summary["Supporters"] >= 50
].copy()

print("FIRST-GIFT AMOUNT STABILITY ACROSS YEARS")

display(
    reliable_year_amount.rename(
        columns={
            "First_Gift_Amount_Band": "First-Gift Amount",
            "First_Gift_Year": "First-Gift Year",
            "Repeat_Donors": "Repeat Donors",
            "Repeat_Rate": "Repeat Rate (%)"
        }
    )
)


# ---------------------------------------------------------
# Create comparison chart
# ---------------------------------------------------------

fig, ax = plt.subplots(figsize=(10.5, 6))

for year in [2021, 2022, 2023]:

    year_plot = reliable_year_amount[
        reliable_year_amount["First_Gift_Year"] == year
    ].copy()

    ax.plot(
        year_plot["First_Gift_Amount_Band"].astype(str),
        year_plot["Repeat_Rate"],
        marker="o",
        linewidth=2.2,
        markersize=6,
        label=str(year)
    )

ax.set_title(
    "First-Gift Amount and Repeat Rate Across Years",
    fontsize=15,
    fontweight="bold",
    pad=15
)

ax.set_xlabel("First-Gift Amount")
ax.set_ylabel("Repeat Rate (%)")

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.25
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(
    title="First-Gift Year",
    frameon=False
)

plt.xticks(
    rotation=20,
    ha="right"
)

plt.tight_layout()
plt.show()

**Observation:** The relationship between first-gift value and repeat giving remains broadly consistent across the 2021, 2022 and 2023 cohorts. Lower-value first gifts generally record lower repeat rates, while supporters giving in the higher-value bands tend to show stronger repeat behaviour across all three years.

At the same time, repeat rates decline across most amount bands from 2021 to 2023. This indicates that first-gift amount remains a useful and relatively stable retention signal, even though the overall likelihood of repeat giving has changed over time.

**Business implication:** First-gift value should be retained as an important prioritisation and predictive feature. However, the decline in repeat rates across yearly cohorts also reinforces the need to evaluate predictive models on later time periods so that performance reflects more recent supporter behaviour.

## 6.7.4 Stability of Primary Product Across Years

Primary first-gift product showed meaningful differences in repeat giving during the earlier analysis.

The major products are compared across the 2021, 2022 and 2023 first-gift cohorts to determine whether product-level repeat patterns remain visible over time despite the overall decline in repeat giving.

In [ ]:
# ---------------------------------------------------------
# 6.7.4 Stability of Primary Product Across Years
# ---------------------------------------------------------

# Keep major products with at least 500 supporters overall
major_products = (
    analysis_data["First_Gift_Primary_Product"]
    .value_counts()
)

major_products = major_products[
    major_products >= 500
].index.tolist()


# ---------------------------------------------------------
# Keep major products and complete yearly cohorts
# ---------------------------------------------------------

product_year_data = analysis_data[
    analysis_data["First_Gift_Primary_Product"].isin(
        major_products
    )
    &
    analysis_data["First_Gift_Year"].isin(
        [2021, 2022, 2023]
    )
].copy()


# ---------------------------------------------------------
# Summarise repeat behaviour by product and year
# ---------------------------------------------------------

product_year_summary = (
    product_year_data
    .groupby(
        [
            "First_Gift_Primary_Product",
            "First_Gift_Year"
        ]
    )
    .agg(
        Supporters=("Deakin_Supporter_ID", "count"),
        Repeat_Donors=("Repeat_Donor", "sum"),
        Repeat_Rate=("Repeat_Donor", "mean")
    )
    .reset_index()
)

# Convert repeat rate to percentage
product_year_summary["Repeat_Rate"] = (
    product_year_summary["Repeat_Rate"] * 100
).round(1)


# ---------------------------------------------------------
# Keep reliable product × year combinations
# ---------------------------------------------------------

reliable_product_year = product_year_summary[
    product_year_summary["Supporters"] >= 50
].copy()

reliable_product_year = reliable_product_year.sort_values(
    [
        "First_Gift_Primary_Product",
        "First_Gift_Year"
    ]
)

print("PRIMARY PRODUCT STABILITY ACROSS YEARS")

display(
    reliable_product_year.rename(
        columns={
            "First_Gift_Primary_Product": "Primary Product",
            "First_Gift_Year": "First-Gift Year",
            "Repeat_Donors": "Repeat Donors",
            "Repeat_Rate": "Repeat Rate (%)"
        }
    )
)


# ---------------------------------------------------------
# Create comparison chart
# ---------------------------------------------------------

fig, ax = plt.subplots(figsize=(10.5, 6))

for product in major_products:

    product_plot = reliable_product_year[
        reliable_product_year[
            "First_Gift_Primary_Product"
        ] == product
    ].sort_values(
        "First_Gift_Year"
    )

    # Plot only products with reliable yearly observations
    if len(product_plot) > 0:

        ax.plot(
            product_plot["First_Gift_Year"],
            product_plot["Repeat_Rate"],
            marker="o",
            linewidth=2.2,
            markersize=6,
            label=product
        )


ax.set_title(
    "Primary First-Gift Product Across Yearly Cohorts",
    fontsize=15,
    fontweight="bold",
    pad=15
)

ax.set_xlabel("First-Gift Year")
ax.set_ylabel("Repeat Rate (%)")

ax.set_xticks(
    [2021, 2022, 2023]
)

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.25
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(
    title="Primary Product",
    frameon=False,
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

**Observation:** The relative differences between major primary products remain visible across the 2021, 2022 and 2023 cohorts, even though repeat rates decline over time. Multiplying Gift Appeal remains the strongest product in every year, while WV Regular Giving also performs consistently above several other major products. In contrast, Child Rescue and World Vision Gifts show weaker repeat behaviour, particularly in the 2023 cohort.

**Business implication:** Primary product appears to be a meaningful and reasonably stable retention signal rather than a pattern driven by only one cohort. It should therefore be retained as an important candidate predictive feature. However, the general decline in repeat rates across years also indicates that time effects should be considered when interpreting product performance and evaluating predictive models.

# 6.8 EDA Summary and Key Findings

The exploratory analysis identified several meaningful patterns associated with repeat one-off giving.

### Stronger retention signals

- **First-gift value:** Repeat giving generally increased with first-gift amount. Supporters giving higher amounts showed stronger repeat behaviour, although lower-value supporters still represented a substantial share of the overall donor population.

- **Primary first-gift product:** Repeat rates varied considerably across products. Multiplying Gift Appeal consistently recorded particularly strong repeat behaviour, while large groups such as World Vision Gifts and Emergency Relief generally performed closer to or below the overall baseline.

- **Acquisition channel:** Repeat behaviour differed substantially across channels. Events Internal performed strongly, while Retail Stand Internal showed consistently low repeat rates even when compared within the same products. Website remained strategically important because it accounted for the majority of supporters.

- **Previous WVA history:** Supporters already known to WVA before their first qualifying one-off gift generally showed stronger repeat behaviour than supporters joining on their first-gift date. Supporters known to WVA for less than three years showed particularly strong repeat rates.

### Timing of repeat giving

Repeat giving was concentrated within the first year after the initial donation. Around 29.0% of repeat donors returned within three months and 75.7% returned within twelve months.

The monthly analysis showed two particularly important periods: immediately after the first gift and around the first anniversary. This supports a sustained supporter journey rather than relying on a single follow-up contact.

### Supporting rather than primary signals

Geographic location, Helix Community, Helix Persona and Lewers Segment showed some variation in repeat behaviour but generally provided weaker differentiation than first-gift behaviour, acquisition characteristics and previous WVA history.

### Changes over time

Overall repeat rates declined across the 2021–2023 cohorts. However, the relationships associated with first-gift amount and primary product remained broadly visible across years.

This indicates that these variables may provide useful predictive information, while also highlighting the importance of evaluating predictive models using later time periods rather than relying only on random train-test splits.

### Implication for predictive modelling

The modelling stage will therefore prioritise information available at or before the first qualifying one-off gift, particularly first-gift amount, product, acquisition channel and previous WVA history.

The predictive model will be used as a decision-support tool to rank supporters by repeat-giving propensity rather than attempting to predict individual behaviour with certainty.

# 7. Data Preparation and Feature Selection

The exploratory analysis identified several supporter characteristics associated with repeat one-off giving.

Before predictive modelling, the data must be prepared carefully so that the model uses only information that would genuinely be available at the time of the first qualifying gift.

This section prepares the modelling dataset by:

- defining the prediction target and prediction point;
- removing identifiers and future information that could cause data leakage;
- reviewing and selecting useful first-gift, acquisition and previous WVA relationship features;
- creating any additional derived variables that may improve predictive performance;
- preparing a broader historical modelling population so that different training time windows can be compared;
- retaining only features that are methodologically appropriate and useful for modelling.

The final prepared dataset will then be used in Section 8 to compare multiple classification models and historical training windows.

## 7.1 Target and Prediction Point

The target variable is `Repeat_Donor`, where:

- **1 = another qualifying one-off donation within 24 months**
- **0 = no qualifying repeat donation within 24 months**

The prediction is made at the end of the supporter's first qualifying giving occasion. Only information available at or before this point can be used by the model.

In [ ]:
# ---------------------------------------------------------
# 7.1 Check Target Distribution
# ---------------------------------------------------------

target_summary = (
    analysis_data["Repeat_Donor"]
    .value_counts()
    .sort_index()
    .rename_axis("Repeat_Donor")
    .reset_index(name="Supporters")
)

target_summary["Percentage"] = (
    target_summary["Supporters"]
    / len(analysis_data)
    * 100
).round(1)

display(target_summary)

print("Total supporters:", f"{len(analysis_data):,}")
print(
    "Repeat rate:",
    f"{analysis_data['Repeat_Donor'].mean() * 100:.1f}%"
)

## 7.2 Historical Modelling Population

Recent historical supporters from 2019 onward are retained so that alternative training windows can be compared while avoiding very old donor behaviour.

In [ ]:
# ---------------------------------------------------------
# 7.2 Create Historical Modelling Population
# ---------------------------------------------------------

# Use recent history only for model development
historical_start = pd.Timestamp("2019-01-01")

# Keep eligible supporters with a complete 24-month outcome window
historical_supporters = eligible_supporters[
    (eligible_supporters["Index_Donation_Date"] >= historical_start) &
    (eligible_supporters["Index_Donation_Date"] <= outcome_cutoff)
].copy()

# Add the end of each supporter's 24-month repeat window
historical_supporters["Repeat_Window_End_Date"] = (
    historical_supporters["Index_Donation_Date"]
    + pd.DateOffset(months=24)
)


# ---------------------------------------------------------
# Create the repeat-donor target
# ---------------------------------------------------------

historical_future = qualifying_one_off.merge(
    historical_supporters[
        [
            "Deakin_Supporter_ID",
            "Index_Donation_Date",
            "Repeat_Window_End_Date"
        ]
    ],
    on="Deakin_Supporter_ID",
    how="inner"
)

historical_future = historical_future[
    (historical_future["Fulfilment_Date"] >
     historical_future["Index_Donation_Date"]) &
    (historical_future["Fulfilment_Date"] <=
     historical_future["Repeat_Window_End_Date"])
]

historical_first_repeat = (
    historical_future
    .groupby("Deakin_Supporter_ID", as_index=False)
    .agg(
        Repeat_Donation_Date=("Fulfilment_Date", "min")
    )
)

historical_supporters = historical_supporters.merge(
    historical_first_repeat,
    on="Deakin_Supporter_ID",
    how="left"
)

historical_supporters["Repeat_Donor"] = (
    historical_supporters["Repeat_Donation_Date"]
    .notna()
    .astype(int)
)


# ---------------------------------------------------------
# Identify first-gift acquisition business unit
# ---------------------------------------------------------

historical_first_day = qualifying_one_off.merge(
    historical_supporters[
        ["Deakin_Supporter_ID", "Index_Donation_Date"]
    ],
    on="Deakin_Supporter_ID",
    how="inner"
)

historical_first_day = historical_first_day[
    historical_first_day["Fulfilment_Date"] ==
    historical_first_day["Index_Donation_Date"]
].copy()

historical_first_day = historical_first_day.sort_values(
    ["Deakin_Supporter_ID", "Total_Paid"],
    ascending=[True, False]
)

historical_acquisition = (
    historical_first_day
    .drop_duplicates("Deakin_Supporter_ID")
    [
        [
            "Deakin_Supporter_ID",
            "Acquisition_Segment_Business_Unit"
        ]
    ]
    .rename(
        columns={
            "Acquisition_Segment_Business_Unit":
            "First_Gift_Acquisition_Business_Unit"
        }
    )
)

historical_supporters = historical_supporters.merge(
    historical_acquisition,
    on="Deakin_Supporter_ID",
    how="left",
    validate="one_to_one"
)


# ---------------------------------------------------------
# Keep Mass supporters only
# ---------------------------------------------------------

historical_model_base = historical_supporters[
    historical_supporters[
        "First_Gift_Acquisition_Business_Unit"
    ] == "Mass"
].copy()

historical_model_base = historical_model_base.rename(
    columns={
        "Index_Donation_Date": "First_Gift_Date",
        "Relationship_At_Index": "Relationship_At_First_Gift"
    }
)

historical_model_base["First_Gift_Year"] = (
    historical_model_base["First_Gift_Date"].dt.year
)


# ---------------------------------------------------------
# Review recent historical population
# ---------------------------------------------------------

year_summary = (
    historical_model_base
    .groupby("First_Gift_Year")
    .agg(
        Supporters=("Deakin_Supporter_ID", "count"),
        Repeat_Donors=("Repeat_Donor", "sum"),
        Repeat_Rate=("Repeat_Donor", "mean")
    )
    .reset_index()
)

year_summary["Repeat_Rate"] = (
    year_summary["Repeat_Rate"] * 100
).round(1)

display(year_summary)

print(
    "Historical Mass supporters:",
    f"{len(historical_model_base):,}"
)

## 7.3 Create First-Gift Features

Supporters may behave differently depending on how they first engage with WVA.

The historical modelling population is therefore enriched with information from each supporter's first qualifying giving occasion, including donation amount, channel, product, transaction count and acquisition context.

When several transactions occur on the first donation date, they are treated as one giving occasion. The total amount is combined, while the largest transaction is used to identify the primary product and acquisition context.

Only information available on the first giving day is used so that these variables can safely be included as potential model predictors.

In [ ]:
# ---------------------------------------------------------
# 7.3 Create Historical First-Gift Features
# ---------------------------------------------------------

# Keep first-day transactions for Mass supporters only
historical_first_transactions = historical_first_day[
    historical_first_day["Deakin_Supporter_ID"].isin(
        historical_model_base["Deakin_Supporter_ID"]
    )
].copy()


# ---------------------------------------------------------
# Identify the largest transaction on the first-gift date
# ---------------------------------------------------------

historical_first_sorted = historical_first_transactions.sort_values(
    ["Deakin_Supporter_ID", "Total_Paid"],
    ascending=[True, False]
)

# Use the largest transaction to represent primary product
# and acquisition context
historical_primary = (
    historical_first_sorted
    .drop_duplicates("Deakin_Supporter_ID")
    [
        [
            "Deakin_Supporter_ID",
            "Product_Category",
            "Product_Code_Clean",
            "Attribution_Channel",
            "Acquisition_Segment_Team",
            "Acquisition_Segment_Sub_Team"
        ]
    ]
    .rename(
        columns={
            "Product_Category": "First_Gift_Primary_Product",
            "Product_Code_Clean": "First_Gift_Primary_Product_Code",
            "Attribution_Channel": "First_Gift_Channel_First",
            "Acquisition_Segment_Team": "First_Gift_Acquisition_Team",
            "Acquisition_Segment_Sub_Team": "First_Gift_Acquisition_Sub_Team"
        }
    )
)


# ---------------------------------------------------------
# Summarise the complete first giving occasion
# ---------------------------------------------------------

historical_first_features = (
    historical_first_transactions
    .groupby("Deakin_Supporter_ID", as_index=False)
    .agg(
        First_Gift_Total_Amount=("Total_Paid", "sum"),
        First_Gift_Transaction_Count=("DMRT_Commitment_Key", "count"),
        First_Gift_Channel_Count=("Attribution_Channel", "nunique"),
        First_Gift_Product_Count=("Product_Code_Clean", "nunique"),
        Age_At_First_Gift=("Age_At_Acquisition", "first")
    )
)

# Add primary transaction information
historical_first_features = historical_first_features.merge(
    historical_primary,
    on="Deakin_Supporter_ID",
    how="left",
    validate="one_to_one"
)

# If more than one channel was used, label it MULTIPLE
historical_first_features["First_Gift_Channel"] = np.where(
    historical_first_features["First_Gift_Channel_Count"] == 1,
    historical_first_features["First_Gift_Channel_First"].astype(str),
    "MULTIPLE"
)

# Flag supporters who donated across more than one product
historical_first_features["First_Gift_Multiple_Products"] = (
    historical_first_features["First_Gift_Product_Count"] > 1
).astype(int)


# ---------------------------------------------------------
# Add first-gift features to historical modelling data
# ---------------------------------------------------------

historical_model_data = historical_model_base.merge(
    historical_first_features,
    on="Deakin_Supporter_ID",
    how="left",
    validate="one_to_one"
)


# ---------------------------------------------------------
# Validate
# ---------------------------------------------------------

print(
    "Historical modelling records:",
    f"{len(historical_model_data):,}"
)

print(
    "Unique supporters:",
    f"{historical_model_data['Deakin_Supporter_ID'].nunique():,}"
)

print("\nMissing key first-gift features:")

print(
    historical_model_data[
        [
            "First_Gift_Total_Amount",
            "First_Gift_Channel",
            "First_Gift_Primary_Product",
            "First_Gift_Primary_Product_Code"
        ]
    ].isna().sum()
)

# Portfolio privacy edit: supporter-level preview suppressed.

## 7.4 Add Previous WVA Relationship Features

First-gift behaviour alone may not fully describe a supporter.

Previous WVA history is therefore added to show whether the supporter was newly acquired or already known to WVA before their first qualifying one-off gift. This includes relationship duration and previous completed ongoing commitments.

Only information available before the first gift is used.

In [ ]:
# ---------------------------------------------------------
# 7.4 Add Previous WVA Relationship Features
# ---------------------------------------------------------

# ---------------------------------------------------------
# Add WVA database join date
# ---------------------------------------------------------

history_demographics = demographics[
    [
        "Deakin_Supporter_ID",
        "Date_Joined_WVA_Database"
    ]
].copy()

history_demographics["Date_Joined_WVA_Database"] = pd.to_datetime(
    history_demographics["Date_Joined_WVA_Database"],
    errors="coerce"
)

historical_model_data = historical_model_data.merge(
    history_demographics,
    on="Deakin_Supporter_ID",
    how="left",
    validate="one_to_one"
)


# ---------------------------------------------------------
# Calculate how long WVA knew the supporter before first gift
# ---------------------------------------------------------

historical_model_data["Days_With_WVA_Before_First_Gift"] = (
    historical_model_data["First_Gift_Date"]
    - historical_model_data["Date_Joined_WVA_Database"]
).dt.days

# Negative values are logically inconsistent, so treat them as missing
historical_model_data.loc[
    historical_model_data["Days_With_WVA_Before_First_Gift"] < 0,
    "Days_With_WVA_Before_First_Gift"
] = np.nan


# ---------------------------------------------------------
# Create simple WVA history category
# ---------------------------------------------------------

historical_model_data["WVA_History_Before_First_Gift"] = np.select(
    [
        historical_model_data["Days_With_WVA_Before_First_Gift"] == 0,
        historical_model_data["Days_With_WVA_Before_First_Gift"] > 0
    ],
    [
        "Joined on First Gift Date",
        "Known to WVA Before First Gift"
    ],
    default="Unknown / Inconsistent"
)


# ---------------------------------------------------------
# Create easy-to-interpret WVA tenure bands
# ---------------------------------------------------------

historical_model_data["WVA_Tenure_Band"] = pd.cut(
    historical_model_data["Days_With_WVA_Before_First_Gift"],
    bins=[
        -0.1,
        0,
        365,
        1095,
        1825,
        float("inf")
    ],
    labels=[
        "Joined on First Gift Date",
        "1 Year or Less",
        "1–3 Years",
        "3–5 Years",
        "More than 5 Years"
    ]
)

historical_model_data["WVA_Tenure_Band"] = (
    historical_model_data["WVA_Tenure_Band"]
    .astype("object")
    .fillna("Unknown / Inconsistent")
)


# ---------------------------------------------------------
# Count completed ongoing commitments before first gift
# ---------------------------------------------------------

ongoing_before_gift = ongoing[
    ongoing["Date_Consistency_Flag"] == "Valid"
].copy()

ongoing_before_gift = ongoing_before_gift.merge(
    historical_model_data[
        [
            "Deakin_Supporter_ID",
            "First_Gift_Date"
        ]
    ],
    on="Deakin_Supporter_ID",
    how="inner"
)

# Check whether the commitment had ended before the first gift
ongoing_before_gift["Ended_Before_First_Gift"] = (
    (
        ongoing_before_gift["Fulfilment_Ended_Date"].notna()
        &
        (
            ongoing_before_gift["Fulfilment_Ended_Date"]
            < ongoing_before_gift["First_Gift_Date"]
        )
    )
    |
    (
        ongoing_before_gift["Closure_Date"].notna()
        &
        (
            ongoing_before_gift["Closure_Date"]
            < ongoing_before_gift["First_Gift_Date"]
        )
    )
)

# Keep only commitments that started and ended before the first gift
ongoing_before_gift = ongoing_before_gift[
    (
        ongoing_before_gift["Fulfilment_Date"]
        <= ongoing_before_gift["First_Gift_Date"]
    )
    &
    ongoing_before_gift["Ended_Before_First_Gift"]
]

previous_ongoing_summary = (
    ongoing_before_gift
    .groupby("Deakin_Supporter_ID", as_index=False)
    .agg(
        Previous_Ongoing_Commitment_Count=(
            "DMRT_Commitment_Key",
            "nunique"
        )
    )
)

historical_model_data = historical_model_data.merge(
    previous_ongoing_summary,
    on="Deakin_Supporter_ID",
    how="left",
    validate="one_to_one"
)

historical_model_data["Previous_Ongoing_Commitment_Count"] = (
    historical_model_data["Previous_Ongoing_Commitment_Count"]
    .fillna(0)
    .astype(int)
)


# ---------------------------------------------------------
# Validate the new relationship features
# ---------------------------------------------------------

print("Historical modelling records:",
      f"{len(historical_model_data):,}")

print(
    "Missing history duration:",
    historical_model_data[
        "Days_With_WVA_Before_First_Gift"
    ].isna().sum()
)

print("\nWVA history groups:")
print(
    historical_model_data[
        "WVA_History_Before_First_Gift"
    ].value_counts()
)

print(
    "\nSupporters with previous ongoing commitments:",
    (
        historical_model_data[
            "Previous_Ongoing_Commitment_Count"
        ] > 0
    ).sum()
)

# Portfolio privacy edit: supporter-level preview suppressed.

## 7.5 Remove Data Leakage and Unsuitable Variables

Before selecting predictors, variables that reveal the future outcome or do not provide genuine predictive information are excluded.

Supporter identifiers, repeat-donation outcome dates and other post-prediction information cannot be used by the model. Variables with poor historical reliability or excessive missing data will also be excluded from the main feature set.

In [ ]:
# ---------------------------------------------------------
# 7.5 Define Direct Model Exclusions
# ---------------------------------------------------------

direct_exclusions = {
    "Deakin_Supporter_ID": "Supporter identifier only",
    "Repeat_Donor": "Target variable",
    "Repeat_Donation_Date": "Future outcome information",
    "Repeat_Window_End_Date": "Derived outcome-window date",
    "First_Gift_Date": "Raw date used only for temporal splitting",
    "Date_Joined_WVA_Database": "Raw date already represented by WVA history features",
    "Age_At_First_Gift": "Too much missing historical information"
}

exclusion_summary = pd.DataFrame(
    direct_exclusions.items(),
    columns=["Variable", "Reason"]
)

display(exclusion_summary)

print(
    "Available modelling columns:",
    historical_model_data.shape[1]
)

print(
    "Directly excluded columns:",
    len(direct_exclusions)
)

## 7.6 Define Candidate Feature Sets

Rather than using every available variable, several feature sets are prepared for comparison.

The first set contains core first-gift behaviour. Additional sets progressively include previous WVA relationship history and more detailed acquisition information.

This allows the modelling stage to test whether extra variables genuinely improve prediction rather than assuming that a larger feature set is better.

In [ ]:
# ---------------------------------------------------------
# 7.6 Prepare Candidate Features
# ---------------------------------------------------------

# Create log versions of highly skewed numeric variables
historical_model_data["First_Gift_Total_Amount_Log"] = np.log1p(
    historical_model_data["First_Gift_Total_Amount"]
)

historical_model_data["First_Gift_Transaction_Count_Log"] = np.log1p(
    historical_model_data["First_Gift_Transaction_Count"]
)


# ---------------------------------------------------------
# Feature Set 1: Core first-gift behaviour
# ---------------------------------------------------------

feature_set_1 = [
    "First_Gift_Total_Amount_Log",
    "First_Gift_Transaction_Count_Log",
    "First_Gift_Product_Count",
    "First_Gift_Channel",
    "First_Gift_Primary_Product"
]


# ---------------------------------------------------------
# Feature Set 2: Add previous WVA relationship
# ---------------------------------------------------------

feature_set_2 = feature_set_1 + [
    "Relationship_At_First_Gift",
    "WVA_Tenure_Band",
    "Previous_Ongoing_Commitment_Count"
]


# ---------------------------------------------------------
# Feature Set 3: Add detailed acquisition information
# ---------------------------------------------------------

feature_set_3 = feature_set_2 + [
    "First_Gift_Primary_Product_Code",
    "First_Gift_Acquisition_Sub_Team"
]


# ---------------------------------------------------------
# Review candidate feature sets
# ---------------------------------------------------------

feature_sets = {
    "Set 1 - First Gift": feature_set_1,
    "Set 2 - First Gift + WVA History": feature_set_2,
    "Set 3 - Extended": feature_set_3
}

for name, features in feature_sets.items():
    print(f"\n{name}")
    print("-" * len(name))

    for feature in features:
        print("•", feature)

## 7.7 Prepare the Final Modelling Dataset

The data is organised chronologically to reflect how the model would be used in practice.

Supporters from 2019–2021 form the model-development period, while 2022 is reserved for validation and model selection. Supporters from 2023–2024 are kept separate as the final unseen test period.

Within the development period, alternative training windows will be compared in the modelling stage to determine whether older historical data improves prediction.

In [ ]:
# ---------------------------------------------------------
# 7.7 Prepare Final Modelling Dataset
# ---------------------------------------------------------

modelling_ready = historical_model_data.copy()


# ---------------------------------------------------------
# Assign temporal modelling periods
# ---------------------------------------------------------

modelling_ready["Model_Period"] = np.select(
    [
        modelling_ready["First_Gift_Year"].between(2019, 2021),
        modelling_ready["First_Gift_Year"] == 2022,
        modelling_ready["First_Gift_Year"].between(2023, 2024)
    ],
    [
        "Development: 2019–2021",
        "Validation: 2022",
        "Final Test: 2023–2024"
    ],
    default="Outside Modelling Period"
)


# ---------------------------------------------------------
# Check missing values in the largest candidate feature set
# ---------------------------------------------------------

feature_missing = (
    modelling_ready[feature_set_3]
    .isna()
    .sum()
    .reset_index()
)

feature_missing.columns = [
    "Feature",
    "Missing_Values"
]

display(feature_missing)


# ---------------------------------------------------------
# Summarise modelling periods
# ---------------------------------------------------------

period_summary = (
    modelling_ready
    .groupby("Model_Period")
    .agg(
        Supporters=("Deakin_Supporter_ID", "count"),
        Repeat_Donors=("Repeat_Donor", "sum"),
        Repeat_Rate=("Repeat_Donor", "mean")
    )
    .reset_index()
)

period_summary["Repeat_Rate"] = (
    period_summary["Repeat_Rate"] * 100
).round(1)

period_order = [
    "Development: 2019–2021",
    "Validation: 2022",
    "Final Test: 2023–2024"
]

period_summary["Model_Period"] = pd.Categorical(
    period_summary["Model_Period"],
    categories=period_order,
    ordered=True
)

period_summary = period_summary.sort_values(
    "Model_Period"
)

display(period_summary)


# ---------------------------------------------------------
# Final checks
# ---------------------------------------------------------

print(
    "Final modelling records:",
    f"{len(modelling_ready):,}"
)

print(
    "Unique supporters:",
    f"{modelling_ready['Deakin_Supporter_ID'].nunique():,}"
)

print(
    "Target missing values:",
    modelling_ready["Repeat_Donor"].isna().sum()
)

# 8. Predictive Modelling

## 8.1 Temporal Training and Validation Strategy

A time-based modelling approach is used so that earlier supporter behaviour is used to predict later supporters.

Three training periods are compared: 2021 only, 2020–2021 and 2019–2021. Each is evaluated using the same 2022 validation population.

Supporters from 2023–2024 are kept separate as the final unseen test population.

In [ ]:
# ---------------------------------------------------------
# 8.1 Define Temporal Training and Validation Sets
# ---------------------------------------------------------

# Validation population
validation_data = modelling_ready[
    modelling_ready["First_Gift_Year"] == 2022
].copy()

# Final test population
# Keep untouched until the final model is selected
final_test_data = modelling_ready[
    modelling_ready["First_Gift_Year"].between(2023, 2024)
].copy()


# ---------------------------------------------------------
# Create alternative training windows
# ---------------------------------------------------------

training_windows = {
    "2021 Only": modelling_ready[
        modelling_ready["First_Gift_Year"] == 2021
    ].copy(),

    "2020–2021": modelling_ready[
        modelling_ready["First_Gift_Year"].between(2020, 2021)
    ].copy(),

    "2019–2021": modelling_ready[
        modelling_ready["First_Gift_Year"].between(2019, 2021)
    ].copy()
}


# ---------------------------------------------------------
# Summarise training windows
# ---------------------------------------------------------

training_summary = []

for window_name, data in training_windows.items():

    training_summary.append({
        "Training_Window": window_name,
        "Supporters": len(data),
        "Repeat_Donors": data["Repeat_Donor"].sum(),
        "Repeat_Rate": round(
            data["Repeat_Donor"].mean() * 100,
            1
        )
    })

training_summary = pd.DataFrame(training_summary)

display(training_summary)


# ---------------------------------------------------------
# Check validation and final test populations
# ---------------------------------------------------------

print(
    "Validation 2022:",
    f"{len(validation_data):,}",
    "supporters |",
    f"{validation_data['Repeat_Donor'].mean() * 100:.1f}% repeat"
)

print(
    "Final Test 2023–2024:",
    f"{len(final_test_data):,}",
    "supporters |",
    f"{final_test_data['Repeat_Donor'].mean() * 100:.1f}% repeat"
)

## 8.2 Model Selection and Evaluation Metrics

Three classification models are compared: Logistic Regression, Random Forest and CatBoost. These models provide different levels of complexity and allow comparison between a simple baseline and more flexible tree-based approaches.

Model performance is assessed using Accuracy, Precision, Recall, F1-score, Cohen's Kappa and ROC-AUC. Lift and Top-K performance are also examined because the business objective is to rank supporters by their likelihood of making another qualifying one-off donation within 24 months.

In [ ]:
# ---------------------------------------------------------
# 8.2 Load Modelling Libraries
# ---------------------------------------------------------

# Install CatBoost in Colab if required
!pip -q install catboost

# Core modelling tools
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier

# Evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    cohen_kappa_score,
    roc_auc_score
)

print("Modelling libraries loaded successfully.")

## 8.3 Feature Preparation

The candidate predictors contain both numeric and categorical variables.

For Logistic Regression and Random Forest, categorical variables are converted into numeric indicators using one-hot encoding. Numeric variables are standardised for Logistic Regression, while Random Forest uses their original numeric scale.

CatBoost can handle categorical variables directly, so the original category values are retained for this model.

In [ ]:
# ---------------------------------------------------------
# 8.3 Define Numeric and Categorical Features
# ---------------------------------------------------------

# Numeric candidate variables
numeric_candidates = [
    "First_Gift_Total_Amount_Log",
    "First_Gift_Transaction_Count_Log",
    "First_Gift_Product_Count",
    "Previous_Ongoing_Commitment_Count"
]

# Function to separate numeric and categorical variables
# for any of our three feature sets
def get_feature_types(features):

    numeric_features = [
        feature for feature in features
        if feature in numeric_candidates
    ]

    categorical_features = [
        feature for feature in features
        if feature not in numeric_candidates
    ]

    return numeric_features, categorical_features


# ---------------------------------------------------------
# Create preprocessing for each feature set
# ---------------------------------------------------------

preprocessing = {}

for set_name, features in feature_sets.items():

    numeric_features, categorical_features = (
        get_feature_types(features)
    )

    # Logistic Regression:
    # scale numeric variables + encode categories
    logistic_preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                StandardScaler(),
                numeric_features
            ),
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore"
                ),
                categorical_features
            )
        ]
    )

    # Random Forest:
    # keep numeric values as they are + encode categories
    forest_preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                "passthrough",
                numeric_features
            ),
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore"
                ),
                categorical_features
            )
        ]
    )

    preprocessing[set_name] = {
        "Logistic": logistic_preprocessor,
        "Random Forest": forest_preprocessor,
        "Numeric": numeric_features,
        "Categorical": categorical_features
    }


# ---------------------------------------------------------
# Review feature types
# ---------------------------------------------------------

feature_type_summary = []

for set_name, details in preprocessing.items():

    feature_type_summary.append({
        "Feature_Set": set_name,
        "Total_Features":
            len(feature_sets[set_name]),
        "Numeric_Features":
            len(details["Numeric"]),
        "Categorical_Features":
            len(details["Categorical"])
    })

feature_type_summary = pd.DataFrame(
    feature_type_summary
)

display(feature_type_summary)

print("\nCatBoost will use the same feature sets")
print("without one-hot encoding the categorical variables.")

## 8.4 Logistic Regression

Logistic Regression is used as the first predictive model and provides a simple baseline for comparison.

Each model produces a repeat-donation probability for every supporter. Because the optimal classification threshold may differ from the standard 0.50 cut-off, thresholds from 0.05 to 0.95 are evaluated using the 2022 validation population.

The threshold with the strongest Cohen's Kappa is retained for each experiment, with F1-score used as a secondary criterion. Model performance is then compared using Accuracy, Precision, Recall, F1-score, Cohen's Kappa, ROC-AUC and Top 10% Lift.

In [ ]:
# ---------------------------------------------------------
# 8.4 Logistic Regression with Threshold Selection
# ---------------------------------------------------------

from sklearn.base import clone

logistic_results = []


# ---------------------------------------------------------
# Test every training window and feature set
# ---------------------------------------------------------

for window_name, train_data in training_windows.items():

    for feature_set_name, features in feature_sets.items():

        # ---------------------------------------------
        # Prepare training and validation data
        # ---------------------------------------------

        X_train = train_data[features]
        y_train = train_data["Repeat_Donor"]

        X_valid = validation_data[features]
        y_valid = validation_data["Repeat_Donor"]


        # ---------------------------------------------
        # Build Logistic Regression model
        # ---------------------------------------------

        model_preprocessor = clone(
            preprocessing[feature_set_name]["Logistic"]
        )

        logistic_model = Pipeline(
            steps=[
                (
                    "preprocessing",
                    model_preprocessor
                ),
                (
                    "model",
                    LogisticRegression(
                        max_iter=1000,
                        random_state=42
                    )
                )
            ]
        )


        # ---------------------------------------------
        # Train model
        # ---------------------------------------------

        logistic_model.fit(
            X_train,
            y_train
        )


        # ---------------------------------------------
        # Predict repeat-donation probabilities
        # ---------------------------------------------

        valid_probability = logistic_model.predict_proba(
            X_valid
        )[:, 1]


        # ---------------------------------------------
        # ROC-AUC
        # ---------------------------------------------

        auc = roc_auc_score(
            y_valid,
            valid_probability
        )


        # ---------------------------------------------
        # Test different classification thresholds
        # ---------------------------------------------

        threshold_results = []

        thresholds = np.round(
            np.arange(0.05, 0.96, 0.01),
            2
        )

        for threshold in thresholds:

            valid_prediction = (
                valid_probability >= threshold
            ).astype(int)

            threshold_results.append({
                "Threshold": threshold,

                "Accuracy": accuracy_score(
                    y_valid,
                    valid_prediction
                ),

                "Precision": precision_score(
                    y_valid,
                    valid_prediction,
                    zero_division=0
                ),

                "Recall": recall_score(
                    y_valid,
                    valid_prediction,
                    zero_division=0
                ),

                "F1": f1_score(
                    y_valid,
                    valid_prediction,
                    zero_division=0
                ),

                "Kappa": cohen_kappa_score(
                    y_valid,
                    valid_prediction
                )
            })


        threshold_results = pd.DataFrame(
            threshold_results
        )


        # ---------------------------------------------
        # Select best threshold
        # Highest Kappa, then F1
        # ---------------------------------------------

        best_threshold_row = (
            threshold_results
            .sort_values(
                ["Kappa", "F1"],
                ascending=False
            )
            .iloc[0]
        )

        best_threshold = (
            best_threshold_row["Threshold"]
        )


        # ---------------------------------------------
        # Top 10% Lift
        # ---------------------------------------------

        ranking = pd.DataFrame({
            "Actual": y_valid.values,
            "Probability": valid_probability
        }).sort_values(
            "Probability",
            ascending=False
        )

        top_10_count = max(
            1,
            int(np.ceil(len(ranking) * 0.10))
        )

        top_10 = ranking.head(
            top_10_count
        )

        overall_repeat_rate = (
            ranking["Actual"].mean()
        )

        top_10_repeat_rate = (
            top_10["Actual"].mean()
        )

        top_10_lift = (
            top_10_repeat_rate
            / overall_repeat_rate
        )


        # ---------------------------------------------
        # Save experiment result
        # ---------------------------------------------

        logistic_results.append({

            "Training_Window":
                window_name,

            "Feature_Set":
                feature_set_name,

            "Best_Threshold":
                round(best_threshold, 2),

            "Accuracy":
                round(
                    best_threshold_row["Accuracy"],
                    4
                ),

            "Precision":
                round(
                    best_threshold_row["Precision"],
                    4
                ),

            "Recall":
                round(
                    best_threshold_row["Recall"],
                    4
                ),

            "F1":
                round(
                    best_threshold_row["F1"],
                    4
                ),

            "Kappa":
                round(
                    best_threshold_row["Kappa"],
                    4
                ),

            "ROC_AUC":
                round(
                    auc,
                    4
                ),

            "Top_10_Repeat_Rate":
                round(
                    top_10_repeat_rate * 100,
                    1
                ),

            "Top_10_Lift":
                round(
                    top_10_lift,
                    2
                )
        })


# ---------------------------------------------------------
# Compare all Logistic Regression experiments
# ---------------------------------------------------------

logistic_results = pd.DataFrame(
    logistic_results
)

logistic_results = logistic_results.sort_values(
    [
        "Kappa",
        "ROC_AUC",
        "Top_10_Lift"
    ],
    ascending=False
).reset_index(drop=True)

display(logistic_results)

## 8.5 Random Forest

Random Forest is used as a second model to capture more complex and non-linear relationships between supporter characteristics.

The model combines predictions from many decision trees. The same training windows, feature sets and validation approach used for Logistic Regression are retained to ensure a fair comparison.

For each experiment, classification thresholds from 0.05 to 0.95 are evaluated using the 2022 validation population, with Cohen's Kappa used as the primary threshold-selection criterion.

In [ ]:
# ---------------------------------------------------------
# 8.5 Random Forest with Threshold Selection
# ---------------------------------------------------------

random_forest_results = []


# ---------------------------------------------------------
# Test every training window and feature set
# ---------------------------------------------------------

for window_name, train_data in training_windows.items():

    for feature_set_name, features in feature_sets.items():

        # ---------------------------------------------
        # Prepare training and validation data
        # ---------------------------------------------

        X_train = train_data[features]
        y_train = train_data["Repeat_Donor"]

        X_valid = validation_data[features]
        y_valid = validation_data["Repeat_Donor"]


        # ---------------------------------------------
        # Build Random Forest model
        # ---------------------------------------------

        model_preprocessor = clone(
            preprocessing[feature_set_name]["Random Forest"]
        )

        random_forest_model = Pipeline(
            steps=[
                (
                    "preprocessing",
                    model_preprocessor
                ),
                (
                    "model",
                    RandomForestClassifier(
                        n_estimators=300,
                        max_depth=8,
                        min_samples_leaf=20,
                        random_state=42,
                        n_jobs=-1
                    )
                )
            ]
        )


        # ---------------------------------------------
        # Train model
        # ---------------------------------------------

        random_forest_model.fit(
            X_train,
            y_train
        )


        # ---------------------------------------------
        # Predict validation probabilities
        # ---------------------------------------------

        valid_probability = random_forest_model.predict_proba(
            X_valid
        )[:, 1]


        # ---------------------------------------------
        # ROC-AUC
        # ---------------------------------------------

        auc = roc_auc_score(
            y_valid,
            valid_probability
        )


        # ---------------------------------------------
        # Test classification thresholds
        # ---------------------------------------------

        threshold_results = []

        thresholds = np.round(
            np.arange(0.05, 0.96, 0.01),
            2
        )

        for threshold in thresholds:

            valid_prediction = (
                valid_probability >= threshold
            ).astype(int)

            threshold_results.append({

                "Threshold": threshold,

                "Accuracy": accuracy_score(
                    y_valid,
                    valid_prediction
                ),

                "Precision": precision_score(
                    y_valid,
                    valid_prediction,
                    zero_division=0
                ),

                "Recall": recall_score(
                    y_valid,
                    valid_prediction,
                    zero_division=0
                ),

                "F1": f1_score(
                    y_valid,
                    valid_prediction,
                    zero_division=0
                ),

                "Kappa": cohen_kappa_score(
                    y_valid,
                    valid_prediction
                )
            })


        threshold_results = pd.DataFrame(
            threshold_results
        )


        # ---------------------------------------------
        # Select best threshold
        # Highest Kappa, then F1
        # ---------------------------------------------

        best_threshold_row = (
            threshold_results
            .sort_values(
                ["Kappa", "F1"],
                ascending=False
            )
            .iloc[0]
        )

        best_threshold = (
            best_threshold_row["Threshold"]
        )


        # ---------------------------------------------
        # Top 10% Lift
        # ---------------------------------------------

        ranking = pd.DataFrame({
            "Actual": y_valid.values,
            "Probability": valid_probability
        }).sort_values(
            "Probability",
            ascending=False
        )

        top_10_count = max(
            1,
            int(np.ceil(len(ranking) * 0.10))
        )

        top_10 = ranking.head(
            top_10_count
        )

        overall_repeat_rate = (
            ranking["Actual"].mean()
        )

        top_10_repeat_rate = (
            top_10["Actual"].mean()
        )

        top_10_lift = (
            top_10_repeat_rate
            / overall_repeat_rate
        )


        # ---------------------------------------------
        # Save results
        # ---------------------------------------------

        random_forest_results.append({

            "Training_Window":
                window_name,

            "Feature_Set":
                feature_set_name,

            "Best_Threshold":
                round(best_threshold, 2),

            "Accuracy":
                round(
                    best_threshold_row["Accuracy"],
                    4
                ),

            "Precision":
                round(
                    best_threshold_row["Precision"],
                    4
                ),

            "Recall":
                round(
                    best_threshold_row["Recall"],
                    4
                ),

            "F1":
                round(
                    best_threshold_row["F1"],
                    4
                ),

            "Kappa":
                round(
                    best_threshold_row["Kappa"],
                    4
                ),

            "ROC_AUC":
                round(
                    auc,
                    4
                ),

            "Top_10_Repeat_Rate":
                round(
                    top_10_repeat_rate * 100,
                    1
                ),

            "Top_10_Lift":
                round(
                    top_10_lift,
                    2
                )
        })


# ---------------------------------------------------------
# Compare Random Forest experiments
# ---------------------------------------------------------

random_forest_results = pd.DataFrame(
    random_forest_results
)

random_forest_results = random_forest_results.sort_values(
    [
        "Kappa",
        "ROC_AUC",
        "Top_10_Lift"
    ],
    ascending=False
).reset_index(drop=True)

display(random_forest_results)

## 8.6 CatBoost

CatBoost is used as the third predictive model. Unlike Random Forest, which builds many independent decision trees, CatBoost builds trees sequentially so that each new tree attempts to improve errors made by earlier trees.

CatBoost can also work directly with categorical variables such as donation channel, product and previous WVA relationship. The same training windows, feature sets and 2022 validation population are retained for a consistent comparison.

Classification thresholds from 0.05 to 0.95 are again evaluated, with Cohen's Kappa used as the primary threshold-selection criterion.

In [ ]:
# ---------------------------------------------------------
# 8.6 CatBoost with Threshold Selection
# ---------------------------------------------------------

catboost_results = []


# ---------------------------------------------------------
# Test every training window and feature set
# ---------------------------------------------------------

for window_name, train_data in training_windows.items():

    for feature_set_name, features in feature_sets.items():

        # ---------------------------------------------
        # Prepare training and validation data
        # ---------------------------------------------

        X_train = train_data[features].copy()
        y_train = train_data["Repeat_Donor"]

        X_valid = validation_data[features].copy()
        y_valid = validation_data["Repeat_Donor"]


        # ---------------------------------------------
        # Identify categorical variables
        # ---------------------------------------------

        numeric_features, categorical_features = (
            get_feature_types(features)
        )


        # ---------------------------------------------
        # Build CatBoost model
        # ---------------------------------------------

        catboost_model = CatBoostClassifier(
            iterations=300,
            depth=6,
            learning_rate=0.05,
            loss_function="Logloss",
            random_seed=42,
            verbose=0,
            allow_writing_files=False
        )


        # ---------------------------------------------
        # Train model
        # ---------------------------------------------

        catboost_model.fit(
            X_train,
            y_train,
            cat_features=categorical_features
        )


        # ---------------------------------------------
        # Predict validation probabilities
        # ---------------------------------------------

        valid_probability = catboost_model.predict_proba(
            X_valid
        )[:, 1]


        # ---------------------------------------------
        # ROC-AUC
        # ---------------------------------------------

        auc = roc_auc_score(
            y_valid,
            valid_probability
        )


        # ---------------------------------------------
        # Test classification thresholds
        # ---------------------------------------------

        threshold_results = []

        thresholds = np.round(
            np.arange(0.05, 0.96, 0.01),
            2
        )

        for threshold in thresholds:

            valid_prediction = (
                valid_probability >= threshold
            ).astype(int)

            threshold_results.append({

                "Threshold": threshold,

                "Accuracy": accuracy_score(
                    y_valid,
                    valid_prediction
                ),

                "Precision": precision_score(
                    y_valid,
                    valid_prediction,
                    zero_division=0
                ),

                "Recall": recall_score(
                    y_valid,
                    valid_prediction,
                    zero_division=0
                ),

                "F1": f1_score(
                    y_valid,
                    valid_prediction,
                    zero_division=0
                ),

                "Kappa": cohen_kappa_score(
                    y_valid,
                    valid_prediction
                )
            })


        threshold_results = pd.DataFrame(
            threshold_results
        )


        # ---------------------------------------------
        # Select best threshold
        # Highest Kappa, then F1
        # ---------------------------------------------

        best_threshold_row = (
            threshold_results
            .sort_values(
                ["Kappa", "F1"],
                ascending=False
            )
            .iloc[0]
        )

        best_threshold = (
            best_threshold_row["Threshold"]
        )


        # ---------------------------------------------
        # Top 10% Lift
        # ---------------------------------------------

        ranking = pd.DataFrame({
            "Actual": y_valid.values,
            "Probability": valid_probability
        }).sort_values(
            "Probability",
            ascending=False
        )

        top_10_count = max(
            1,
            int(np.ceil(len(ranking) * 0.10))
        )

        top_10 = ranking.head(
            top_10_count
        )

        overall_repeat_rate = (
            ranking["Actual"].mean()
        )

        top_10_repeat_rate = (
            top_10["Actual"].mean()
        )

        top_10_lift = (
            top_10_repeat_rate
            / overall_repeat_rate
        )


        # ---------------------------------------------
        # Save results
        # ---------------------------------------------

        catboost_results.append({

            "Training_Window":
                window_name,

            "Feature_Set":
                feature_set_name,

            "Best_Threshold":
                round(
                    best_threshold,
                    2
                ),

            "Accuracy":
                round(
                    best_threshold_row["Accuracy"],
                    4
                ),

            "Precision":
                round(
                    best_threshold_row["Precision"],
                    4
                ),

            "Recall":
                round(
                    best_threshold_row["Recall"],
                    4
                ),

            "F1":
                round(
                    best_threshold_row["F1"],
                    4
                ),

            "Kappa":
                round(
                    best_threshold_row["Kappa"],
                    4
                ),

            "ROC_AUC":
                round(
                    auc,
                    4
                ),

            "Top_10_Repeat_Rate":
                round(
                    top_10_repeat_rate * 100,
                    1
                ),

            "Top_10_Lift":
                round(
                    top_10_lift,
                    2
                )
        })


# ---------------------------------------------------------
# Compare CatBoost experiments
# ---------------------------------------------------------

catboost_results = pd.DataFrame(
    catboost_results
)

catboost_results = catboost_results.sort_values(
    [
        "Kappa",
        "ROC_AUC",
        "Top_10_Lift"
    ],
    ascending=False
).reset_index(drop=True)

display(catboost_results)

## 8.7 Model Comparison

Across the three algorithms, the 2019–2021 training period with Feature Set 2 produced the strongest or near-strongest validation ranking performance.

Feature Set 2 combines first-gift behaviour with previous WVA relationship information without the additional complexity of the extended variables in Feature Set 3.

The three models are therefore compared using the same training period and feature set to provide a consistent like-for-like comparison.

In [ ]:
# ---------------------------------------------------------
# 8.7 Compare Models Using the Same Configuration
# ---------------------------------------------------------

# Add model names to each result table
logistic_compare = logistic_results.copy()
logistic_compare["Model"] = "Logistic Regression"

forest_compare = random_forest_results.copy()
forest_compare["Model"] = "Random Forest"

catboost_compare = catboost_results.copy()
catboost_compare["Model"] = "CatBoost"


# Combine all results
all_model_results = pd.concat(
    [
        logistic_compare,
        forest_compare,
        catboost_compare
    ],
    ignore_index=True
)


# ---------------------------------------------------------
# Compare the same training window and feature set
# ---------------------------------------------------------

model_comparison = all_model_results[
    (all_model_results["Training_Window"] == "2019–2021") &
    (
        all_model_results["Feature_Set"] ==
        "Set 2 - First Gift + WVA History"
    )
].copy()


# Select useful columns
model_comparison = model_comparison[
    [
        "Model",
        "Best_Threshold",
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "Kappa",
        "ROC_AUC",
        "Top_10_Repeat_Rate",
        "Top_10_Lift"
    ]
]


# Sort primarily by ROC-AUC
model_comparison = model_comparison.sort_values(
    "ROC_AUC",
    ascending=False
).reset_index(drop=True)

display(model_comparison)

## 8.8 Probability Ranking and Top-K Performance

Because the objective is to prioritise supporters for retention activity, model performance is also evaluated using probability rankings.

For each model, supporters in the 2022 validation population are ranked from highest to lowest predicted repeat-donation probability. Performance is then assessed across the Top 10%, 20%, 30% and 40% of supporters.

Repeat rate, lift and capture rate are used to show how effectively each model concentrates future repeat donors within progressively larger priority groups.

In [ ]:
# ---------------------------------------------------------
# 8.8 Compare Top-K Ranking Performance
# ---------------------------------------------------------

# Use the common comparison setup
selected_features = feature_sets[
    "Set 2 - First Gift + WVA History"
]

train_compare = training_windows[
    "2019–2021"
].copy()

X_train = train_compare[selected_features]
y_train = train_compare["Repeat_Donor"]

X_valid = validation_data[selected_features]
y_valid = validation_data["Repeat_Donor"]


# ---------------------------------------------------------
# 1. Logistic Regression
# ---------------------------------------------------------

logistic_final_compare = Pipeline(
    steps=[
        (
            "preprocessing",
            clone(
                preprocessing[
                    "Set 2 - First Gift + WVA History"
                ]["Logistic"]
            )
        ),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

logistic_final_compare.fit(
    X_train,
    y_train
)

logistic_probability = (
    logistic_final_compare.predict_proba(
        X_valid
    )[:, 1]
)


# ---------------------------------------------------------
# 2. Random Forest
# ---------------------------------------------------------

forest_final_compare = Pipeline(
    steps=[
        (
            "preprocessing",
            clone(
                preprocessing[
                    "Set 2 - First Gift + WVA History"
                ]["Random Forest"]
            )
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                max_depth=8,
                min_samples_leaf=20,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

forest_final_compare.fit(
    X_train,
    y_train
)

forest_probability = (
    forest_final_compare.predict_proba(
        X_valid
    )[:, 1]
)


# ---------------------------------------------------------
# 3. CatBoost
# ---------------------------------------------------------

numeric_features, categorical_features = (
    get_feature_types(
        selected_features
    )
)

catboost_final_compare = CatBoostClassifier(
    iterations=300,
    depth=6,
    learning_rate=0.05,
    loss_function="Logloss",
    random_seed=42,
    verbose=0,
    allow_writing_files=False
)

catboost_final_compare.fit(
    X_train,
    y_train,
    cat_features=categorical_features
)

catboost_probability = (
    catboost_final_compare.predict_proba(
        X_valid
    )[:, 1]
)


# ---------------------------------------------------------
# Store model probabilities
# ---------------------------------------------------------

model_probabilities = {
    "Logistic Regression": logistic_probability,
    "Random Forest": forest_probability,
    "CatBoost": catboost_probability
}


# ---------------------------------------------------------
# Calculate Top-K performance
# ---------------------------------------------------------

top_k_results = []

overall_repeat_rate = y_valid.mean()
total_repeat_donors = y_valid.sum()

top_k_levels = [
    0.10,
    0.20,
    0.30,
    0.40
]


for model_name, probabilities in model_probabilities.items():

    ranking = pd.DataFrame({
        "Actual": y_valid.values,
        "Probability": probabilities
    }).sort_values(
        "Probability",
        ascending=False
    ).reset_index(drop=True)


    for top_k in top_k_levels:

        supporters_targeted = int(
            np.ceil(
                len(ranking) * top_k
            )
        )

        targeted_group = ranking.head(
            supporters_targeted
        )

        repeat_donors_captured = (
            targeted_group["Actual"].sum()
        )

        repeat_rate = (
            targeted_group["Actual"].mean()
        )

        lift = (
            repeat_rate
            / overall_repeat_rate
        )

        capture_rate = (
            repeat_donors_captured
            / total_repeat_donors
        )


        top_k_results.append({

            "Model":
                model_name,

            "Priority_Group":
                f"Top {int(top_k * 100)}%",

            "Supporters_Targeted":
                supporters_targeted,

            "Repeat_Donors_Captured":
                int(repeat_donors_captured),

            "Repeat_Rate":
                round(
                    repeat_rate * 100,
                    1
                ),

            "Lift":
                round(
                    lift,
                    2
                ),

            "Capture_Rate":
                round(
                    capture_rate * 100,
                    1
                )
        })


# ---------------------------------------------------------
# Display results
# ---------------------------------------------------------

top_k_results = pd.DataFrame(
    top_k_results
)

display(top_k_results)

## 8.9 Final Model Selection

Logistic Regression was selected as the final model because it provided the strongest overall balance between classification performance, supporter ranking and interpretability.

Although Random Forest produced a slightly higher ROC-AUC, the difference was small. Logistic Regression achieved the highest Top 10% lift, stronger Top 20% capture, and the best F1-score and Cohen's Kappa in the like-for-like comparison.

The selected model therefore uses the 2019–2021 training period with Feature Set 2, which combines first-gift behaviour with previous WVA relationship information.

## 8.10 Final Model Evaluation

After model selection, the chosen Logistic Regression model is retrained using supporters from 2019–2022 so that all available development information is used.

The model is then evaluated once on the untouched 2023–2024 population. The validation-selected threshold of 0.36 is retained rather than being adjusted using the final test data.

This provides a more realistic assessment of how the model may perform when applied to newer supporters.

In [ ]:
# ---------------------------------------------------------
# 8.10 Train Final Logistic Regression Model
# ---------------------------------------------------------

# Selected configuration
final_features = feature_sets[
    "Set 2 - First Gift + WVA History"
]

final_threshold = 0.36


# ---------------------------------------------------------
# Final training population: 2019–2022
# ---------------------------------------------------------

final_training_data = modelling_ready[
    modelling_ready["First_Gift_Year"].between(2019, 2022)
].copy()


# Predictors and target
X_final_train = final_training_data[
    final_features
]

y_final_train = final_training_data[
    "Repeat_Donor"
]

X_final_test = final_test_data[
    final_features
]

y_final_test = final_test_data[
    "Repeat_Donor"
]


# ---------------------------------------------------------
# Build final Logistic Regression model
# ---------------------------------------------------------

final_logistic_model = Pipeline(
    steps=[
        (
            "preprocessing",
            clone(
                preprocessing[
                    "Set 2 - First Gift + WVA History"
                ]["Logistic"]
            )
        ),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)


# ---------------------------------------------------------
# Train model
# ---------------------------------------------------------

final_logistic_model.fit(
    X_final_train,
    y_final_train
)


# ---------------------------------------------------------
# Predict 2023–2024 supporters
# ---------------------------------------------------------

final_test_probability = (
    final_logistic_model.predict_proba(
        X_final_test
    )[:, 1]
)

final_test_prediction = (
    final_test_probability >= final_threshold
).astype(int)


# ---------------------------------------------------------
# Calculate final test metrics
# ---------------------------------------------------------

final_metrics = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "Cohen's Kappa",
        "ROC-AUC"
    ],

    "Score": [
        accuracy_score(
            y_final_test,
            final_test_prediction
        ),

        precision_score(
            y_final_test,
            final_test_prediction,
            zero_division=0
        ),

        recall_score(
            y_final_test,
            final_test_prediction,
            zero_division=0
        ),

        f1_score(
            y_final_test,
            final_test_prediction,
            zero_division=0
        ),

        cohen_kappa_score(
            y_final_test,
            final_test_prediction
        ),

        roc_auc_score(
            y_final_test,
            final_test_probability
        )
    ]
})


final_metrics["Score"] = (
    final_metrics["Score"]
    .round(4)
)


# ---------------------------------------------------------
# Display final results
# ---------------------------------------------------------

print(
    "Final training supporters:",
    f"{len(final_training_data):,}"
)

print(
    "Final test supporters:",
    f"{len(final_test_data):,}"
)

print(
    "Final test repeat rate:",
    f"{y_final_test.mean() * 100:.1f}%"
)

print(
    "Fixed classification threshold:",
    final_threshold
)

display(final_metrics)

## 8.11 Final Top-K Performance

The final model is also evaluated using ranked supporter groups because WVA can prioritise retention activity according to available campaign capacity.

Supporters in the 2023–2024 test population are ranked by predicted repeat-donation probability. Top 10%, 20%, 30% and 40% groups are evaluated using repeat rate, lift and capture rate.

This provides a practical view of how effectively the model can concentrate future repeat donors within smaller priority groups.

In [ ]:
# ---------------------------------------------------------
# 8.11 Final Top-K Performance
# ---------------------------------------------------------

# Create final ranking
final_ranking = pd.DataFrame({
    "Actual": y_final_test.values,
    "Probability": final_test_probability
}).sort_values(
    "Probability",
    ascending=False
).reset_index(drop=True)


# Overall final-test information
overall_repeat_rate = final_ranking[
    "Actual"
].mean()

total_repeat_donors = final_ranking[
    "Actual"
].sum()


# Priority group sizes
top_k_levels = [
    0.10,
    0.20,
    0.30,
    0.40
]


final_top_k_results = []


for top_k in top_k_levels:

    # Number of supporters in this priority group
    supporters_targeted = int(
        np.ceil(
            len(final_ranking) * top_k
        )
    )

    # Highest-ranked supporters
    targeted_group = final_ranking.head(
        supporters_targeted
    )

    # Number of real repeat donors captured
    repeat_donors_captured = (
        targeted_group["Actual"].sum()
    )

    # Repeat rate inside priority group
    repeat_rate = (
        targeted_group["Actual"].mean()
    )

    # Improvement over normal repeat rate
    lift = (
        repeat_rate
        / overall_repeat_rate
    )

    # Percentage of all repeat donors captured
    capture_rate = (
        repeat_donors_captured
        / total_repeat_donors
    )


    final_top_k_results.append({

        "Priority_Group":
            f"Top {int(top_k * 100)}%",

        "Supporters_Targeted":
            supporters_targeted,

        "Repeat_Donors_Captured":
            int(repeat_donors_captured),

        "Repeat_Rate":
            round(
                repeat_rate * 100,
                1
            ),

        "Lift":
            round(
                lift,
                2
            ),

        "Capture_Rate":
            round(
                capture_rate * 100,
                1
            )
    })


final_top_k_results = pd.DataFrame(
    final_top_k_results
)

print(
    "Overall final-test repeat rate:",
    f"{overall_repeat_rate * 100:.1f}%"
)

display(final_top_k_results)

## 8.12 Supporter Priority Segmentation

The predicted probabilities can be converted into practical supporter priority groups rather than relying only on a binary repeat/non-repeat classification.

Based on validation ranking performance, the highest-ranked 20% of supporters are classified as High Priority, the next 20% as Nurture, and the remaining 60% as Standard. This allows retention effort to be differentiated according to predicted repeat-donation potential.

These groups represent relative priority rather than certainty that an individual supporter will or will not donate again.

In [ ]:
# ---------------------------------------------------------
# 8.12 Create Supporter Priority Segments
# ---------------------------------------------------------

# Create supporter-level ranking table
supporter_segments = final_test_data[
    [
        "Deakin_Supporter_ID",
        "Repeat_Donor"
    ]
].copy()

supporter_segments["Repeat_Probability"] = (
    final_test_probability
)


# ---------------------------------------------------------
# Rank supporters from highest to lowest probability
# ---------------------------------------------------------

supporter_segments = supporter_segments.sort_values(
    "Repeat_Probability",
    ascending=False
).reset_index(drop=True)

supporter_segments["Rank"] = (
    np.arange(1, len(supporter_segments) + 1)
)

supporter_segments["Rank_Percentile"] = (
    supporter_segments["Rank"]
    / len(supporter_segments)
)


# ---------------------------------------------------------
# Create priority segments
# ---------------------------------------------------------

supporter_segments["Priority_Segment"] = np.select(
    [
        supporter_segments["Rank_Percentile"] <= 0.20,
        supporter_segments["Rank_Percentile"] <= 0.40
    ],
    [
        "High Priority - Top 20%",
        "Nurture - Next 20%"
    ],
    default="Standard - Remaining 60%"
)


# Keep segments in logical order
segment_order = [
    "High Priority - Top 20%",
    "Nurture - Next 20%",
    "Standard - Remaining 60%"
]

supporter_segments["Priority_Segment"] = pd.Categorical(
    supporter_segments["Priority_Segment"],
    categories=segment_order,
    ordered=True
)


# ---------------------------------------------------------
# Summarise segment performance
# ---------------------------------------------------------

segment_summary = (
    supporter_segments
    .groupby(
        "Priority_Segment",
        observed=True
    )
    .agg(
        Supporters=("Repeat_Donor", "size"),
        Repeat_Donors=("Repeat_Donor", "sum"),
        Repeat_Rate=("Repeat_Donor", "mean")
    )
    .reset_index()
)


# Overall repeat rate
overall_repeat_rate = (
    supporter_segments["Repeat_Donor"].mean()
)


# Additional business measures
segment_summary["Supporter_Share"] = (
    segment_summary["Supporters"]
    / len(supporter_segments)
    * 100
)

segment_summary["Repeat_Rate"] = (
    segment_summary["Repeat_Rate"]
    * 100
)

segment_summary["Lift"] = (
    segment_summary["Repeat_Rate"]
    / (overall_repeat_rate * 100)
)

segment_summary["Capture_Rate"] = (
    segment_summary["Repeat_Donors"]
    / supporter_segments["Repeat_Donor"].sum()
    * 100
)


# Round results
segment_summary[
    [
        "Supporter_Share",
        "Repeat_Rate",
        "Lift",
        "Capture_Rate"
    ]
] = segment_summary[
    [
        "Supporter_Share",
        "Repeat_Rate",
        "Lift",
        "Capture_Rate"
    ]
].round(1)


display(segment_summary)

## 8.13 Model Feature Importance

Permutation importance is used to examine which variables contribute most to the final model's predictive ranking.

For each predictor, its values are randomly shuffled while the remaining variables are left unchanged. A larger reduction in ROC-AUC indicates that the model relies more strongly on that variable.

These results indicate predictive importance rather than causal effects and are interpreted alongside the earlier exploratory analysis.

In [ ]:
# ---------------------------------------------------------
# 8.13 Permutation Feature Importance
# ---------------------------------------------------------

from sklearn.inspection import permutation_importance


# Calculate importance using the final unseen test population
importance_result = permutation_importance(
    final_logistic_model,
    X_final_test,
    y_final_test,
    scoring="roc_auc",
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)


# Create summary table
feature_importance = pd.DataFrame({
    "Feature": final_features,
    "Importance": importance_result.importances_mean
})


# Sort from most important to least important
feature_importance = (
    feature_importance
    .sort_values(
        "Importance",
        ascending=False
    )
    .reset_index(drop=True)
)


# Round for easier reading
feature_importance["Importance"] = (
    feature_importance["Importance"].round(4)
)

display(feature_importance)

## 8.14 Feature Importance Stability

A brief stability check is performed to determine whether the main predictive variables remain broadly important across different time periods.

Permutation importance from the 2022 validation population is compared with the 2023–2024 final test population. The purpose is not to expect identical importance values, but to confirm whether the main predictors remain reasonably consistent over time.

In [ ]:
# ---------------------------------------------------------
# 8.14 Feature Importance Stability Check
# ---------------------------------------------------------

# Importance on 2022 validation data
# Model was trained only on 2019–2021
validation_importance_result = permutation_importance(
    logistic_final_compare,
    X_valid,
    y_valid,
    scoring="roc_auc",
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

validation_importance = pd.DataFrame({
    "Feature": selected_features,
    "Validation_2022_Importance":
        validation_importance_result.importances_mean
})


# ---------------------------------------------------------
# Final-test importance from Section 8.13
# ---------------------------------------------------------

final_importance = feature_importance[
    ["Feature", "Importance"]
].copy()

final_importance = final_importance.rename(
    columns={
        "Importance": "Final_2023_24_Importance"
    }
)


# ---------------------------------------------------------
# Compare the two periods
# ---------------------------------------------------------

importance_stability = validation_importance.merge(
    final_importance,
    on="Feature",
    how="left"
)

# Create ranks
importance_stability["Validation_Rank"] = (
    importance_stability[
        "Validation_2022_Importance"
    ]
    .rank(
        ascending=False,
        method="min"
    )
    .astype(int)
)

importance_stability["Final_Rank"] = (
    importance_stability[
        "Final_2023_24_Importance"
    ]
    .rank(
        ascending=False,
        method="min"
    )
    .astype(int)
)


# Round values
importance_stability[
    [
        "Validation_2022_Importance",
        "Final_2023_24_Importance"
    ]
] = importance_stability[
    [
        "Validation_2022_Importance",
        "Final_2023_24_Importance"
    ]
].round(4)


# Sort using final-test importance
importance_stability = (
    importance_stability
    .sort_values(
        "Final_Rank"
    )
    .reset_index(drop=True)
)

display(importance_stability)

## 8.15 Predictive Modelling Summary

Logistic Regression was selected as the final model because it provided the strongest overall balance between supporter ranking, classification performance and interpretability.

On the unseen 2023–2024 test population, the model achieved a ROC-AUC of 0.6266. Although this indicates moderate rather than strong individual-level prediction, the ranking was useful for prioritisation. The highest-ranked 10% of supporters achieved a 24.0% repeat rate compared with 13.8% overall, representing a 1.74 lift.

The model also produced meaningful supporter segments. The High Priority group had a 23.2% repeat rate, compared with 15.5% for the Nurture group and 10.1% for the Standard group.

Feature importance was reasonably stable across the validation and final test periods. First-gift product, donation amount and channel consistently emerged as the strongest predictors. The model should therefore be used as a prioritisation tool rather than as a deterministic prediction of whether an individual supporter will repeat.

# 9. Recommendations

The recommendations below are mainly based on the EDA findings, because the EDA gives the clearest picture of how one-off donors behave. The predictive model is then used to help WVA decide which supporters should receive more attention.

## 9.1 Create a Clear First-Year Journey After the First Donation

Most repeat donations happened relatively early. Around 75.7% of repeat donations occurred within the first 12 months, and 91.3% occurred within 18 months.

This means WVA should not simply wait and hope that a one-off donor gives again. After the first donation, the supporter should enter a planned follow-up journey.

For example, WVA can first thank the donor, then show how their donation created impact, continue useful communication, and later provide another relevant donation opportunity. The first-gift anniversary can also be used as a reminder to reconnect with the supporter.

The exact timing should be tested, but the main focus should be on keeping the supporter engaged during the first year.

---

## 9.2 Use Different Follow-Up Journeys for Different Products

The type of product connected with the first donation showed a strong relationship with repeat giving. It was also one of the most important variables in the predictive model.

For example, MGI supporters showed stronger repeat behaviour, while some products such as WVGifts and Child Rescue showed lower repeat rates.

WVA should therefore avoid sending exactly the same follow-up communication to every donor.

Instead, communication should remain connected to the supporter’s original reason for giving. WVA should also study what is working well in stronger-performing products such as MGI and test whether similar ideas can improve weaker product journeys.

---

## 9.3 Improve the Follow-Up Journey for Retail Stand Donors

Retail Stand Internal supporters had a much lower repeat rate than many other acquisition channels.

This means the problem may not only be getting the first donation, but what happens after the supporter gives.

WVA should review the Retail Stand journey, including how the supporter is introduced to WVA, what information they receive after donating, whether they clearly understand the impact of their donation, and how they are contacted afterwards.

A new welcome and follow-up journey for Retail Stand supporters should then be tested against the current approach to see whether repeat donations improve.

---

## 9.4 Use First Donation Amount as One Signal

Supporters who gave larger first donations generally showed higher repeat rates. First-gift amount was also one of the strongest predictors in the model.

WVA can therefore use the first donation amount as one signal when deciding how much retention effort to give a supporter.

However, WVA should not simply focus only on high-value donors.

A smaller donor may still have good repeat potential because of their product, acquisition channel, previous relationship with WVA or model ranking.

Therefore, donation amount should be used together with other information rather than as a single rule.

---

## 9.5 Treat New Supporters Differently from Supporters Already Known to WVA

Supporters who already had some history with WVA generally showed stronger repeat behaviour than completely new supporters.

This suggests that the same communication should not necessarily be used for both groups.

A completely new supporter may need more information about WVA, trust building and evidence of impact.

A supporter who already knows WVA may be more ready for deeper engagement because they already have some familiarity with the organisation.

Previous WVA history should therefore be used to help personalise communication, but it should not be the main targeting factor.

---

## 9.6 Use the Model to Decide Who Should Receive More Attention

The model should not be used to say that a supporter will definitely donate again.

Instead, it should help WVA rank supporters from higher to lower repeat-donation potential.

In the final test data, the overall repeat rate was 13.8%. However, among the model's highest-ranked 20% of supporters, the repeat rate increased to 23.2%.

WVA can therefore use three simple priority groups:

- **High Priority - Top 20%:** receive stronger and more personalised retention activity.
- **Nurture - Next 20%:** continue active engagement using lower-cost communication.
- **Standard - Remaining 60%:** remain in the normal scalable supporter journey.

The Standard group should not be ignored. It simply means that these supporters have lower relative priority when WVA has limited resources.

---

## 9.7 Focus on the Variables That Actually Showed Useful Patterns

Product, first donation amount and acquisition channel were the strongest and most consistent predictors of repeat behaviour.

Other variables such as geography and some demographic or segmentation variables showed much weaker patterns.

WVA should therefore avoid making the retention strategy unnecessarily complicated.

The organisation should first focus on the information that showed the clearest value: what the supporter donated to, how much they gave, how they were acquired and their overall model ranking.

---

## 9.8 Keep Checking Whether Donor Behaviour Is Changing

Repeat rates changed noticeably across different years.

This means WVA should not assume that a strategy that worked several years ago will always work in the same way.

WVA should regularly check whether repeat rates, important products, acquisition channels and model performance are changing.

The predictive model should also be refreshed when enough new supporter data becomes available.

This is also why ranking supporters is more practical than relying permanently on one fixed probability threshold.

---

## 9.9 Test New Retention Ideas Before Applying Them to Everyone

The analysis shows which supporter characteristics are linked with repeat donations, but it does not prove that changing one particular action will automatically cause more donations.

Therefore, major recommendations should be tested before being introduced to all supporters.

For example, WVA could give one group of Retail Stand supporters the current follow-up journey and another group an improved journey.

If the improved group produces a higher repeat-donation rate, WVA has much stronger evidence that the new approach works.

WVA should measure success using repeat donations, additional revenue and campaign cost, rather than only measures such as email opens or clicks.

# 10. Assumptions, Limitations and Future Work

## 10.1 Key Assumptions

The following assumptions were used consistently throughout the analysis:

1. **Definition of a qualifying one-off donation**  
   Donations identified as Bounceback were excluded because they form part of a different sponsorship-related supporter journey. All remaining eligible one-off donations were treated as qualifying donations.

2. **Definition of a repeat donor**  
   A supporter was classified as a repeat donor only when another qualifying one-off donation occurred on a later date within 24 months of the first qualifying giving occasion.

3. **Same-day donations were treated as one giving occasion**  
   Multiple transactions made by the same supporter on their first donation date were treated as one first-giving occasion rather than separate repeat donations.

4. **First-gift amount represented the full first-day giving value**  
   Where several transactions occurred on the first day, their amounts were combined to calculate the total first-gift value.

5. **Primary product and acquisition context**  
   Where several transactions occurred on the first donation date, the largest transaction was used to identify the primary product and acquisition context.

6. **Active ongoing supporters were excluded**  
   Supporters who already had an active ongoing commitment at the time of their first qualifying one-off donation were excluded because their relationship with WVA was materially different from a typical one-off donor.

7. **Former ongoing supporters were retained**  
   Supporters whose previous ongoing commitment had already ended before their first qualifying one-off donation were retained because they were not active ongoing supporters at the prediction point.

8. **Mass supporters were used for predictive modelling**  
   The predictive model focused on the Mass supporter population because relationship-managed groups such as Private supporters followed different engagement processes and had relatively small populations.

9. **Only information available at the first giving occasion was used for prediction**  
   Future donations, future communication activity and other information occurring after the prediction point were excluded to prevent data leakage.

10. **A 24-month observation period was considered sufficient to define repeat behaviour**  
    Only supporters with a complete 24-month follow-up period were included when creating the target variable.

11. **Historical behaviour can provide information about future behaviour**  
    The modelling approach assumes that patterns observed in earlier supporters provide some useful information for ranking newer supporters, while recognising that these patterns may change over time.

12. **Priority groups represent relative potential rather than certainty**  
    High Priority, Nurture and Standard groups indicate relative repeat-donation potential. They do not mean that an individual supporter will definitely repeat or will definitely not repeat.

---

## 10.2 Limitations

Several limitations should be considered when interpreting the results.

1. **Moderate predictive performance**  
   The final Logistic Regression model achieved a ROC-AUC of 0.6266. This shows useful ranking ability but not strong enough performance to make confident individual-level predictions. The model should therefore be used for prioritisation rather than definite Yes/No decisions.

2. **The analysis does not establish causation**  
   EDA and modelling identify relationships between characteristics and repeat behaviour. For example, higher repeat rates for a particular product or channel do not prove that the product or channel directly caused the supporter to donate again.

3. **Supporter behaviour changed over time**  
   Repeat rates declined across recent cohorts, indicating that donor behaviour is not stable. Model performance, probability thresholds and retention patterns may therefore change in future periods.

4. **A fixed probability threshold may become less reliable over time**  
   The validation-selected threshold did not maintain the same Recall in the newer test population. Rank-based prioritisation is therefore more suitable than permanently relying on one probability cut-off.

5. **Some potentially useful engagement data had limited historical coverage**  
   Communications, click and portal engagement data were not consistently available across the full supporter history. These variables could therefore not be relied upon as core predictive features.

6. **Communication click data covered a limited historical period**  
   Click information was available for only a relatively short period, making it unsuitable for consistent long-term modelling.

7. **Portal engagement was available for only a small proportion of supporters before their first donation**  
   This reduced its usefulness as a general predictor for the full supporter population.

8. **Some demographic variables contained substantial missing data**  
   Age and several demographic attributes had high levels of missing information. Age at first gift was therefore excluded from the main predictive model.

9. **Helix and other supporter segmentation variables may reflect current rather than historical characteristics**  
   These variables may not accurately represent what was known about the supporter at the time of their first donation and were therefore not relied upon as core historical predictors.

10. **Geographic and demographic variables showed relatively weak predictive value**  
    Although these variables were explored, they provided less useful information than first-gift product, amount and acquisition channel.

11. **Product and channel groups may contain small subgroups**  
    Some products and acquisition channels had relatively small supporter counts. Very high or low repeat rates within these groups should therefore be interpreted carefully.

12. **The 2024 cohort is only a partial calendar year**  
    The available eligible first-gift period ends in March 2024, so the 2024 cohort should not be directly interpreted as representing a complete year of supporter acquisition.

13. **Historical acquisition and campaign practices may have changed**  
    Differences between years may reflect changes in campaigns, products, supporter journeys or organisational practices that are not fully captured by the available variables.

14. **Primary product assignment simplifies complex first-day behaviour**  
    For supporters making several different donations on their first day, using the largest transaction to define the primary product provides a consistent rule but may not fully represent their motivations.

15. **Feature importance represents model usefulness, not real-world importance or causality**  
    Product, amount and channel were consistently useful for prediction, but their importance values should not be interpreted as proof that changing these factors will directly increase repeat donations.

16. **Priority segment boundaries are practical rather than universal rules**  
    The Top 20%, next 20% and remaining 60% segmentation provides a useful decision framework based on current results. WVA may adjust these boundaries depending on campaign capacity, cost and available resources.

17. **The model does not measure the financial value of each future supporter**  
    The current target predicts whether a supporter repeats within 24 months, not the amount of future revenue they may generate. A supporter with a high repeat probability is not necessarily the supporter with the highest future financial value.

---

## 10.3 Future Work

Future work should focus on improving the practical use of the findings rather than simply increasing model complexity.

- Test the recommended retention journeys using controlled A/B experiments.
- Improve consistent collection of communication, digital engagement and supporter interaction data.
- Refresh the model periodically as newer supporter data becomes available.
- Monitor whether product, channel and first-gift patterns continue to remain important.
- Review priority segment sizes according to campaign capacity and cost.
- Consider future modelling of expected donation value in addition to repeat-donation probability.
- Measure the success of retention strategies using repeat rate, incremental donations, revenue and campaign cost.

# 11. Conclusion

This project examined one-off donor behaviour to understand which supporters are more likely to make another qualifying one-off donation and how WVA can use this information to improve retention.

The EDA showed that repeat behaviour differs most clearly by first-gift product, donation amount and acquisition channel. Supporters connected with some products, particularly MGI, showed stronger repeat behaviour, while Retail Stand acquisition showed relatively weak repeat performance. Previous familiarity with WVA also provided some useful context, although it was less important than first-gift behaviour.

Timing was another important finding. Around 75.7% of observed repeat donations occurred within the first 12 months and 91.3% within 18 months. This suggests that the period immediately following the first donation is important for building an ongoing supporter relationship.

The final Logistic Regression model achieved a ROC-AUC of 0.6266 on the unseen 2023–2024 population. While this is not strong enough for definite individual-level predictions, the model was useful for prioritisation. The highest-ranked 20% of supporters achieved a 23.2% repeat rate compared with 13.8% overall.

The main opportunity for WVA is therefore to combine a structured first-year retention journey with product- and channel-specific communication, while using the predictive ranking to decide where greater retention effort should be focused. The model should support decision-making rather than replace it, and future retention strategies should be tested and monitored as supporter behaviour changes over time.